### STEP 0. 실험 목표와 평가 기준 고정
- 노트북 초반에 이번 실험의 목적을 명확히 고정하여 뒤로 가면서 실험이 흔들리지 않게 함

In [1]:
# ============================================================
# [STEP 0 - CELL 1] 실험 목적 정의
# ------------------------------------------------------------
# 이 노트북의 목적:
# 1) 소아청소년질환 Q&A JSON 데이터를 이용해 RAG 기반 소아과 상담 챗봇의 성능을 실험한다.
# 2) 질문/답변 데이터를 정리하여 RAG용 문서셋을 만든다.
# 3) Naive RAG → Hybrid Retrieval → Reranker → Prompt 최적화까지 실험한다.
# 4) 최종적으로 RAGAS와 G-EVAL을 통해 가장 좋은 구조를 선택한다.
# 5) 나중에 Cursor에게 재개발 지시를 내릴 수 있도록 최종 사양을 정리한다.
# ------------------------------------------------------------
# 이번 셀은 설명용 셀이며 실행 필수는 아니다.
# ============================================================

EXPERIMENT_NAME = "baby_doc_RAG"
EXPERIMENT_VERSION = "v1"
TARGET_DOMAIN = "소아청소년질환"
GOAL = "RAG 기반 소아과 상담 챗봇의 최적 구조 도출"

print(f"[INFO] EXPERIMENT_NAME : {EXPERIMENT_NAME}")
print(f"[INFO] EXPERIMENT_VERSION : {EXPERIMENT_VERSION}")
print(f"[INFO] TARGET_DOMAIN : {TARGET_DOMAIN}")
print(f"[INFO] GOAL : {GOAL}")

[INFO] EXPERIMENT_NAME : baby_doc_RAG
[INFO] EXPERIMENT_VERSION : v1
[INFO] TARGET_DOMAIN : 소아청소년질환
[INFO] GOAL : RAG 기반 소아과 상담 챗봇의 최적 구조 도출


In [3]:
# ============================================================
# [STEP 0 - CELL 2] 경로 설정
# ------------------------------------------------------------
# 프로젝트 경로 및 원천 데이터 경로를 지정한다.
# 아래 경로는 현재 사용자 환경 기준으로 작성한다.
# 필요 시 본인 환경에 맞게 수정한다.
# ============================================================

from pathlib import Path

# 프로젝트 루트
PROJECT_ROOT = Path(r"D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj")

# 노트북에서 사용할 실험 산출물 폴더
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EVAL_DIR = DATA_DIR / "eval"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CACHE_DIR = PROJECT_ROOT / "cache"

# 원천 데이터 경로
QUESTION_ROOT = Path(r"D:\PyProject\DocTalk\Large_AI_healthcare_QNA\data\Training\label_data\1.질문\소아청소년질환")
ANSWER_ROOT   = Path(r"D:\PyProject\DocTalk\Large_AI_healthcare_QNA\data\Training\label_data\2.답변\소아청소년질환")

# 폴더 생성
for p in [DATA_DIR, RAW_DIR, PROCESSED_DIR, EVAL_DIR, OUTPUT_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("[INFO] PROJECT_ROOT :", PROJECT_ROOT)
print("[INFO] QUESTION_ROOT:", QUESTION_ROOT)
print("[INFO] ANSWER_ROOT  :", ANSWER_ROOT)
print("[INFO] 폴더 준비 완료")

[INFO] PROJECT_ROOT : D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj
[INFO] QUESTION_ROOT: D:\PyProject\DocTalk\Large_AI_healthcare_QNA\data\Training\label_data\1.질문\소아청소년질환
[INFO] ANSWER_ROOT  : D:\PyProject\DocTalk\Large_AI_healthcare_QNA\data\Training\label_data\2.답변\소아청소년질환
[INFO] 폴더 준비 완료


In [5]:
# ============================================================
# [STEP 0 - CELL 3] 기본 라이브러리 import
# ------------------------------------------------------------
# 데이터 로드, 전처리, 통계 확인, 저장 등에 필요한 기본 라이브러리를 불러온다.
# ============================================================

import os
import re
import json
import math
import random
import textwrap
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

print("[INFO] 기본 라이브러리 import 완료")

[INFO] 기본 라이브러리 import 완료


In [7]:
# ============================================================
# [STEP 0 - CELL 4] 유틸 함수 정의 - JSON 로드 / 저장 / 텍스트 정리
# ------------------------------------------------------------
# 이후 셀에서 반복적으로 사용할 공통 함수들을 정의한다.
# ============================================================

def load_json(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        return json.load(f)

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def clean_text(text):
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\u200b", " ").replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def join_nonempty(parts, sep="\n"):
    parts = [clean_text(x) for x in parts if clean_text(x)]
    return sep.join(parts)

print("[INFO] 공통 유틸 함수 준비 완료")

[INFO] 공통 유틸 함수 준비 완료


### STEP 1. 원천 데이터 구조 점검
- 질문/답변 데이터가 실제로 어떻게 생겼는지 정확히 확인

In [23]:
# ============================================================
# [STEP 1 - CELL 1] 파일 목록 수집 함수 정의
# ------------------------------------------------------------
# 질문 폴더와 답변 폴더 아래의 모든 JSON 파일을 재귀 탐색하기 위한 함수를 정의한다.
# ============================================================

def list_json_files(root_dir: Path):
    if not root_dir.exists():
        print(f"[WARN] 경로가 존재하지 않습니다: {root_dir}")
        return []
    return sorted(root_dir.rglob("*.json"))

question_files = list_json_files(QUESTION_ROOT)
answer_files = list_json_files(ANSWER_ROOT)

print(f"[INFO] question_files 개수: {len(question_files):,}")
print(f"[INFO] answer_files   개수: {len(answer_files):,}")

print("\n[샘플 질문 파일]")
for p in question_files[:3]:
    print(" -", p)

print("\n[샘플 답변 파일]")
for p in answer_files[:3]:
    print(" -", p)

[INFO] question_files 개수: 39,587
[INFO] answer_files   개수: 58,499

[샘플 질문 파일]
 - D:\PyProject\DocTalk\Large_AI_healthcare_QNA\data\Training\label_data\1.질문\소아청소년질환\가와사키병\약물\HC-Q-0434554.json
 - D:\PyProject\DocTalk\Large_AI_healthcare_QNA\data\Training\label_data\1.질문\소아청소년질환\가와사키병\약물\HC-Q-0434555.json
 - D:\PyProject\DocTalk\Large_AI_healthcare_QNA\data\Training\label_data\1.질문\소아청소년질환\가와사키병\약물\HC-Q-0434556.json

[샘플 답변 파일]
 - D:\PyProject\DocTalk\Large_AI_healthcare_QNA\data\Training\label_data\2.답변\소아청소년질환\가와사키병\약물\HC-A-02398884.json
 - D:\PyProject\DocTalk\Large_AI_healthcare_QNA\data\Training\label_data\2.답변\소아청소년질환\가와사키병\약물\HC-A-02398886.json
 - D:\PyProject\DocTalk\Large_AI_healthcare_QNA\data\Training\label_data\2.답변\소아청소년질환\가와사키병\약물\HC-A-02398893.json


In [25]:
# ============================================================
# [STEP 1 - CELL 2] 경로 기반 메타정보 추출 함수 정의
# ------------------------------------------------------------
# 폴더 구조:
# 질문: .../1.질문/소아청소년질환/질병명/intention/*.json
# 답변: .../2.답변/소아청소년질환/질병명/intention/*.json
#
# 이 구조를 활용하여 경로 자체에서
# - disease_category
# - disease_name
# - intention
# 을 1차 추출한다.
# ============================================================

def parse_path_info(file_path: Path, root_dir: Path):
    rel = file_path.relative_to(root_dir)
    parts = list(rel.parts)

    # 예상 구조: 질병명 / intention / 파일.json
    disease_name_from_path = parts[0] if len(parts) >= 3 else None
    intention_from_path = parts[1] if len(parts) >= 3 else None

    return {
        "relative_path": str(rel),
        "disease_name_from_path": disease_name_from_path,
        "intention_from_path": intention_from_path,
    }

# 샘플 확인
sample_q = parse_path_info(question_files[0], QUESTION_ROOT) if question_files else {}
sample_a = parse_path_info(answer_files[0], ANSWER_ROOT) if answer_files else {}

print("[샘플 질문 path info]")
print(sample_q)

print("\n[샘플 답변 path info]")
print(sample_a)

[샘플 질문 path info]
{'relative_path': '가와사키병\\약물\\HC-Q-0434554.json', 'disease_name_from_path': '가와사키병', 'intention_from_path': '약물'}

[샘플 답변 path info]
{'relative_path': '가와사키병\\약물\\HC-A-02398884.json', 'disease_name_from_path': '가와사키병', 'intention_from_path': '약물'}


In [27]:
# ============================================================
# [STEP 1 - CELL 3] 질문 JSON 로더 정의
# ------------------------------------------------------------
# 질문 JSON에서 필요한 핵심 필드를 추출하여 표준화된 dict로 변환한다.
# 이후 질문 통계 확인, 평가셋 생성, query 예시 생성 등에 활용한다.
# ============================================================

def normalize_question_record(path: Path):
    obj = load_json(path)
    path_info = parse_path_info(path, QUESTION_ROOT)

    disease_kor_json = obj.get("disease_name", {}).get("kor", "")
    disease_eng_json = obj.get("disease_name", {}).get("eng", "")

    record = {
        "source_path": str(path),
        "relative_path": path_info["relative_path"],
        "file_name": obj.get("fileName", ""),
        "participant_id": obj.get("participantsInfo", {}).get("participantID", ""),
        "gender": obj.get("participantsInfo", {}).get("gender", ""),
        "age": obj.get("participantsInfo", {}).get("age", ""),
        "occupation": obj.get("participantsInfo", {}).get("occupation", ""),
        "history": obj.get("participantsInfo", {}).get("history", None),
        "region": obj.get("participantsInfo", {}).get("rPlace", ""),
        "disease_category": obj.get("disease_category", ""),
        "disease_kor": disease_kor_json,
        "disease_eng": disease_eng_json,
        "intention": obj.get("intention", ""),
        "question": clean_text(obj.get("question", "")),
        "entities": obj.get("entities", []),
        "num_of_words": obj.get("num_of_words", None),
        "disease_name_from_path": path_info["disease_name_from_path"],
        "intention_from_path": path_info["intention_from_path"],
    }
    return record

# 샘플 1개 확인
if question_files:
    q_sample = normalize_question_record(question_files[0])
    print(json.dumps(q_sample, ensure_ascii=False, indent=2))
else:
    print("[WARN] 질문 파일이 없습니다.")

{
  "source_path": "D:\\PyProject\\DocTalk\\Large_AI_healthcare_QNA\\data\\Training\\label_data\\1.질문\\소아청소년질환\\가와사키병\\약물\\HC-Q-0434554.json",
  "relative_path": "가와사키병\\약물\\HC-Q-0434554.json",
  "file_name": "HC-Q-0434554",
  "participant_id": "QC015",
  "gender": "남성",
  "age": "50대 이상",
  "occupation": "생산/노무직",
  "history": false,
  "region": "부산/대구/울산/경상",
  "disease_category": "소아청소년질환",
  "disease_kor": "가와사키병",
  "disease_eng": "Kawasaki Disease",
  "intention": "약물",
  "question": "가와사키병 약물 치료의 기간은 개인별로 차이가 있는지, 아니면 대부분의 환자에게 동일하게 나타나는지 궁금합니다.",
  "entities": [
    {
      "id": 0,
      "text": "가와사키병",
      "entity": "질환명",
      "position": 0
    }
  ],
  "num_of_words": 13,
  "disease_name_from_path": "가와사키병",
  "intention_from_path": "약물"
}


In [29]:
# ============================================================
# [STEP 1 - CELL 4] 답변 JSON 로더 정의
# ------------------------------------------------------------
# 답변 JSON에서 필요한 핵심 필드를 추출하여 표준화된 dict로 변환한다.
# 이후 RAG 문서 본문, reference answer, grounding source 등에 활용한다.
# ============================================================

def normalize_answer_record(path: Path):
    obj = load_json(path)
    path_info = parse_path_info(path, ANSWER_ROOT)

    disease_kor_json = obj.get("disease_name", {}).get("kor", "")
    disease_eng_json = obj.get("disease_name", {}).get("eng", "")
    answer_obj = obj.get("answer", {}) or {}

    intro = clean_text(answer_obj.get("intro", ""))
    body = clean_text(answer_obj.get("body", ""))
    conclusion = clean_text(answer_obj.get("conclusion", ""))

    record = {
        "source_path": str(path),
        "relative_path": path_info["relative_path"],
        "file_name": obj.get("fileName", ""),
        "disease_category": obj.get("disease_category", ""),
        "disease_kor": disease_kor_json,
        "disease_eng": disease_eng_json,
        "department": obj.get("department", []),
        "intention": obj.get("intention", ""),
        "answer_intro": intro,
        "answer_body": body,
        "answer_conclusion": conclusion,
        "answer_full": join_nonempty([intro, body, conclusion], sep="\n"),
        "num_of_words": obj.get("num_of_words", None),
        "disease_name_from_path": path_info["disease_name_from_path"],
        "intention_from_path": path_info["intention_from_path"],
    }
    return record

# 샘플 1개 확인
if answer_files:
    a_sample = normalize_answer_record(answer_files[0])
    print(json.dumps(a_sample, ensure_ascii=False, indent=2))
else:
    print("[WARN] 답변 파일이 없습니다.")

{
  "source_path": "D:\\PyProject\\DocTalk\\Large_AI_healthcare_QNA\\data\\Training\\label_data\\2.답변\\소아청소년질환\\가와사키병\\약물\\HC-A-02398884.json",
  "relative_path": "가와사키병\\약물\\HC-A-02398884.json",
  "file_name": "HC-A-02398884",
  "disease_category": "소아청소년질환",
  "disease_kor": "가와사키병",
  "disease_eng": "Kawasaki Disease",
  "department": [
    "소아청소년과"
  ],
  "intention": "약물",
  "answer_intro": "가와사키병의 약물 치료는 주로 면역글로불린과 아스피린을 사용하여 치료합니다.",
  "answer_body": "면역글로불린은 혈관 내피의 증식을 억제하여 혈관의 염증을 줄이고, 통증을 완화하는 데 도움을 줍니다. 또한, 아스피린은 관상동맥의 문제를 예방하는 데 효과적입니다. 그러나 스테로이드에 대한 의견은 여전히 분분하여 더 많은 연구가 필요합니다.",
  "answer_conclusion": "약물 치료는 면역글로불린과 아스피린을 통해 관상동맥 문제의 발생을 예방할 수 있으며, 효과에 대한 추가적인 연구가 필요합니다.",
  "answer_full": "가와사키병의 약물 치료는 주로 면역글로불린과 아스피린을 사용하여 치료합니다.\n면역글로불린은 혈관 내피의 증식을 억제하여 혈관의 염증을 줄이고, 통증을 완화하는 데 도움을 줍니다. 또한, 아스피린은 관상동맥의 문제를 예방하는 데 효과적입니다. 그러나 스테로이드에 대한 의견은 여전히 분분하여 더 많은 연구가 필요합니다.\n약물 치료는 면역글로불린과 아스피린을 통해 관상동맥 문제의 발생을 예방할 수 있으며, 효과에 대한 추가적인 연구가 필요합니다.",
  "num_of_words": 54,
  "disease

In [31]:
# ============================================================
# [STEP 1 - CELL 5] 전체 질문/답변 데이터 로드
# ------------------------------------------------------------
# 모든 질문 JSON과 답변 JSON을 DataFrame으로 변환한다.
# 이 단계에서 데이터 개요를 확인할 수 있다.
# ============================================================

question_records = [normalize_question_record(p) for p in question_files]
answer_records   = [normalize_answer_record(p) for p in answer_files]

q_df = pd.DataFrame(question_records)
a_df = pd.DataFrame(answer_records)

print("[INFO] q_df shape:", q_df.shape)
print("[INFO] a_df shape:", a_df.shape)

display(q_df.head(3))
display(a_df.head(3))

[INFO] q_df shape: (39587, 18)
[INFO] a_df shape: (58499, 15)


,source_path,relative_path,file_name,participant_id,gender,age,occupation,history,region,disease_category,disease_kor,disease_eng,intention,question,entities,num_of_words,disease_name_from_path,intention_from_path
0,D:\PyProject\DocTalk\Large_AI_healthcare_QNA\d...,가와사키병\약물\HC-Q-0434554.json,HC-Q-0434554,QC015,남성,50대 이상,생산/노무직,False,부산/대구/울산/경상,소아청소년질환,가와사키병,Kawasaki Disease,약물,"가와사키병 약물 치료의 기간은 개인별로 차이가 있는지, 아니면 대부분의 환자에게 동...","[{'id': 0, 'text': '가와사키병', 'entity': '질환명', '...",13,가와사키병,약물
1,D:\PyProject\DocTalk\Large_AI_healthcare_QNA\d...,가와사키병\약물\HC-Q-0434555.json,HC-Q-0434555,QC062,여성,50대 이상,기타,False,서울/경기/인천,소아청소년질환,가와사키병,Kawasaki Disease,약물,가와사키병의 진단 후에는 어떤 약물 치료와 검사를 받아야 하는지 궁금합니다.,"[{'id': 0, 'text': '가와사키병', 'entity': '질환명', '...",10,가와사키병,약물
2,D:\PyProject\DocTalk\Large_AI_healthcare_QNA\d...,가와사키병\약물\HC-Q-0434556.json,HC-Q-0434556,QC162,남성,40대,사무직,False,서울/경기/인천,소아청소년질환,가와사키병,Kawasaki Disease,약물,가와사키병을 치료하기 위해 아스피린 이외에 어떤 약물을 사용하는지 알려주세요.,"[{'id': 0, 'text': '가와사키병', 'entity': '질환명', '...",9,가와사키병,약물


,source_path,relative_path,file_name,disease_category,disease_kor,disease_eng,department,intention,answer_intro,answer_body,answer_conclusion,answer_full,num_of_words,disease_name_from_path,intention_from_path
0,D:\PyProject\DocTalk\Large_AI_healthcare_QNA\d...,가와사키병\약물\HC-A-02398884.json,HC-A-02398884,소아청소년질환,가와사키병,Kawasaki Disease,[소아청소년과],약물,가와사키병의 약물 치료는 주로 면역글로불린과 아스피린을 사용하여 치료합니다.,"면역글로불린은 혈관 내피의 증식을 억제하여 혈관의 염증을 줄이고, 통증을 완화하는 ...",약물 치료는 면역글로불린과 아스피린을 통해 관상동맥 문제의 발생을 예방할 수 있으며...,가와사키병의 약물 치료는 주로 면역글로불린과 아스피린을 사용하여 치료합니다.\n면역...,54,가와사키병,약물
1,D:\PyProject\DocTalk\Large_AI_healthcare_QNA\d...,가와사키병\약물\HC-A-02398886.json,HC-A-02398886,소아청소년질환,가와사키병,Kawasaki Disease,[소아청소년과],약물,가와사키병의 약물은 어떤 것이 있을까요?,가와사키병의 치료에는 주로 면역글로불린과 아스피린이 사용됩니다. 면역글로불린은 혈관...,"따라서, 면역글로불린과 아스피린은 가와사키병의 치료에 있어 중요한 약물이며, 의사의...",가와사키병의 약물은 어떤 것이 있을까요?\n가와사키병의 치료에는 주로 면역글로불린과...,54,가와사키병,약물
2,D:\PyProject\DocTalk\Large_AI_healthcare_QNA\d...,가와사키병\약물\HC-A-02398893.json,HC-A-02398893,소아청소년질환,가와사키병,Kawasaki Disease,[소아청소년과],약물,가와사키병의 치료에는 다양한 약물이 사용됩니다.,가와사키병의 급성 발작 시에는 면역글로불린과 아스피린이 주로 사용됩니다. 면역글로불...,"결론적으로, 가와사키병의 치료는 면역글로불린과 아스피린의 적절한 사용이 필요합니다....",가와사키병의 치료에는 다양한 약물이 사용됩니다.\n가와사키병의 급성 발작 시에는 면...,82,가와사키병,약물


In [32]:
# ============================================================
# [STEP 1 - CELL 6] 데이터 무결성 점검 - 경로 정보와 JSON 필드 일치 확인
# ------------------------------------------------------------
# 경로에서 추출한 질병명/intention과 JSON 내부 값이 서로 일치하는지 확인한다.
# 이 점검은 이후 매칭 전략 신뢰도에 중요하다.
# ============================================================

def compare_path_vs_json(df, disease_col="disease_kor", intention_col="intention"):
    disease_match = (df[disease_col].fillna("") == df["disease_name_from_path"].fillna("")).mean()
    intention_match = (df[intention_col].fillna("") == df["intention_from_path"].fillna("")).mean()
    return disease_match, intention_match

q_disease_match, q_intention_match = compare_path_vs_json(q_df)
a_disease_match, a_intention_match = compare_path_vs_json(a_df)

print("[질문 데이터]")
print(f" - disease path/json 일치율   : {q_disease_match:.4f}")
print(f" - intention path/json 일치율 : {q_intention_match:.4f}")

print("\n[답변 데이터]")
print(f" - disease path/json 일치율   : {a_disease_match:.4f}")
print(f" - intention path/json 일치율 : {a_intention_match:.4f}")

[질문 데이터]
 - disease path/json 일치율   : 1.0000
 - intention path/json 일치율 : 1.0000

[답변 데이터]
 - disease path/json 일치율   : 1.0000
 - intention path/json 일치율 : 1.0000


In [33]:
# ============================================================
# [STEP 1 - CELL 7] 질환별 / 의도별 분포 확인
# ------------------------------------------------------------
# 어떤 질환, 어떤 의도(intention)가 얼마나 있는지 확인한다.
# 이후 평가셋 구성과 문서셋 품질 점검에 중요하다.
# ============================================================

print("[질문 데이터 - 질환별 상위 20개]")
display(q_df["disease_kor"].value_counts().head(20).to_frame("count"))

print("[질문 데이터 - intention 분포]")
display(q_df["intention"].value_counts().to_frame("count"))

print("[답변 데이터 - 질환별 상위 20개]")
display(a_df["disease_kor"].value_counts().head(20).to_frame("count"))

print("[답변 데이터 - intention 분포]")
display(a_df["intention"].value_counts().to_frame("count"))

[질문 데이터 - 질환별 상위 20개]


,count
disease_kor,
성홍열,2483
유당분해효소결핍증,2483
가와사키병,2482
라이 증후군,2330
아스퍼거 증후군,2313
유행성 이하선염,1943
성조숙증,1882
발달 장애,1788
기관지폐 형성이상,1651


[질문 데이터 - intention 분포]


,count
intention,
정의,7224
원인,6992
치료,6677
진단,6353
증상,5926
예방,2417
약물,2091
"식이, 생활",1403
재활,321


[답변 데이터 - 질환별 상위 20개]


,count
disease_kor,
발달 장애,7585
저신장증,6677
성조숙증,5241
가와사키병,4879
아스퍼거 증후군,4597
성홍열,4559
유당분해효소결핍증,3262
신생아 호흡곤란증후군,2856
유행성 이하선염,2129


[답변 데이터 - intention 분포]


,count
intention,
원인,14274
치료,11889
약물,9124
정의,7244
진단,4600
증상,3680
예방,3016
"식이, 생활",2822
재활,1274


In [34]:
# ============================================================
# [STEP 1 - CELL 8] 질문/답변 키 조합 점검
# ------------------------------------------------------------
# 파일명으로 직접 매칭이 어려운 경우,
# disease_kor + intention 조합으로 매칭 가능한지 먼저 확인한다.
# 이 셀은 이후 문서 빌드 전략의 핵심 근거가 된다.
# ============================================================

q_key_counts = (
    q_df.groupby(["disease_kor", "intention"])
    .size()
    .reset_index(name="question_count")
)

a_key_counts = (
    a_df.groupby(["disease_kor", "intention"])
    .size()
    .reset_index(name="answer_count")
)

key_merge_df = q_key_counts.merge(
    a_key_counts,
    on=["disease_kor", "intention"],
    how="outer"
).fillna(0)

key_merge_df["question_count"] = key_merge_df["question_count"].astype(int)
key_merge_df["answer_count"] = key_merge_df["answer_count"].astype(int)

print("[INFO] disease_kor + intention 조합 수:", len(key_merge_df))
display(key_merge_df.head(20))

print("\n[INFO] 질문은 있는데 답변이 없는 조합]")
display(key_merge_df[key_merge_df["answer_count"] == 0].head(20))

print("\n[INFO] 답변은 있는데 질문이 없는 조합]")
display(key_merge_df[key_merge_df["question_count"] == 0].head(20))

[INFO] disease_kor + intention 조합 수: 139


,disease_kor,intention,question_count,answer_count
0,가와사키병,약물,416,1358
1,가와사키병,예방,238,160
2,가와사키병,원인,387,1466
3,가와사키병,정의,312,1415
4,가와사키병,증상,341,160
5,가와사키병,진단,429,160
6,가와사키병,치료,359,160
7,과숙아,원인,271,160
8,과숙아,정의,378,160
9,과숙아,증상,282,160



[INFO] 질문은 있는데 답변이 없는 조합]


,disease_kor,intention,question_count,answer_count



[INFO] 답변은 있는데 질문이 없는 조합]


,disease_kor,intention,question_count,answer_count


#### 정리
지금 데이터에서 가장 중요한 결론
4. “질환명 + intention = 대표 지식문서 1개”로 압축해야 합니다

현재는 각 조합마다 답변이 많으므로, 다음 중 하나로 가야 합니다.

### * 추천 방식 A. 대표 답변 압축 문서

각 질환+intention 그룹에서

중복 제거
유사 답변 축약
대표 답변 상위 N개만 사용
또는 LLM/규칙 기반으로 통합 요약

이렇게 해서 최종 RAG 문서는 1개 대표 문서로 만드는 방식입니다.

이 방식의 장점:

검색 단위가 선명함
chunking 없이도 실험 가능
metadata filtering과 잘 맞음
Cursor에게 넘길 때 구조가 단순함

### * 추천 방식 B. 질환+intention 그룹 안에서 소문서 여러 개

예:

가와사키병__약물__001
가와사키병__약물__002
가와사키병__약물__003

이렇게 답변 몇 개씩 묶어서 여러 문서로 나누는 방식입니다.

이건 나중에 성능 비교용으로는 좋지만,
#### * 첫 baseline은 A가 더 좋습니다.
#### 현재 데이터는 disease_kor + intention 기준의 RAG 문서화에 매우 적합하며, 가장 큰 다음 과제는 “많은 답변을 대표 지식문서로 어떻게 압축할지”입니다

### STEP 2. 질문-답변 페어링 전략 확정
 - 방식 A. 질환명 + intention 기준 대표 QA 문서화: 이 방식이 구조는 안정적
 - 방식 B. 질문 하나마다 가장 적절한 답변을 붙여 pseudo pair 생성: 이건 더 자연스럽지만, 매칭 오류 발생 가능성 있음

In [39]:
# ============================================================
# [STEP 2 - CELL 1] 매칭 전략 결정
# ------------------------------------------------------------
# 현재 데이터 구조에서는 질문 fileName과 답변 fileName이 직접 연결되지 않을 수 있다.
# 따라서 우선 가장 안정적인 전략으로:
#
# [질환명(disease_kor) + intention] 단위 문서화
#
# 를 기본 전략으로 채택한다.
#
# 즉,
# - 질문은 같은 질환/의도 그룹 내 여러 질문 예시로 묶고
# - 답변은 같은 질환/의도 그룹의 대표 지식 문서로 사용한다.
# ------------------------------------------------------------
# 이 셀은 설명용이며, 아래에 실제 문서 빌드 코드를 이어서 작성한다.
# ============================================================

MATCH_KEY_COLS = ["disease_kor", "intention"]

print("[INFO] 매칭 전략:", MATCH_KEY_COLS)
print("[INFO] 기본 전략: disease_kor + intention 기준 그룹 문서화")

[INFO] 매칭 전략: ['disease_kor', 'intention']
[INFO] 기본 전략: disease_kor + intention 기준 그룹 문서화


In [41]:
# ============================================================
# [STEP 2 - CELL 2] 질문 그룹 생성
# ------------------------------------------------------------
# 같은 질환명 + intention 그룹에 속한 질문들을 묶어서
# question_examples 리스트로 관리한다.
# ============================================================

question_grouped = (
    q_df.groupby(MATCH_KEY_COLS, dropna=False)
    .agg(
        disease_eng=("disease_eng", lambda x: next((v for v in x if clean_text(v)), "")),
        disease_category=("disease_category", lambda x: next((v for v in x if clean_text(v)), "")),
        question_examples=("question", lambda x: list(dict.fromkeys([clean_text(v) for v in x if clean_text(v)]))),
        question_count=("question", "count"),
    )
    .reset_index()
)

print("[INFO] question_grouped shape:", question_grouped.shape)
display(question_grouped.head(5))

[INFO] question_grouped shape: (139, 6)


,disease_kor,intention,disease_eng,disease_category,question_examples,question_count
0,가와사키병,약물,Kawasaki Disease,소아청소년질환,"[가와사키병 약물 치료의 기간은 개인별로 차이가 있는지, 아니면 대부분의 환자에게 ...",416
1,가와사키병,예방,Kawasaki Disease,소아청소년질환,"[가와사키병의 예방을 위해 어떤 종류의 운동이 도움이 될 수 있을까요?, 가와사키병...",238
2,가와사키병,원인,Kawasaki Disease,소아청소년질환,"[가와사키병의 원인에 대해 더 자세한 내용을 알려주세요., 가와사키병에 의해 혀가 ...",387
3,가와사키병,정의,Kawasaki Disease,소아청소년질환,"[가와사키병이라는 질병에 대해 자세히 알려주세요., 가와사키병이라는 용어가 소아에서...",312
4,가와사키병,증상,Kawasaki Disease,소아청소년질환,"[가와사키병 진단 후에도 다른 증상이 나타날 수 있는지 궁금합니다., 가와사키병의 ...",341


In [43]:
# ============================================================
# [STEP 2 - CELL 3] 답변 그룹 생성
# ------------------------------------------------------------
# 같은 질환명 + intention 그룹의 답변들을 묶는다.
# 답변은 보통 intro/body/conclusion 구조가 있으므로 이를 합쳐 대표 지식으로 쓴다.
# 답변이 여러 개일 경우 중복 제거 후 합치거나 대표 1개를 선택할 수 있다.
# ------------------------------------------------------------
# 우선은:
# - answer_full 중복 제거
# - department 병합
# 으로 단순화한다.
# ============================================================

def unique_nonempty(values):
    out = []
    seen = set()
    for v in values:
        v = clean_text(v)
        if v and v not in seen:
            out.append(v)
            seen.add(v)
    return out

def flatten_unique_lists(series):
    seen = set()
    out = []
    for item in series:
        if isinstance(item, list):
            for v in item:
                v = clean_text(v)
                if v and v not in seen:
                    seen.add(v)
                    out.append(v)
    return out

answer_grouped = (
    a_df.groupby(MATCH_KEY_COLS, dropna=False)
    .agg(
        disease_eng=("disease_eng", lambda x: next((v for v in x if clean_text(v)), "")),
        disease_category=("disease_category", lambda x: next((v for v in x if clean_text(v)), "")),
        departments=("department", flatten_unique_lists),
        answer_texts=("answer_full", unique_nonempty),
        answer_count=("answer_full", "count"),
    )
    .reset_index()
)

print("[INFO] answer_grouped shape:", answer_grouped.shape)
display(answer_grouped.head(5))

[INFO] answer_grouped shape: (139, 7)


,disease_kor,intention,disease_eng,disease_category,departments,answer_texts,answer_count
0,가와사키병,약물,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병의 약물 치료는 주로 면역글로불린과 아스피린을 사용하여 치료합니다. 면역...,1358
1,가와사키병,예방,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병은 아직까지 백신이 개발되지 않았기 때문에 일반적인 예방법을 알아보겠습니...,160
2,가와사키병,원인,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병은 아직 그 원인에 대해서 확실히 밝혀지지 않은 질환입니다. 그러나 약 ...,1466
3,가와사키병,정의,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병은 일반적으로 6개월부터 2세까지의 영아에게 발생하는 급성 혈관염 질환으...,1415
4,가와사키병,증상,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병은 아직 원인이 명확히 밝혀지지 않았습니다. 현재까지는 이 질환과 연관된...,160


In [45]:
# ============================================================
# [STEP 2 - CELL 4] 질문 그룹 + 답변 그룹 병합
# ------------------------------------------------------------
# 질환명 + intention 기준으로 질문/답변 그룹을 병합한다.
# 이후 이 병합 결과가 실제 RAG 문서 생성의 기반이 된다.
# ============================================================

merged_group_df = question_grouped.merge(
    answer_grouped,
    on=MATCH_KEY_COLS,
    how="outer",
    suffixes=("_q", "_a")
)

# 대표 값 정리
merged_group_df["disease_eng"] = merged_group_df["disease_eng_q"].fillna("").where(
    merged_group_df["disease_eng_q"].fillna("") != "",
    merged_group_df["disease_eng_a"].fillna("")
)

merged_group_df["disease_category"] = merged_group_df["disease_category_q"].fillna("").where(
    merged_group_df["disease_category_q"].fillna("") != "",
    merged_group_df["disease_category_a"].fillna("")
)

merged_group_df["question_examples"] = merged_group_df["question_examples"].apply(
    lambda x: x if isinstance(x, list) else []
)
merged_group_df["answer_texts"] = merged_group_df["answer_texts"].apply(
    lambda x: x if isinstance(x, list) else []
)
merged_group_df["departments"] = merged_group_df["departments"].apply(
    lambda x: x if isinstance(x, list) else []
)

merged_group_df["question_count"] = merged_group_df["question_count"].fillna(0).astype(int)
merged_group_df["answer_count"] = merged_group_df["answer_count"].fillna(0).astype(int)

print("[INFO] merged_group_df shape:", merged_group_df.shape)
display(merged_group_df.head(5))

[INFO] merged_group_df shape: (139, 13)


,disease_kor,intention,disease_eng_q,disease_category_q,question_examples,question_count,disease_eng_a,disease_category_a,departments,answer_texts,answer_count,disease_eng,disease_category
0,가와사키병,약물,Kawasaki Disease,소아청소년질환,"[가와사키병 약물 치료의 기간은 개인별로 차이가 있는지, 아니면 대부분의 환자에게 ...",416,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병의 약물 치료는 주로 면역글로불린과 아스피린을 사용하여 치료합니다. 면역...,1358,Kawasaki Disease,소아청소년질환
1,가와사키병,예방,Kawasaki Disease,소아청소년질환,"[가와사키병의 예방을 위해 어떤 종류의 운동이 도움이 될 수 있을까요?, 가와사키병...",238,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병은 아직까지 백신이 개발되지 않았기 때문에 일반적인 예방법을 알아보겠습니...,160,Kawasaki Disease,소아청소년질환
2,가와사키병,원인,Kawasaki Disease,소아청소년질환,"[가와사키병의 원인에 대해 더 자세한 내용을 알려주세요., 가와사키병에 의해 혀가 ...",387,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병은 아직 그 원인에 대해서 확실히 밝혀지지 않은 질환입니다. 그러나 약 ...,1466,Kawasaki Disease,소아청소년질환
3,가와사키병,정의,Kawasaki Disease,소아청소년질환,"[가와사키병이라는 질병에 대해 자세히 알려주세요., 가와사키병이라는 용어가 소아에서...",312,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병은 일반적으로 6개월부터 2세까지의 영아에게 발생하는 급성 혈관염 질환으...,1415,Kawasaki Disease,소아청소년질환
4,가와사키병,증상,Kawasaki Disease,소아청소년질환,"[가와사키병 진단 후에도 다른 증상이 나타날 수 있는지 궁금합니다., 가와사키병의 ...",341,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병은 아직 원인이 명확히 밝혀지지 않았습니다. 현재까지는 이 질환과 연관된...,160,Kawasaki Disease,소아청소년질환


In [47]:
# ============================================================
# [STEP 2 - CELL 5] 병합 결과 품질 점검
# ------------------------------------------------------------
# 실제로 문서로 쓸 수 있는 그룹만 얼마나 되는지 확인한다.
# 기준:
# - 질문이 1개 이상
# - 답변이 1개 이상
# ------------------------------------------------------------
# 이 조건을 만족하는 그룹이 많을수록 현재 매칭 전략이 적절하다고 볼 수 있다.
# ============================================================

usable_group_df = merged_group_df[
    (merged_group_df["question_count"] > 0) &
    (merged_group_df["answer_count"] > 0)
].copy()

print("[INFO] 전체 그룹 수:", len(merged_group_df))
print("[INFO] 사용 가능한 그룹 수:", len(usable_group_df))
print("[INFO] 사용 가능 비율:", round(len(usable_group_df) / max(len(merged_group_df), 1), 4))

display(usable_group_df.head(5))

[INFO] 전체 그룹 수: 139
[INFO] 사용 가능한 그룹 수: 139
[INFO] 사용 가능 비율: 1.0


,disease_kor,intention,disease_eng_q,disease_category_q,question_examples,question_count,disease_eng_a,disease_category_a,departments,answer_texts,answer_count,disease_eng,disease_category
0,가와사키병,약물,Kawasaki Disease,소아청소년질환,"[가와사키병 약물 치료의 기간은 개인별로 차이가 있는지, 아니면 대부분의 환자에게 ...",416,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병의 약물 치료는 주로 면역글로불린과 아스피린을 사용하여 치료합니다. 면역...,1358,Kawasaki Disease,소아청소년질환
1,가와사키병,예방,Kawasaki Disease,소아청소년질환,"[가와사키병의 예방을 위해 어떤 종류의 운동이 도움이 될 수 있을까요?, 가와사키병...",238,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병은 아직까지 백신이 개발되지 않았기 때문에 일반적인 예방법을 알아보겠습니...,160,Kawasaki Disease,소아청소년질환
2,가와사키병,원인,Kawasaki Disease,소아청소년질환,"[가와사키병의 원인에 대해 더 자세한 내용을 알려주세요., 가와사키병에 의해 혀가 ...",387,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병은 아직 그 원인에 대해서 확실히 밝혀지지 않은 질환입니다. 그러나 약 ...,1466,Kawasaki Disease,소아청소년질환
3,가와사키병,정의,Kawasaki Disease,소아청소년질환,"[가와사키병이라는 질병에 대해 자세히 알려주세요., 가와사키병이라는 용어가 소아에서...",312,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병은 일반적으로 6개월부터 2세까지의 영아에게 발생하는 급성 혈관염 질환으...,1415,Kawasaki Disease,소아청소년질환
4,가와사키병,증상,Kawasaki Disease,소아청소년질환,"[가와사키병 진단 후에도 다른 증상이 나타날 수 있는지 궁금합니다., 가와사키병의 ...",341,Kawasaki Disease,소아청소년질환,[소아청소년과],[가와사키병은 아직 원인이 명확히 밝혀지지 않았습니다. 현재까지는 이 질환과 연관된...,160,Kawasaki Disease,소아청소년질환


### STEP 3. RAG용 문서 빌드
- 의료 QA는 그냥 답변만 넣는 것보다, 질환명/의도/질문 예시를 함께 넣는 것이 검색 성능에 유리

### step 2 내용 분석을 기반으로 Step 3-0 추가
-  [STEP 3-0 - CELL A] 그룹 크기 통계 확인
- [STEP 3-0 - CELL B] 대표 질문/답변 압축용 함수 정의
- [STEP 3-0 - CELL C] 그룹 압축 수행
- [STEP 3-0 - CELL D] 압축 결과 점검

In [63]:
# ============================================================
# [STEP 3-0 - CELL A] 그룹 크기 통계 확인
# ------------------------------------------------------------
# STEP 2에서 만든 usable_group_df의 질문/답변 개수 분포를 확인한다.
# 이 통계를 바탕으로 대표 질문 수, 대표 답변 수를 얼마로 제한할지 판단한다.
# ============================================================

print("[INFO] usable_group_df shape:", usable_group_df.shape)

print("\n[INFO] question_count 통계]")
print(usable_group_df["question_count"].describe())

print("\n[INFO] answer_count 통계]")
print(usable_group_df["answer_count"].describe())

print("\n[INFO] question_count 상위 20개 그룹]")
display(
    usable_group_df[["disease_kor", "intention", "question_count", "answer_count"]]
    .sort_values("question_count", ascending=False)
    .head(20)
)

print("\n[INFO] answer_count 상위 20개 그룹]")
display(
    usable_group_df[["disease_kor", "intention", "question_count", "answer_count"]]
    .sort_values("answer_count", ascending=False)
    .head(20)
)

[INFO] usable_group_df shape: (139, 13)

[INFO] question_count 통계]
count    139.000000
mean     284.798561
std       74.154772
min      113.000000
25%      231.000000
50%      285.000000
75%      341.000000
max      498.000000
Name: question_count, dtype: float64

[INFO] answer_count 통계]
count     139.000000
mean      420.856115
std       696.597981
min       160.000000
25%       160.000000
50%       160.000000
75%       160.000000
max      3637.000000
Name: answer_count, dtype: float64

[INFO] question_count 상위 20개 그룹]


,disease_kor,intention,question_count,answer_count
114,유당분해효소결핍증,증상,498,160
51,성조숙증,진단,494,160
5,가와사키병,진단,429,160
0,가와사키병,약물,416,1358
115,유당분해효소결핍증,진단,404,160
57,성홍열,정의,399,160
117,유행성 이하선염,약물,399,1169
96,아스퍼거 증후군,정의,394,160
129,제대 탈장,원인,394,160
66,신생아 경련,원인,392,160



[INFO] answer_count 상위 20개 그룹]


,disease_kor,intention,question_count,answer_count
94,아스퍼거 증후군,약물,346,3637
48,성조숙증,원인,336,3492
128,저신장증,치료,304,3456
32,발달 장애,원인,288,3375
124,저신장증,원인,269,2741
36,발달 장애,치료,271,2490
2,가와사키병,원인,387,1466
81,신생아 호흡곤란증후군,치료,316,1440
3,가와사키병,정의,312,1415
0,가와사키병,약물,416,1358


In [65]:
# ============================================================
# [STEP 3-0 - CELL B - FAST] 빠른 대표 질문/답변 압축 함수
# ------------------------------------------------------------
# 기존 SequenceMatcher 기반 유사도 제거는 매우 느릴 수 있다.
# 따라서 우선 빠른 baseline을 위해:
# - exact duplicate 제거
# - 최소 길이 필터
# - 길이 다양성 기반 샘플링
# 만 적용한다.
# ============================================================

MAX_QUESTION_EXAMPLES = 12
MAX_ANSWER_TEXTS = 12

MIN_QUESTION_LEN = 8
MIN_ANSWER_LEN = 30

def unique_clean_texts(texts, min_len=1):
    out = []
    seen = set()
    for t in texts:
        t = clean_text(t)
        if len(t) < min_len:
            continue
        if t not in seen:
            seen.add(t)
            out.append(t)
    return out

def select_representative_questions_fast(question_list,
                                         max_items=12,
                                         min_len=8):
    questions = unique_clean_texts(question_list, min_len=min_len)

    if len(questions) <= max_items:
        return questions

    # 길이순으로 정렬 후 고르게 샘플링
    questions_sorted = sorted(questions, key=len)
    idxs = np.linspace(0, len(questions_sorted) - 1, max_items, dtype=int)
    sampled = [questions_sorted[i] for i in idxs]
    sampled = list(dict.fromkeys(sampled))
    return sampled[:max_items]

def select_representative_answers_fast(answer_list,
                                       max_items=12,
                                       min_len=30):
    answers = unique_clean_texts(answer_list, min_len=min_len)

    if len(answers) <= max_items:
        return answers

    # 정보량이 어느 정도 있는 답변을 우선 보기 위해 길이 내림차순
    answers_sorted = sorted(answers, key=len, reverse=True)
    return answers_sorted[:max_items]

print("[INFO] FAST 압축 함수 정의 완료")
print(f"[INFO] MAX_QUESTION_EXAMPLES = {MAX_QUESTION_EXAMPLES}")
print(f"[INFO] MAX_ANSWER_TEXTS      = {MAX_ANSWER_TEXTS}")

[INFO] FAST 압축 함수 정의 완료
[INFO] MAX_QUESTION_EXAMPLES = 12
[INFO] MAX_ANSWER_TEXTS      = 12


In [67]:
# ============================================================
# [STEP 3-0 - CELL C - FAST] 그룹 압축 수행
# ------------------------------------------------------------
# 빠른 baseline용 압축:
# - exact dedup
# - 최소 길이 필터
# - 대표 질문/답변 최대 개수 제한
# ============================================================

compressed_rows = []

for _, row in usable_group_df.iterrows():
    row_dict = row.to_dict()

    raw_questions = row_dict.get("question_examples", [])
    raw_answers = row_dict.get("answer_texts", [])

    representative_questions = select_representative_questions_fast(
        raw_questions,
        max_items=MAX_QUESTION_EXAMPLES,
        min_len=MIN_QUESTION_LEN,
    )

    representative_answers = select_representative_answers_fast(
        raw_answers,
        max_items=MAX_ANSWER_TEXTS,
        min_len=MIN_ANSWER_LEN,
    )

    new_row = dict(row_dict)
    new_row["raw_question_count"] = len(raw_questions) if isinstance(raw_questions, list) else 0
    new_row["raw_answer_count"] = len(raw_answers) if isinstance(raw_answers, list) else 0
    new_row["representative_questions"] = representative_questions
    new_row["representative_answers"] = representative_answers
    new_row["representative_question_count"] = len(representative_questions)
    new_row["representative_answer_count"] = len(representative_answers)

    compressed_rows.append(new_row)

compressed_group_df = pd.DataFrame(compressed_rows)

print("[INFO] compressed_group_df shape:", compressed_group_df.shape)

display(
    compressed_group_df[
        [
            "disease_kor", "intention",
            "raw_question_count", "representative_question_count",
            "raw_answer_count", "representative_answer_count"
        ]
    ].head(10)
)

[INFO] compressed_group_df shape: (139, 19)


,disease_kor,intention,raw_question_count,representative_question_count,raw_answer_count,representative_answer_count
0,가와사키병,약물,406,12,1358,12
1,가와사키병,예방,221,12,160,12
2,가와사키병,원인,340,12,1466,12
3,가와사키병,정의,275,12,1415,12
4,가와사키병,증상,335,12,160,12
5,가와사키병,진단,403,12,160,12
6,가와사키병,치료,337,12,160,12
7,과숙아,원인,267,12,160,12
8,과숙아,정의,366,12,160,12
9,과숙아,증상,274,12,160,12


In [57]:
# ============================================================
# [STEP 3-0 - CELL B] 대표 질문/답변 압축용 함수 정의
# ------------------------------------------------------------
# 이 셀에서는 그룹 내부 내용을 압축하기 위한 규칙 기반 함수를 정의한다.
#
# [질문 압축 전략]
# - 중복 제거
# - 너무 짧은 질문 제거
# - 길이와 표현 다양성을 고려하여 최대 N개만 선택
#
# [답변 압축 전략]
# - 중복 제거
# - 너무 짧은 답변 제거
# - 지나치게 유사한 답변 제거 (문자열 유사도 기반)
# - 최대 N개만 유지
# ------------------------------------------------------------
# 초기 버전은 과도하게 복잡하게 가지 않고, 규칙 기반으로 안정적으로 압축한다.
# ============================================================

from difflib import SequenceMatcher

# -----------------------------
# 설정값
# -----------------------------
MAX_QUESTION_EXAMPLES = 12
MAX_ANSWER_TEXTS = 12

MIN_QUESTION_LEN = 8
MIN_ANSWER_LEN = 30

QUESTION_SIM_THRESHOLD = 0.90
ANSWER_SIM_THRESHOLD = 0.92

# -----------------------------
# 유사도 함수
# -----------------------------
def text_similarity(a, b):
    a = clean_text(a)
    b = clean_text(b)
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()

# -----------------------------
# 중복/유사 문장 제거 함수
# -----------------------------
def dedup_similar_texts(texts, sim_threshold=0.92, min_len=1):
    cleaned = []
    for t in texts:
        t = clean_text(t)
        if len(t) < min_len:
            continue
        if t not in cleaned:
            cleaned.append(t)

    selected = []
    for t in cleaned:
        is_too_similar = False
        for s in selected:
            if text_similarity(t, s) >= sim_threshold:
                is_too_similar = True
                break
        if not is_too_similar:
            selected.append(t)

    return selected

# -----------------------------
# 질문 대표 샘플 선택
# -----------------------------
def select_representative_questions(question_list,
                                    max_items=12,
                                    min_len=8,
                                    sim_threshold=0.90):
    questions = [clean_text(q) for q in question_list if clean_text(q)]
    questions = [q for q in questions if len(q) >= min_len]

    # 1차: exact 중복 제거
    questions = list(dict.fromkeys(questions))

    # 2차: 유사 질문 제거
    questions = dedup_similar_texts(
        questions,
        sim_threshold=sim_threshold,
        min_len=min_len
    )

    if len(questions) <= max_items:
        return questions

    # 3차: 길이 다양성 반영
    # 너무 비슷한 길이의 질문만 남지 않도록 길이순 정렬 후
    # 앞/중간/뒤를 섞어서 뽑는다.
    questions_sorted = sorted(questions, key=len)

    idxs = np.linspace(0, len(questions_sorted) - 1, max_items, dtype=int)
    sampled = [questions_sorted[i] for i in idxs]

    # 순서 중복 제거
    sampled = list(dict.fromkeys(sampled))
    return sampled[:max_items]

# -----------------------------
# 답변 대표 샘플 선택
# -----------------------------
def select_representative_answers(answer_list,
                                  max_items=12,
                                  min_len=30,
                                  sim_threshold=0.92):
    answers = [clean_text(a) for a in answer_list if clean_text(a)]
    answers = [a for a in answers if len(a) >= min_len]

    # 1차: exact 중복 제거
    answers = list(dict.fromkeys(answers))

    # 2차: 유사 답변 제거
    answers = dedup_similar_texts(
        answers,
        sim_threshold=sim_threshold,
        min_len=min_len
    )

    if len(answers) <= max_items:
        return answers

    # 3차: 길이 기준으로 정보량이 있는 답변 우선
    # 너무 짧은 것보다 적절히 긴 답변이 대표성이 좋을 수 있어
    # 길이 내림차순 상위 일부를 먼저 고려
    answers_sorted = sorted(answers, key=len, reverse=True)

    selected = answers_sorted[:max_items]
    return selected

print("[INFO] 압축 함수 정의 완료")
print(f"[INFO] MAX_QUESTION_EXAMPLES = {MAX_QUESTION_EXAMPLES}")
print(f"[INFO] MAX_ANSWER_TEXTS      = {MAX_ANSWER_TEXTS}")

[INFO] 압축 함수 정의 완료
[INFO] MAX_QUESTION_EXAMPLES = 12
[INFO] MAX_ANSWER_TEXTS      = 12


In [69]:
# ============================================================
# [STEP 3-0 - CELL D] 압축 결과 점검
# ------------------------------------------------------------
# 압축 전/후 질문 수와 답변 수가 어떻게 줄었는지 확인한다.
# 일부 샘플 그룹을 육안으로 검토하여 대표 질문/대표 답변이 적절한지 살핀다.
# ============================================================

print("[INFO] 대표 질문 개수 통계]")
print(compressed_group_df["representative_question_count"].describe())

print("\n[INFO] 대표 답변 개수 통계]")
print(compressed_group_df["representative_answer_count"].describe())

print("\n[INFO] 답변 압축이 많이 된 그룹 TOP 10]")
tmp_df = compressed_group_df.copy()
tmp_df["answer_reduction"] = tmp_df["raw_answer_count"] - tmp_df["representative_answer_count"]

display(
    tmp_df[
        [
            "disease_kor", "intention",
            "raw_answer_count", "representative_answer_count",
            "answer_reduction"
        ]
    ]
    .sort_values("answer_reduction", ascending=False)
    .head(10)
)

# 샘플 그룹 2개 정도 육안 점검
N_SHOW = 2

for i in range(min(N_SHOW, len(compressed_group_df))):
    row = compressed_group_df.iloc[i]

    print("=" * 120)
    print(f"[샘플 그룹 {i+1}] {row['disease_kor']} / {row['intention']}")
    print("-" * 120)
    print(f"원본 질문 수: {row['raw_question_count']} -> 대표 질문 수: {row['representative_question_count']}")
    print(f"원본 답변 수: {row['raw_answer_count']} -> 대표 답변 수: {row['representative_answer_count']}")

    print("\n[대표 질문]")
    for q in row["representative_questions"][:10]:
        print(" -", q)

    print("\n[대표 답변 샘플]")
    for j, a in enumerate(row["representative_answers"][:3], start=1):
        print(f" ({j}) {textwrap.shorten(a, width=300, placeholder=' ...')}")
    print()

[INFO] 대표 질문 개수 통계]
count    139.0
mean      12.0
std        0.0
min       12.0
25%       12.0
50%       12.0
75%       12.0
max       12.0
Name: representative_question_count, dtype: float64

[INFO] 대표 답변 개수 통계]
count    139.0
mean      12.0
std        0.0
min       12.0
25%       12.0
50%       12.0
75%       12.0
max       12.0
Name: representative_answer_count, dtype: float64

[INFO] 답변 압축이 많이 된 그룹 TOP 10]


,disease_kor,intention,raw_answer_count,representative_answer_count,answer_reduction
94,아스퍼거 증후군,약물,3637,12,3625
48,성조숙증,원인,3492,12,3480
128,저신장증,치료,3456,12,3444
32,발달 장애,원인,3375,12,3363
124,저신장증,원인,2741,12,2729
36,발달 장애,치료,2490,12,2478
2,가와사키병,원인,1466,12,1454
81,신생아 호흡곤란증후군,치료,1440,12,1428
3,가와사키병,정의,1415,12,1403
0,가와사키병,약물,1358,12,1346


[샘플 그룹 1] 가와사키병 / 약물
------------------------------------------------------------------------------------------------------------------------
원본 질문 수: 406 -> 대표 질문 수: 12
원본 답변 수: 1358 -> 대표 답변 수: 12

[대표 질문]
 - 가와사키병의 약물치료가 효과적인가요?
 - 약물치료를 받을 때 주의해야 할 부작용이 있을까요?
 - 가와사키병 치료를 위해 어떤 종류의 약물이 효과적일까요?
 - 가와사키병 치료에 도움이 되는 약물은 어떤 것들이 있나요?
 - 가와사키병의 치료에 사용되는 약물의 종류와 특징은 무엇인가요?
 - 가와사키병의 약물 치료 방법에는 어떤 것들이 있는지 알려주세요.
 - 가와사키병을 효과적으로 치료하기 위해 필요한 생활 습관이 있을까요?
 - 가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다.
 - 가와사키병 치료를 위해 어떤 약이 부작용을 일으킬 수 있는지 알고 싶어요.
 - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 약물을 처방받으면 좋을까요?

[대표 답변 샘플]
 (1) 가와사키병은 면역 반응 이상으로 인해 발생하는 질환으로, 아직까지 정확한 원인은 알려지지 않았습니다. 면역 반응은 다양한 생리학적 기전에 의해 조절됩니다. 그러나 가와사키병의 증상은 다양하며, 면역 체계가 잘못된 반응을 일으키거나 혈관에 이상이 생기면 발생할 수 있습니다. 가와사키병은 미국에서 10만 명 중 2~3명에게 발병하고, 한국에서도 5만 명 중 1명이 이 질병으로 진단됩니다. 가와사키병은 주로 5세 미만의 소아에서 발생하며, 일본에서는 4만 명 중 1명의 빈도로 나타납니다. 가와사키병의 증상에는 발열, 발진, 홍조 ...
 (2) 가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및 대증 치료로 이뤄집니다. 가와사키병의 치료에는 대증요법이 일반적으로 사용됩니다

In [71]:
# ============================================================
# [STEP 3 - CELL 1] 압축된 그룹을 사용한 RAG 문서 생성 함수
# ------------------------------------------------------------
# STEP 3-0에서 만든 representative_questions / representative_answers를
# 사용하여 실제 RAG 검색용 문서를 생성한다.
# ============================================================

def build_rag_document(row):
    disease_category = clean_text(row.get("disease_category", ""))
    disease_kor = clean_text(row.get("disease_kor", ""))
    disease_eng = clean_text(row.get("disease_eng", ""))
    intention = clean_text(row.get("intention", ""))
    departments = row.get("departments", []) if isinstance(row.get("departments", []), list) else []

    question_examples = row.get("representative_questions", [])
    if not isinstance(question_examples, list) or len(question_examples) == 0:
        question_examples = row.get("question_examples", []) if isinstance(row.get("question_examples", []), list) else []

    answer_texts = row.get("representative_answers", [])
    if not isinstance(answer_texts, list) or len(answer_texts) == 0:
        answer_texts = row.get("answer_texts", []) if isinstance(row.get("answer_texts", []), list) else []

    department_text = ", ".join([clean_text(x) for x in departments if clean_text(x)])
    question_block = "\n".join([f"- {clean_text(q)}" for q in question_examples if clean_text(q)])
    answer_block = "\n\n".join([clean_text(a) for a in answer_texts if clean_text(a)])

    full_text = join_nonempty([
        f"[질환분류] {disease_category}" if disease_category else "",
        f"[질환명] {disease_kor}" if disease_kor else "",
        f"[영문명] {disease_eng}" if disease_eng else "",
        f"[진료과] {department_text}" if department_text else "",
        f"[상담주제] {intention}" if intention else "",
        "[대표 질문 예시]",
        question_block,
        "[의료 답변]",
        answer_block,
    ], sep="\n")

    doc_id = f"{disease_kor}__{intention}"
    doc_id = re.sub(r"\s+", "_", doc_id)

    return {
        "doc_id": doc_id,
        "disease_category": disease_category,
        "disease_kor": disease_kor,
        "disease_eng": disease_eng,
        "intention": intention,
        "departments": departments,
        "question_examples": question_examples,
        "answer_texts": answer_texts,
        "question_count": int(row.get("representative_question_count", len(question_examples))),
        "answer_count": int(row.get("representative_answer_count", len(answer_texts))),
        "full_text": full_text,
        "char_len": len(full_text),
        "question_example_count": len(question_examples),
        "answer_text_count": len(answer_texts),
        "raw_question_count": int(row.get("raw_question_count", 0)),
        "raw_answer_count": int(row.get("raw_answer_count", 0)),
    }

print("[INFO] build_rag_document 준비 완료")

[INFO] build_rag_document 준비 완료


In [73]:
# ============================================================
# [STEP 3 - CELL 2] 전체 RAG 문서셋 생성
# ------------------------------------------------------------
# compressed_group_df를 기반으로 실제 RAG용 문서 리스트를 생성한다.
# ============================================================

rag_docs = [build_rag_document(row) for _, row in compressed_group_df.iterrows()]
rag_docs_df = pd.DataFrame(rag_docs)

print("[INFO] rag_docs 개수:", len(rag_docs))
print("[INFO] rag_docs_df shape:", rag_docs_df.shape)

display(rag_docs_df.head(10))

[INFO] rag_docs 개수: 139
[INFO] rag_docs_df shape: (139, 16)


,doc_id,disease_category,disease_kor,disease_eng,intention,departments,question_examples,answer_texts,question_count,answer_count,full_text,char_len,question_example_count,answer_text_count,raw_question_count,raw_answer_count
0,가와사키병__약물,소아청소년질환,가와사키병,Kawasaki Disease,약물,[소아청소년과],"[가와사키병의 약물치료가 효과적인가요?, 약물치료를 받을 때 주의해야 할 부작용이 ...","[가와사키병은 면역 반응 이상으로 인해 발생하는 질환으로, 아직까지 정확한 원인은 ...",12,12,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,9302,12,12,406,1358
1,가와사키병__예방,소아청소년질환,가와사키병,Kawasaki Disease,예방,[소아청소년과],"[가와사키병에 대한 예방법을 알고 싶어요., 가와사키병을 예방하기 위한 방법에 대해...","[가와사키병은 다양한 원인으로 발생할 수 있으며, 여러 가지 유전적, 감염적, 환경...",12,12,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,9236,12,12,221,160
2,가와사키병__원인,소아청소년질환,가와사키병,Kawasaki Disease,원인,[소아청소년과],"[가와사키병의 원인은 무엇인가요?, 가와사키병이 발생하는 주요 원인은 무엇인가요?,...","[가와사키병은 주로 고열, 발적, 부종, 피부 발진 등을 동반하는 소아에서 발생하는...",12,12,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,9570,12,12,340,1466
3,가와사키병__정의,소아청소년질환,가와사키병,Kawasaki Disease,정의,[소아청소년과],"[가와사키병이란 무엇인가요?, 가와사키병이란 정확한 정의를 알려주세요., 가와사키병...","[가와사키병은 일본에서 가장 흔한 질병으로, 겨울에 가장 많이 발생합니다. 코로나바...",12,12,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,10901,12,12,275,1415
4,가와사키병__증상,소아청소년질환,가와사키병,Kawasaki Disease,증상,[소아청소년과],"[가와사키병이란 어떤 질병인가요?, 가와사키병의 증상이 얼마나 심각해질 수 있나요?...",[가와사키병은 일반적으로 생후 3개월부터 6개월까지 가장 흔히 발생하는 소아 질환 ...,12,12,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,8868,12,12,335,160
5,가와사키병__진단,소아청소년질환,가와사키병,Kawasaki Disease,진단,[소아청소년과],"[가와사키병 진료과에 대해 알려주세요., 가와사키병의 진단을 받으면 어떤 조치를 해...","[가와사키병은 주로 5세 이하 소아에게서 발생하는 급성 열성 혈관염으로, 원인은 아...",12,12,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,7503,12,12,403,160
6,가와사키병__치료,소아청소년질환,가와사키병,Kawasaki Disease,치료,[소아청소년과],"[가와사키병 치료 방법은 무엇인가요?, 급성 열성 발진증의 치료 기간은 얼마나 걸리...",[가와사키병은 관상동맥의 염증으로 인한 합병증을 예방하기 위한 치료법이 존재합니다....,12,12,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,8468,12,12,337,160
7,과숙아__원인,소아청소년질환,과숙아,Post-term infant,원인,[소아청소년과],"[과숙아의 원인이 무엇인가요?, 과숙아로 태어나는 이유에 대해 알려주세요., 과숙아...",[과숙아는 미숙아와는 달리 배아의 발달 기간이 3배 더 연장되는 것으로 알려져 있습...,12,12,[질환분류] 소아청소년질환\n[질환명] 과숙아\n[영문명] Post-term inf...,7261,12,12,267,160
8,과숙아__정의,소아청소년질환,과숙아,Post-term infant,정의,[소아청소년과],"[과숙아는 어떤 질병인가요?, 과숙아의 특징과 증상에 대해 알려주세요., 과숙아의 ...",[과숙아는 태어난 아기가 정상 출생일보다 2주 이상 늦게 태어나는 아기를 말합니다....,12,12,[질환분류] 소아청소년질환\n[질환명] 과숙아\n[영문명] Post-term inf...,7789,12,12,366,160
9,과숙아__증상,소아청소년질환,과숙아,Post-term infant,증상,[소아청소년과],"[과숙아의 증상에 대해 알려주세요., 과숙아라는 질병의 특징과 증상을 알려주세요.,...",[과숙아는 임신 기간이 정상 임신의 42주(294일)보다 더 오래 유지되는 상태를 ...,12,12,[질환분류] 소아청소년질환\n[질환명] 과숙아\n[영문명] Post-term inf...,9037,12,12,274,160


In [75]:
# ============================================================
# [STEP 3 - CELL 3] 문서 길이 및 분포 점검
# ------------------------------------------------------------
# 생성된 문서 길이와 질문/답변 개수를 확인하여
# 현재 문서 단위가 RAG baseline에 적절한지 점검한다.
# ============================================================

if len(rag_docs_df) > 0:
    print("[INFO] char_len 통계")
    print(rag_docs_df["char_len"].describe())

    print("\n[INFO] question_example_count 통계")
    print(rag_docs_df["question_example_count"].describe())

    print("\n[INFO] answer_text_count 통계")
    print(rag_docs_df["answer_text_count"].describe())

    print("\n[INFO] 가장 긴 문서 TOP 15]")
    display(
        rag_docs_df[
            ["doc_id", "disease_kor", "intention", "char_len",
             "question_example_count", "answer_text_count",
             "raw_question_count", "raw_answer_count"]
        ]
        .sort_values("char_len", ascending=False)
        .head(15)
    )
else:
    print("[WARN] rag_docs_df가 비어 있습니다.")

[INFO] char_len 통계
count      139.000000
mean      8114.726619
std       1225.912017
min       6118.000000
25%       7321.500000
50%       7843.000000
75%       8868.500000
max      13085.000000
Name: char_len, dtype: float64

[INFO] question_example_count 통계
count    139.0
mean      12.0
std        0.0
min       12.0
25%       12.0
50%       12.0
75%       12.0
max       12.0
Name: question_example_count, dtype: float64

[INFO] answer_text_count 통계
count    139.0
mean      12.0
std        0.0
min       12.0
25%       12.0
50%       12.0
75%       12.0
max       12.0
Name: answer_text_count, dtype: float64

[INFO] 가장 긴 문서 TOP 15]


,doc_id,disease_kor,intention,char_len,question_example_count,answer_text_count,raw_question_count,raw_answer_count
48,성조숙증__원인,성조숙증,원인,13085,12,12,320,3492
49,성조숙증__정의,성조숙증,정의,11823,12,12,317,1269
94,아스퍼거_증후군__약물,아스퍼거 증후군,약물,11083,12,12,343,3637
3,가와사키병__정의,가와사키병,정의,10901,12,12,275,1415
110,"유당분해효소결핍증__식이,_생활",유당분해효소결핍증,"식이, 생활",10737,12,12,288,1188
32,발달_장애__원인,발달 장애,원인,10699,12,12,274,3375
128,저신장증__치료,저신장증,치료,10435,12,12,302,3456
121,유행성_이하선염__증상,유행성 이하선염,증상,10430,12,12,137,160
54,성홍열__약물,성홍열,약물,10242,12,12,350,1325
63,신경모세포종__증상,신경모세포종,증상,10193,12,12,233,160


In [77]:
# ============================================================
# [STEP 3 - CELL 4] 샘플 문서 육안 검토
# ------------------------------------------------------------
# 실제 full_text가 검색용 문서로 적절한지 확인한다.
# ============================================================

N_SHOW = 3

for i, doc in enumerate(rag_docs[:N_SHOW], start=1):
    print("=" * 120)
    print(f"[문서 {i}] doc_id = {doc['doc_id']}")
    print("-" * 120)
    print(textwrap.shorten(doc["full_text"], width=1800, placeholder=" ... [생략]"))
    print()

[문서 1] doc_id = 가와사키병__약물
------------------------------------------------------------------------------------------------------------------------
[질환분류] 소아청소년질환 [질환명] 가와사키병 [영문명] Kawasaki Disease [진료과] 소아청소년과 [상담주제] 약물 [대표 질문 예시] - 가와사키병의 약물치료가 효과적인가요? - 약물치료를 받을 때 주의해야 할 부작용이 있을까요? - 가와사키병 치료를 위해 어떤 종류의 약물이 효과적일까요? - 가와사키병 치료에 도움이 되는 약물은 어떤 것들이 있나요? - 가와사키병의 치료에 사용되는 약물의 종류와 특징은 무엇인가요? - 가와사키병의 약물 치료 방법에는 어떤 것들이 있는지 알려주세요. - 가와사키병을 효과적으로 치료하기 위해 필요한 생활 습관이 있을까요? - 가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다. - 가와사키병 치료를 위해 어떤 약이 부작용을 일으킬 수 있는지 알고 싶어요. - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 약물을 처방받으면 좋을까요? - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 종류의 약물을 처방받으면 좋을까요? - 가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유 중인 상황에 어떤 영향이 미칠 수 있는지 알려주세요. [의료 답변] 가와사키병은 면역 반응 이상으로 인해 발생하는 질환으로, 아직까지 정확한 원인은 알려지지 않았습니다. 면역 반응은 다양한 생리학적 기전에 의해 조절됩니다. 그러나 가와사키병의 증상은 다양하며, 면역 체계가 잘못된 반응을 일으키거나 혈관에 이상이 생기면 발생할 수 있습니다. 가와사키병은 미국에서 10만 명 중 2~3명에게 발병하고, 한국에서도 5만 명 중 1명이 이 질병으로 진단됩니다. 가와사키병은 주로 5세 미만의 소아에서 발생하며, 일본에서는 4만 명 중 1명의 

In [79]:
# ============================================================
# [STEP 3 - CELL 5] RAG 문서셋 저장
# ------------------------------------------------------------
# 이후 단계에서 반복 로드를 쉽게 하기 위해 JSONL, JSON, CSV로 저장한다.
# ============================================================

RAG_DOCS_JSONL_PATH = PROCESSED_DIR / "rag_docs.jsonl"
RAG_DOCS_JSON_PATH  = PROCESSED_DIR / "rag_docs.json"
RAG_DOCS_CSV_PATH   = PROCESSED_DIR / "rag_docs.csv"

save_jsonl(rag_docs, RAG_DOCS_JSONL_PATH)
save_json(rag_docs, RAG_DOCS_JSON_PATH)
rag_docs_df.to_csv(RAG_DOCS_CSV_PATH, index=False, encoding="utf-8-sig")

print("[INFO] 저장 완료")
print(" -", RAG_DOCS_JSONL_PATH)
print(" -", RAG_DOCS_JSON_PATH)
print(" -", RAG_DOCS_CSV_PATH)

[INFO] 저장 완료
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\processed\rag_docs.jsonl
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\processed\rag_docs.json
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\processed\rag_docs.csv


In [81]:
# ============================================================
# [STEP 3 - CELL 6] 저장 결과 재로딩 테스트
# ------------------------------------------------------------
# 저장된 문서셋이 정상적으로 다시 로드되는지 확인한다.
# ============================================================

loaded_rag_docs = load_jsonl(RAG_DOCS_JSONL_PATH)

print("[INFO] loaded_rag_docs 개수:", len(loaded_rag_docs))

if loaded_rag_docs:
    print("\n[샘플 문서 미리보기]")
    print(json.dumps(loaded_rag_docs[0], ensure_ascii=False, indent=2)[:2500])

[INFO] loaded_rag_docs 개수: 139

[샘플 문서 미리보기]
{
  "doc_id": "가와사키병__약물",
  "disease_category": "소아청소년질환",
  "disease_kor": "가와사키병",
  "disease_eng": "Kawasaki Disease",
  "intention": "약물",
  "departments": [
    "소아청소년과"
  ],
  "question_examples": [
    "가와사키병의 약물치료가 효과적인가요?",
    "약물치료를 받을 때 주의해야 할 부작용이 있을까요?",
    "가와사키병 치료를 위해 어떤 종류의 약물이 효과적일까요?",
    "가와사키병 치료에 도움이 되는 약물은 어떤 것들이 있나요?",
    "가와사키병의 치료에 사용되는 약물의 종류와 특징은 무엇인가요?",
    "가와사키병의 약물 치료 방법에는 어떤 것들이 있는지 알려주세요.",
    "가와사키병을 효과적으로 치료하기 위해 필요한 생활 습관이 있을까요?",
    "가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다.",
    "가와사키병 치료를 위해 어떤 약이 부작용을 일으킬 수 있는지 알고 싶어요.",
    "가와사키병 환자의 발열과 결막염이 있을 때, 어떤 약물을 처방받으면 좋을까요?",
    "가와사키병 환자의 발열과 결막염이 있을 때, 어떤 종류의 약물을 처방받으면 좋을까요?",
    "가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유 중인 상황에 어떤 영향이 미칠 수 있는지 알려주세요."
  ],
  "answer_texts": [
    "가와사키병은 면역 반응 이상으로 인해 발생하는 질환으로, 아직까지 정확한 원인은 알려지지 않았습니다. 면역 반응은 다양한 생리학적 기전에 의해 조절됩니다. 그러나 가와사키병의 증상은 다양하며, 면역 체계가 잘못된 반응을 일으키거나 혈관에 이상이 생기면 발생할 수 있습니다. 가와사키병은 미

In [83]:
# ============================================================
# [STEP 3 - CELL 7] STEP 3 요약
# ------------------------------------------------------------
# 여기까지 완료되면:
# - 질환명 + intention 기준 대표 문서셋 139개가 생성되었고
# - 이후 STEP 4에서 평가셋을 구축할 수 있다.
# ============================================================

step3_summary = {
    "rag_doc_count": len(rag_docs),
    "rag_docs_jsonl_path": str(RAG_DOCS_JSONL_PATH),
    "avg_char_len": float(rag_docs_df["char_len"].mean()) if len(rag_docs_df) > 0 else 0.0,
    "max_char_len": int(rag_docs_df["char_len"].max()) if len(rag_docs_df) > 0 else 0,
    "min_char_len": int(rag_docs_df["char_len"].min()) if len(rag_docs_df) > 0 else 0,
}

print(json.dumps(step3_summary, ensure_ascii=False, indent=2))

{
  "rag_doc_count": 139,
  "rag_docs_jsonl_path": "D:\\PyProject\\AIFFEL_AI\\LLM\\NLP\\RAG_proj\\data\\processed\\rag_docs.jsonl",
  "avg_char_len": 8114.726618705036,
  "max_char_len": 13085,
  "min_char_len": 6118
}


#### 데이터가 너무 길고 많아서 2차 정제

In [85]:
# ============================================================
# [STEP 3.5 - CELL 1] intention별 키워드 사전 정의
# ------------------------------------------------------------
# 대표 답변이 현재 intention과 얼마나 잘 맞는지 간단한 규칙 기반으로 점수화하기 위해
# intention별 우선 키워드 / 감점 키워드를 정의한다.
# ------------------------------------------------------------
# 이 셀은 의료적으로 완벽한 분류기가 아니라,
# 현재 데이터셋의 문서 품질을 빠르게 정제하기 위한 lightweight 규칙이다.
# ============================================================

INTENTION_PRIORITY_KEYWORDS = {
    "약물": [
        "약물", "복용", "투여", "처방", "부작용", "아스피린", "면역글로불린",
        "해열제", "소염제", "항생제", "스테로이드", "용량", "복약"
    ],
    "예방": [
        "예방", "예방법", "위생", "손 씻기", "생활 습관", "청결", "환기",
        "예방접종", "백신", "관리", "주의", "막기"
    ],
    "원인": [
        "원인", "유전", "감염", "면역", "위험 요인", "발생 원인",
        "기전", "요인", "유발", "발병"
    ],
    "정의": [
        "정의", "의미", "란", "이란", "무엇", "설명", "개념", "질환", "질병"
    ],
    "증상": [
        "증상", "징후", "발열", "발진", "통증", "구토", "설사", "기침",
        "호흡곤란", "부종", "피부", "홍반", "충혈"
    ],
    "진단": [
        "진단", "검사", "혈액 검사", "소변 검사", "영상 검사", "엑스레이",
        "초음파", "판단", "확인", "검진", "진찰"
    ],
    "치료": [
        "치료", "입원", "수술", "치료법", "경과 관찰", "시술", "회복",
        "관리", "중재", "대처", "완화"
    ],
    "식이, 생활": [
        "식이", "생활", "식사", "음식", "영양", "운동", "수면", "습관",
        "섭취", "생활 습관", "관리"
    ],
    "재활": [
        "재활", "훈련", "회복", "기능 회복", "운동 치료", "치료 계획", "발달"
    ],
    "검진": [
        "검진", "정기 검진", "검사", "선별", "확인", "진찰"
    ]
}

INTENTION_PENALTY_KEYWORDS = {
    "약물": ["예방", "예방법", "원인", "유전", "정의", "무엇인가", "예방접종"],
    "예방": ["약물", "복용", "투여", "처방", "수술"],
    "원인": ["치료", "복용", "투여", "예방법", "재활"],
    "정의": ["복용", "투여", "수술", "재활", "예방법"],
    "증상": ["예방", "예방법", "복용 방법", "수술", "재활"],
    "진단": ["예방법", "예방", "복약", "재활"],
    "치료": ["원인", "유전", "정의", "예방법"],
    "식이, 생활": ["수술", "투여", "영상 검사"],
    "재활": ["예방접종", "원인", "정의"],
    "검진": ["복약", "투여", "재활"]
}

print("[INFO] intention 키워드 사전 준비 완료")
print("[INFO] 지원 intention 목록:", list(INTENTION_PRIORITY_KEYWORDS.keys()))

[INFO] intention 키워드 사전 준비 완료
[INFO] 지원 intention 목록: ['약물', '예방', '원인', '정의', '증상', '진단', '치료', '식이, 생활', '재활', '검진']


In [87]:
# ============================================================
# [STEP 3.5 - CELL 2] 답변 품질 점수 함수 정의
# ------------------------------------------------------------
# 각 답변 텍스트에 대해 아래 기준으로 간단히 점수화한다.
#
# [가점]
# - 현재 intention 관련 키워드 포함
# - 질환명 포함
# - 너무 짧지 않고, 적당한 길이
#
# [감점]
# - 다른 intention 냄새가 강한 키워드
# - 지나치게 일반론적 표현
# - 너무 길거나 너무 짧은 답변
# ============================================================

GENERIC_PENALTY_PHRASES = [
    "정확한 원인은 알려지지 않았습니다",
    "전문가의 도움을 받는 것이 좋습니다",
    "병원을 찾아 진단을 받아야 합니다",
    "상황에 따라 다를 수 있습니다",
    "의사의 지시에 따라",
]

def count_keyword_hits(text, keywords):
    text = clean_text(text)
    return sum(1 for kw in keywords if kw in text)

def score_answer_for_intention(answer_text, intention, disease_kor=""):
    text = clean_text(answer_text)
    if not text:
        return -9999

    score = 0.0
    text_len = len(text)

    # 1) 길이 점수
    if 80 <= text_len <= 1200:
        score += 3.0
    elif 50 <= text_len <= 1800:
        score += 1.5
    else:
        score -= 1.0

    # 2) intention 키워드 가점
    priority_keywords = INTENTION_PRIORITY_KEYWORDS.get(intention, [])
    priority_hits = count_keyword_hits(text, priority_keywords)
    score += priority_hits * 2.0

    # 3) penalty 키워드 감점
    penalty_keywords = INTENTION_PENALTY_KEYWORDS.get(intention, [])
    penalty_hits = count_keyword_hits(text, penalty_keywords)
    score -= penalty_hits * 1.5

    # 4) 질환명 포함 가점
    if disease_kor and disease_kor in text:
        score += 1.5

    # 5) 일반론적 문장 과다 감점
    generic_hits = count_keyword_hits(text, GENERIC_PENALTY_PHRASES)
    score -= generic_hits * 0.7

    # 6) 문장 수가 너무 적으면 감점
    sentence_like_count = len(re.findall(r"[.!?。]|다\.", text))
    if sentence_like_count <= 1:
        score -= 1.0

    return score

In [89]:
# ============================================================
# [STEP 3.5 - CELL 3] 대표 답변 재선정 함수 정의
# ------------------------------------------------------------
# 기존 representative_answers를 입력으로 받아
# intention 정합성이 높은 답변을 우선 선택한다.
#
# 기본 전략:
# - 현재 대표 답변 목록만 재정렬/정제
# - 점수 높은 순 정렬
# - 최종 상위 N개만 선택
# ------------------------------------------------------------
# 우선 최종 답변 수는 8개로 줄인다.
# 필요 시 6 또는 10으로 조절 가능하다.
# ============================================================

FINAL_MAX_ANSWER_TEXTS = 8

def refine_representative_answers(answer_list, intention, disease_kor="", max_items=8):
    if not isinstance(answer_list, list):
        return []

    cleaned = []
    seen = set()

    for a in answer_list:
        a = clean_text(a)
        if not a:
            continue
        if a not in seen:
            seen.add(a)
            cleaned.append(a)

    scored = []
    for a in cleaned:
        s = score_answer_for_intention(a, intention=intention, disease_kor=disease_kor)
        scored.append((a, s))

    scored = sorted(scored, key=lambda x: x[1], reverse=True)

    selected = [a for a, s in scored[:max_items]]

    return selected

print("[INFO] 답변 정제 함수 준비 완료")
print(f"[INFO] FINAL_MAX_ANSWER_TEXTS = {FINAL_MAX_ANSWER_TEXTS}")

[INFO] 답변 정제 함수 준비 완료
[INFO] FINAL_MAX_ANSWER_TEXTS = 8


In [91]:
# ============================================================
# [STEP 3.5 - CELL 4] 그룹별 대표 답변 정제 수행
# ------------------------------------------------------------
# compressed_group_df의 representative_answers를 intention 기준으로 다시 정제하여
# refined_group_df를 생성한다.
# ============================================================

refined_rows = []

for _, row in compressed_group_df.iterrows():
    row_dict = row.to_dict()

    disease_kor = clean_text(row_dict.get("disease_kor", ""))
    intention = clean_text(row_dict.get("intention", ""))

    original_rep_answers = row_dict.get("representative_answers", [])
    refined_answers = refine_representative_answers(
        original_rep_answers,
        intention=intention,
        disease_kor=disease_kor,
        max_items=FINAL_MAX_ANSWER_TEXTS
    )

    new_row = dict(row_dict)
    new_row["pre_refine_answer_count"] = len(original_rep_answers) if isinstance(original_rep_answers, list) else 0
    new_row["refined_answers"] = refined_answers
    new_row["refined_answer_count"] = len(refined_answers)

    refined_rows.append(new_row)

refined_group_df = pd.DataFrame(refined_rows)

print("[INFO] refined_group_df shape:", refined_group_df.shape)

display(
    refined_group_df[
        [
            "disease_kor", "intention",
            "pre_refine_answer_count", "refined_answer_count",
            "raw_answer_count"
        ]
    ].head(10)
)

[INFO] refined_group_df shape: (139, 22)


,disease_kor,intention,pre_refine_answer_count,refined_answer_count,raw_answer_count
0,가와사키병,약물,12,8,1358
1,가와사키병,예방,12,8,160
2,가와사키병,원인,12,8,1466
3,가와사키병,정의,12,8,1415
4,가와사키병,증상,12,8,160
5,가와사키병,진단,12,8,160
6,가와사키병,치료,12,8,160
7,과숙아,원인,12,8,160
8,과숙아,정의,12,8,160
9,과숙아,증상,12,8,160


In [93]:
# ============================================================
# [STEP 3.5 - CELL 5] 정제 결과 품질 점검
# ------------------------------------------------------------
# 정제 전/후 답변 수를 확인하고,
# 샘플 그룹에서 refined_answers가 더 intention에 맞는지 육안 검토한다.
# ============================================================

print("[INFO] refined_answer_count 통계]")
print(refined_group_df["refined_answer_count"].describe())

N_SHOW = 2

for i in range(min(N_SHOW, len(refined_group_df))):
    row = refined_group_df.iloc[i]

    print("=" * 120)
    print(f"[샘플 그룹 {i+1}] {row['disease_kor']} / {row['intention']}")
    print("-" * 120)
    print(f"정제 전 대표 답변 수: {row['pre_refine_answer_count']}")
    print(f"정제 후 대표 답변 수: {row['refined_answer_count']}")

    print("\n[정제 후 대표 답변]")
    for j, a in enumerate(row["refined_answers"][:5], start=1):
        print(f" ({j}) {textwrap.shorten(a, width=350, placeholder=' ...')}")
    print()

[INFO] refined_answer_count 통계]
count    139.0
mean       8.0
std        0.0
min        8.0
25%        8.0
50%        8.0
75%        8.0
max        8.0
Name: refined_answer_count, dtype: float64
[샘플 그룹 1] 가와사키병 / 약물
------------------------------------------------------------------------------------------------------------------------
정제 전 대표 답변 수: 12
정제 후 대표 답변 수: 8

[정제 후 대표 답변]
 (1) 가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및 대증 치료로 이뤄집니다. 가와사키병의 치료에는 대증요법이 일반적으로 사용됩니다. 대부분의 소아는 열이나 구토와 같은 증상으로 인해 체온을 떨어뜨리기 위해 아스피린이나 아세트아미노펜과 같은 비스테로이드성 소염제를 사용합니다. 그러나 심한 증상이 나타나면 소염제를 복용할 수 없으므로, 병의 증상이 심각해지기 전까지는 입원을 통해 적절한 치료를 받아야 합니다. 고열이 있는 환자는 아스피린을 복용해야 하며, 이 경우 증상이 더 심해질 수 있습니다. 따라서 체온이 38℃ 이상인 환자에게는 아세트아미노펜이나 해열제를 투여합니다. 만약 발열이 ...
 (2) 가와사키병은 최근에 많이 나타나는 신경계 및 순환기계의 이상증후군입니다. 아직까지 가와사키병의 원인은 명확히 알려져 있지 않습니다. 하지만, 혈액 내에는 다양한 면역반응 이상에 기인하는 면역글로불린이 생성됩니다. 이 면역글로불린은 염증을 일으키고 심장 및 혈관 질환을 유발할 수 있습니다. 가와사키병의 치료에는 다양한 약물이 사용됩니다. 혈액과 체액의 성분을 조절하여 감염을 치료하는 것이 중요합니다. 염증을 억제하기 위해 사용되는 약물 중 가장 흔히 사용

In [95]:
# ============================================================
# [STEP 3.5 - CELL 6] refined_group_df를 사용하는 문서 생성 함수로 교체
# ------------------------------------------------------------
# 기존 build_rag_document()를 refined_answers 우선 사용 버전으로 갱신한다.
# ============================================================

def build_rag_document(row):
    disease_category = clean_text(row.get("disease_category", ""))
    disease_kor = clean_text(row.get("disease_kor", ""))
    disease_eng = clean_text(row.get("disease_eng", ""))
    intention = clean_text(row.get("intention", ""))
    departments = row.get("departments", []) if isinstance(row.get("departments", []), list) else []

    question_examples = row.get("representative_questions", [])
    if not isinstance(question_examples, list) or len(question_examples) == 0:
        question_examples = row.get("question_examples", []) if isinstance(row.get("question_examples", []), list) else []

    # refined_answers 우선 사용
    answer_texts = row.get("refined_answers", [])
    if not isinstance(answer_texts, list) or len(answer_texts) == 0:
        answer_texts = row.get("representative_answers", [])
    if not isinstance(answer_texts, list) or len(answer_texts) == 0:
        answer_texts = row.get("answer_texts", []) if isinstance(row.get("answer_texts", []), list) else []

    department_text = ", ".join([clean_text(x) for x in departments if clean_text(x)])
    question_block = "\n".join([f"- {clean_text(q)}" for q in question_examples if clean_text(q)])
    answer_block = "\n\n".join([clean_text(a) for a in answer_texts if clean_text(a)])

    full_text = join_nonempty([
        f"[질환분류] {disease_category}" if disease_category else "",
        f"[질환명] {disease_kor}" if disease_kor else "",
        f"[영문명] {disease_eng}" if disease_eng else "",
        f"[진료과] {department_text}" if department_text else "",
        f"[상담주제] {intention}" if intention else "",
        "[대표 질문 예시]",
        question_block,
        "[의료 답변]",
        answer_block,
    ], sep="\n")

    doc_id = f"{disease_kor}__{intention}"
    doc_id = re.sub(r"\s+", "_", doc_id)

    return {
        "doc_id": doc_id,
        "disease_category": disease_category,
        "disease_kor": disease_kor,
        "disease_eng": disease_eng,
        "intention": intention,
        "departments": departments,
        "question_examples": question_examples,
        "answer_texts": answer_texts,
        "question_count": int(row.get("representative_question_count", len(question_examples))),
        "answer_count": len(answer_texts),
        "full_text": full_text,
        "char_len": len(full_text),
        "question_example_count": len(question_examples),
        "answer_text_count": len(answer_texts),
        "raw_question_count": int(row.get("raw_question_count", 0)),
        "raw_answer_count": int(row.get("raw_answer_count", 0)),
        "pre_refine_answer_count": int(row.get("pre_refine_answer_count", 0)),
        "refined_answer_count": int(row.get("refined_answer_count", len(answer_texts))),
    }

print("[INFO] refined build_rag_document 준비 완료")

[INFO] refined build_rag_document 준비 완료


In [97]:
# ============================================================
# [STEP 3.5 - CELL 7] 정제된 문서셋 재생성
# ------------------------------------------------------------
# refined_group_df를 기준으로 rag_docs_v2를 생성한다.
# ============================================================

rag_docs_v2 = [build_rag_document(row) for _, row in refined_group_df.iterrows()]
rag_docs_v2_df = pd.DataFrame(rag_docs_v2)

print("[INFO] rag_docs_v2 개수:", len(rag_docs_v2))
print("[INFO] rag_docs_v2_df shape:", rag_docs_v2_df.shape)

display(rag_docs_v2_df.head(10))

[INFO] rag_docs_v2 개수: 139
[INFO] rag_docs_v2_df shape: (139, 18)


,doc_id,disease_category,disease_kor,disease_eng,intention,departments,question_examples,answer_texts,question_count,answer_count,full_text,char_len,question_example_count,answer_text_count,raw_question_count,raw_answer_count,pre_refine_answer_count,refined_answer_count
0,가와사키병__약물,소아청소년질환,가와사키병,Kawasaki Disease,약물,[소아청소년과],"[가와사키병의 약물치료가 효과적인가요?, 약물치료를 받을 때 주의해야 할 부작용이 ...","[가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 ...",12,8,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,6401,12,8,406,1358,12,8
1,가와사키병__예방,소아청소년질환,가와사키병,Kawasaki Disease,예방,[소아청소년과],"[가와사키병에 대한 예방법을 알고 싶어요., 가와사키병을 예방하기 위한 방법에 대해...",[가와사키병은 호흡기 감염 바이러스의 유전적 변이로 인해 발생하는 질병입니다. 이 ...,12,8,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,6414,12,8,221,160,12,8
2,가와사키병__원인,소아청소년질환,가와사키병,Kawasaki Disease,원인,[소아청소년과],"[가와사키병의 원인은 무엇인가요?, 가와사키병이 발생하는 주요 원인은 무엇인가요?,...","[가와사키병은 소아에서 발생하는 흔한 질환으로 알려져 있으며, 주로 소아에서 발열과...",12,8,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,6688,12,8,340,1466,12,8
3,가와사키병__정의,소아청소년질환,가와사키병,Kawasaki Disease,정의,[소아청소년과],"[가와사키병이란 무엇인가요?, 가와사키병이란 정확한 정의를 알려주세요., 가와사키병...",[가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 ...,12,8,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,7678,12,8,275,1415,12,8
4,가와사키병__증상,소아청소년질환,가와사키병,Kawasaki Disease,증상,[소아청소년과],"[가와사키병이란 어떤 질병인가요?, 가와사키병의 증상이 얼마나 심각해질 수 있나요?...","[가와사키병은 5일 이상 지속되는 고열, 손과 발 부종, 다양한 형태의 피부 발진 ...",12,8,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,6271,12,8,335,160,12,8
5,가와사키병__진단,소아청소년질환,가와사키병,Kawasaki Disease,진단,[소아청소년과],"[가와사키병 진료과에 대해 알려주세요., 가와사키병의 진단을 받으면 어떤 조치를 해...","[가와사키병은 원인불명으로 발생하는 심한 열, 안구 결막 충혈, 입술 홍조 및 균열...",12,8,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,5159,12,8,403,160,12,8
6,가와사키병__치료,소아청소년질환,가와사키병,Kawasaki Disease,치료,[소아청소년과],"[가와사키병 치료 방법은 무엇인가요?, 급성 열성 발진증의 치료 기간은 얼마나 걸리...",[가와사키병은 초기에는 관상동맥이 수축하고 혈류장애가 생겨 급사의 위험을 초래할 수...,12,8,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,6205,12,8,337,160,12,8
7,과숙아__원인,소아청소년질환,과숙아,Post-term infant,원인,[소아청소년과],"[과숙아의 원인이 무엇인가요?, 과숙아로 태어나는 이유에 대해 알려주세요., 과숙아...",[과숙아는 여러 가지 원인으로 인해 발생할 수 있습니다. 과숙아가 발생하는 원인은 ...,12,8,[질환분류] 소아청소년질환\n[질환명] 과숙아\n[영문명] Post-term inf...,4868,12,8,267,160,12,8
8,과숙아__정의,소아청소년질환,과숙아,Post-term infant,정의,[소아청소년과],"[과숙아는 어떤 질병인가요?, 과숙아의 특징과 증상에 대해 알려주세요., 과숙아의 ...",[과숙아는 임신 기간이 42주(294일) 이상인 아기를 의미합니다. 이는 산모의 마...,12,8,[질환분류] 소아청소년질환\n[질환명] 과숙아\n[영문명] Post-term inf...,5388,12,8,366,160,12,8
9,과숙아__증상,소아청소년질환,과숙아,Post-term infant,증상,[소아청소년과],"[과숙아의 증상에 대해 알려주세요., 과숙아라는 질병의 특징과 증상을 알려주세요.,...",[과숙아와 정상 만삭아를 구분하는 것은 어렵습니다. 일반적으로 과숙아는 태어나기 전...,12,8,[질환분류] 소아청소년질환\n[질환명] 과숙아\n[영문명] Post-term inf...,6269,12,8,274,160,12,8


In [99]:
# ============================================================
# [STEP 3.5 - CELL 8] 정제 후 문서 길이 및 샘플 확인
# ------------------------------------------------------------
# 정제 후 문서 길이가 줄었는지, 답변 품질이 나아졌는지 확인한다.
# ============================================================

print("[INFO] rag_docs_v2 char_len 통계]")
print(rag_docs_v2_df["char_len"].describe())

print("\n[INFO] 가장 긴 문서 TOP 15]")
display(
    rag_docs_v2_df[
        [
            "doc_id", "disease_kor", "intention", "char_len",
            "answer_text_count", "pre_refine_answer_count", "refined_answer_count"
        ]
    ]
    .sort_values("char_len", ascending=False)
    .head(15)
)

N_SHOW = 2
for i, doc in enumerate(rag_docs_v2[:N_SHOW], start=1):
    print("=" * 120)
    print(f"[문서 {i}] doc_id = {doc['doc_id']}")
    print("-" * 120)
    print(textwrap.shorten(doc["full_text"], width=1800, placeholder=" ... [생략]"))
    print()

[INFO] rag_docs_v2 char_len 통계]
count     139.000000
mean     5677.582734
std       841.277556
min      4237.000000
25%      5132.500000
50%      5517.000000
75%      6185.500000
max      8739.000000
Name: char_len, dtype: float64

[INFO] 가장 긴 문서 TOP 15]


,doc_id,disease_kor,intention,char_len,answer_text_count,pre_refine_answer_count,refined_answer_count
48,성조숙증__원인,성조숙증,원인,8739,8,12,8
49,성조숙증__정의,성조숙증,정의,8119,8,12,8
3,가와사키병__정의,가와사키병,정의,7678,8,12,8
110,"유당분해효소결핍증__식이,_생활",유당분해효소결핍증,"식이, 생활",7655,8,12,8
94,아스퍼거_증후군__약물,아스퍼거 증후군,약물,7623,8,12,8
32,발달_장애__원인,발달 장애,원인,7534,8,12,8
121,유행성_이하선염__증상,유행성 이하선염,증상,7414,8,12,8
128,저신장증__치료,저신장증,치료,7182,8,12,8
36,발달_장애__치료,발달 장애,치료,7169,8,12,8
54,성홍열__약물,성홍열,약물,7035,8,12,8


[문서 1] doc_id = 가와사키병__약물
------------------------------------------------------------------------------------------------------------------------
[질환분류] 소아청소년질환 [질환명] 가와사키병 [영문명] Kawasaki Disease [진료과] 소아청소년과 [상담주제] 약물 [대표 질문 예시] - 가와사키병의 약물치료가 효과적인가요? - 약물치료를 받을 때 주의해야 할 부작용이 있을까요? - 가와사키병 치료를 위해 어떤 종류의 약물이 효과적일까요? - 가와사키병 치료에 도움이 되는 약물은 어떤 것들이 있나요? - 가와사키병의 치료에 사용되는 약물의 종류와 특징은 무엇인가요? - 가와사키병의 약물 치료 방법에는 어떤 것들이 있는지 알려주세요. - 가와사키병을 효과적으로 치료하기 위해 필요한 생활 습관이 있을까요? - 가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다. - 가와사키병 치료를 위해 어떤 약이 부작용을 일으킬 수 있는지 알고 싶어요. - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 약물을 처방받으면 좋을까요? - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 종류의 약물을 처방받으면 좋을까요? - 가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유 중인 상황에 어떤 영향이 미칠 수 있는지 알려주세요. [의료 답변] 가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및 대증 치료로 이뤄집니다. 가와사키병의 치료에는 대증요법이 일반적으로 사용됩니다. 대부분의 소아는 열이나 구토와 같은 증상으로 인해 체온을 떨어뜨리기 위해 아스피린이나 아세트아미노펜과 같은 비스테로이드성 소염제를 사용합니다. 그러나 심한 증상이 나타나면 소염제를 복용할 수 없으므로, 병의 증상이 심각해지기 전까지는 입원을 통해 적절한 치료를 받아야 합니다. 고열이 있는 환자는 아스피린

In [101]:
# ============================================================
# [STEP 3.5 - CELL 9] 정제된 문서셋 저장
# ------------------------------------------------------------
# 이후 평가 및 RAG 실험은 v2 문서셋을 기준으로 진행한다.
# ============================================================

RAG_DOCS_V2_JSONL_PATH = PROCESSED_DIR / "rag_docs_v2.jsonl"
RAG_DOCS_V2_JSON_PATH  = PROCESSED_DIR / "rag_docs_v2.json"
RAG_DOCS_V2_CSV_PATH   = PROCESSED_DIR / "rag_docs_v2.csv"

save_jsonl(rag_docs_v2, RAG_DOCS_V2_JSONL_PATH)
save_json(rag_docs_v2, RAG_DOCS_V2_JSON_PATH)
rag_docs_v2_df.to_csv(RAG_DOCS_V2_CSV_PATH, index=False, encoding="utf-8-sig")

print("[INFO] 저장 완료")
print(" -", RAG_DOCS_V2_JSONL_PATH)
print(" -", RAG_DOCS_V2_JSON_PATH)
print(" -", RAG_DOCS_V2_CSV_PATH)

[INFO] 저장 완료
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\processed\rag_docs_v2.jsonl
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\processed\rag_docs_v2.json
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\processed\rag_docs_v2.csv


In [103]:
# ============================================================
# [STEP 3.5 - CELL 10] STEP 3.5 요약
# ------------------------------------------------------------
# 여기까지 완료되면 정제된 문서셋(v2)이 준비된다.
# 이후 STEP 4 평가셋 구축은 rag_docs_v2 기준으로 진행한다.
# ============================================================

step35_summary = {
    "rag_doc_count_v2": len(rag_docs_v2),
    "rag_docs_v2_jsonl_path": str(RAG_DOCS_V2_JSONL_PATH),
    "avg_char_len_v2": float(rag_docs_v2_df["char_len"].mean()) if len(rag_docs_v2_df) > 0 else 0.0,
    "max_char_len_v2": int(rag_docs_v2_df["char_len"].max()) if len(rag_docs_v2_df) > 0 else 0,
    "min_char_len_v2": int(rag_docs_v2_df["char_len"].min()) if len(rag_docs_v2_df) > 0 else 0,
}

print(json.dumps(step35_summary, ensure_ascii=False, indent=2))

{
  "rag_doc_count_v2": 139,
  "rag_docs_v2_jsonl_path": "D:\\PyProject\\AIFFEL_AI\\LLM\\NLP\\RAG_proj\\data\\processed\\rag_docs_v2.jsonl",
  "avg_char_len_v2": 5677.582733812949,
  "max_char_len_v2": 8739,
  "min_char_len_v2": 4237
}


### step 3.7 데이터가 아직도 길고 많아서 3차 정제
- STEP 3.7 = intention 정합성 재점검 강화 버전

In [110]:
# ============================================================
# [STEP 3.7 - CELL 1] 핵심 intention에 대한 강화 규칙 정의
# ------------------------------------------------------------
# 이번 단계에서는 특히 아래 4개 intention에 대해 정합성 필터를 강화한다.
# - 약물
# - 예방
# - 원인
# - 진단
#
# 질문과 답변 모두에 대해:
# - 우선 키워드
# - 강한 감점 키워드
# - 최소 포함 조건
# 을 적용한다.
# ============================================================

FOCUS_INTENTIONS = ["약물", "예방", "원인", "진단"]

QUESTION_PRIORITY_KEYWORDS = {
    "약물": ["약", "약물", "복용", "투여", "처방", "부작용", "아스피린", "면역글로불린"],
    "예방": ["예방", "예방법", "막", "생활 습관", "위생", "예방접종", "백신", "관리"],
    "원인": ["원인", "왜", "이유", "유전", "감염", "면역", "발생", "유발"],
    "진단": ["진단", "검사", "확인", "판단", "혈액", "소변", "영상", "검진"],

    "정의": ["정의", "뜻", "무엇", "이란", "란", "의미"],
    "증상": ["증상", "징후", "발열", "발진", "통증", "기침", "구토", "설사"],
    "치료": ["치료", "치료법", "관리", "수술", "입원", "완화"],
    "식이, 생활": ["식이", "생활", "식사", "음식", "영양", "운동", "수면"],
    "재활": ["재활", "훈련", "회복", "기능 회복"],
    "검진": ["검진", "정기 검진", "선별 검사"]
}

QUESTION_PENALTY_KEYWORDS = {
    "약물": ["예방", "예방법", "원인", "정의", "검진"],
    "예방": ["약물", "복용", "투여", "처방", "수술", "진단"],
    "원인": ["치료", "복용", "예방", "예방법", "진단 방법"],
    "진단": ["복용", "예방법", "예방", "원인으로", "생활 습관"],

    "정의": ["복용", "예방", "검사 방법"],
    "증상": ["예방법", "복용 방법", "수술"],
    "치료": ["원인", "예방법", "정의"],
    "식이, 생활": ["수술", "투여", "영상 검사"],
    "재활": ["예방접종", "원인"],
    "검진": ["복약", "재활", "수술"]
}

ANSWER_STRONG_PENALTY_KEYWORDS = {
    "약물": ["예방법", "예방접종", "원인은", "정의는", "무엇인가", "생활 습관만", "수술적 치료"],
    "예방": ["복용", "투여", "처방", "아스피린", "면역글로불린", "수술", "진단을 위해"],
    "원인": ["복용", "투여", "예방법", "예방접종", "수술", "진단 검사는"],
    "진단": ["복용", "투여", "예방법", "예방접종", "원인은", "생활 습관"],

    "정의": [],
    "증상": [],
    "치료": [],
    "식이, 생활": [],
    "재활": [],
    "검진": []
}

print("[INFO] STEP 3.7 강화 규칙 준비 완료")
print("[INFO] FOCUS_INTENTIONS:", FOCUS_INTENTIONS)

[INFO] STEP 3.7 강화 규칙 준비 완료
[INFO] FOCUS_INTENTIONS: ['약물', '예방', '원인', '진단']


In [112]:
# ============================================================
# [STEP 3.7 - CELL 2] 질문 정합성 점수 함수
# ------------------------------------------------------------
# 질문이 현재 intention과 얼마나 잘 맞는지 점수화한다.
# 특히 focus intention(약물/예방/원인/진단)은 더 엄격하게 본다.
# ============================================================

def score_question_for_intention(question_text, intention, disease_kor=""):
    text = clean_text(question_text)
    if not text:
        return -9999

    score = 0.0
    text_len = len(text)

    # 길이
    if 8 <= text_len <= 120:
        score += 2.0
    elif 8 <= text_len <= 180:
        score += 1.0
    else:
        score -= 0.5

    # 우선 키워드
    priority_keywords = QUESTION_PRIORITY_KEYWORDS.get(intention, [])
    priority_hits = count_keyword_hits(text, priority_keywords)
    score += priority_hits * 2.5

    # 감점 키워드
    penalty_keywords = QUESTION_PENALTY_KEYWORDS.get(intention, [])
    penalty_hits = count_keyword_hits(text, penalty_keywords)
    score -= penalty_hits * 1.7

    # 질환명 포함
    if disease_kor and disease_kor in text:
        score += 1.0

    # focus intention은 최소한 관련 키워드가 없으면 강하게 감점
    if intention in FOCUS_INTENTIONS and priority_hits == 0:
        score -= 3.0

    return score

In [114]:
# ============================================================
# [STEP 3.7 - CELL 3] 강화된 답변 정합성 점수 함수
# ------------------------------------------------------------
# STEP 3.5의 답변 점수 함수보다 더 엄격하게 intention 정합성을 평가한다.
# focus intention에서는 관련 키워드가 없거나, 다른 intention 냄새가 강하면 더 크게 감점한다.
# ============================================================

def score_answer_for_intention_v2(answer_text, intention, disease_kor=""):
    text = clean_text(answer_text)
    if not text:
        return -9999

    score = 0.0
    text_len = len(text)

    # 길이 점수
    if 80 <= text_len <= 900:
        score += 3.0
    elif 60 <= text_len <= 1300:
        score += 1.5
    else:
        score -= 1.0

    # priority 키워드
    priority_keywords = INTENTION_PRIORITY_KEYWORDS.get(intention, [])
    priority_hits = count_keyword_hits(text, priority_keywords)
    score += priority_hits * 2.3

    # 일반 penalty 키워드
    penalty_keywords = INTENTION_PENALTY_KEYWORDS.get(intention, [])
    penalty_hits = count_keyword_hits(text, penalty_keywords)
    score -= penalty_hits * 1.7

    # stronger penalty
    strong_penalty_keywords = ANSWER_STRONG_PENALTY_KEYWORDS.get(intention, [])
    strong_penalty_hits = count_keyword_hits(text, strong_penalty_keywords)
    score -= strong_penalty_hits * 2.5

    # 질환명 포함
    if disease_kor and disease_kor in text:
        score += 1.2

    # 일반론 문장 감점
    generic_hits = count_keyword_hits(text, GENERIC_PENALTY_PHRASES)
    score -= generic_hits * 0.8

    # focus intention은 관련 키워드가 없으면 크게 감점
    if intention in FOCUS_INTENTIONS and priority_hits == 0:
        score -= 4.0

    # 예방 intention 특별 규칙
    if intention == "예방":
        if "예방" not in text and "예방법" not in text and "위생" not in text and "관리" not in text:
            score -= 2.5

    # 진단 intention 특별 규칙
    if intention == "진단":
        if "진단" not in text and "검사" not in text and "혈액" not in text and "검진" not in text:
            score -= 2.5

    # 약물 intention 특별 규칙
    if intention == "약물":
        if "약물" not in text and "복용" not in text and "투여" not in text and "아스피린" not in text and "면역글로불린" not in text:
            score -= 2.5

    # 원인 intention 특별 규칙
    if intention == "원인":
        if "원인" not in text and "유전" not in text and "감염" not in text and "면역" not in text and "발생" not in text:
            score -= 2.5

    return score

In [116]:
# ============================================================
# [STEP 3.7 - CELL 4] 대표 질문 재선정 함수
# ------------------------------------------------------------
# 기존 representative_questions를 대상으로
# intention 정합성이 높은 질문만 다시 선별한다.
# 최종 질문 수는 8개로 줄인다.
# ============================================================

FINAL_MAX_QUESTION_EXAMPLES_V3 = 8
FINAL_MAX_ANSWER_TEXTS_V3 = 6

def refine_representative_questions_v3(question_list, intention, disease_kor="", max_items=8):
    if not isinstance(question_list, list):
        return []

    cleaned = []
    seen = set()

    for q in question_list:
        q = clean_text(q)
        if not q:
            continue
        if q not in seen:
            seen.add(q)
            cleaned.append(q)

    scored = []
    for q in cleaned:
        s = score_question_for_intention(q, intention=intention, disease_kor=disease_kor)
        scored.append((q, s))

    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    selected = [q for q, s in scored[:max_items]]

    return selected

In [118]:
# ============================================================
# [STEP 3.7 - CELL 5] 대표 답변 재선정 함수(v3)
# ------------------------------------------------------------
# STEP 3.5보다 더 엄격한 기준으로 답변을 다시 선별한다.
# 최종 답변 수는 6개로 줄인다.
# ============================================================

def refine_representative_answers_v3(answer_list, intention, disease_kor="", max_items=6):
    if not isinstance(answer_list, list):
        return []

    cleaned = []
    seen = set()

    for a in answer_list:
        a = clean_text(a)
        if not a:
            continue
        if a not in seen:
            seen.add(a)
            cleaned.append(a)

    scored = []
    for a in cleaned:
        s = score_answer_for_intention_v2(a, intention=intention, disease_kor=disease_kor)
        scored.append((a, s))

    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    selected = [a for a, s in scored[:max_items]]

    return selected

print("[INFO] STEP 3.7 질문/답변 재선정 함수 준비 완료")
print(f"[INFO] FINAL_MAX_QUESTION_EXAMPLES_V3 = {FINAL_MAX_QUESTION_EXAMPLES_V3}")
print(f"[INFO] FINAL_MAX_ANSWER_TEXTS_V3      = {FINAL_MAX_ANSWER_TEXTS_V3}")

[INFO] STEP 3.7 질문/답변 재선정 함수 준비 완료
[INFO] FINAL_MAX_QUESTION_EXAMPLES_V3 = 8
[INFO] FINAL_MAX_ANSWER_TEXTS_V3      = 6


In [120]:
# ============================================================
# [STEP 3.7 - CELL 6] 그룹별 질문/답변 재정제 수행
# ------------------------------------------------------------
# refined_group_df를 입력으로 받아
# 질문과 답변을 한 번 더 정제한 refined_group_df_v3를 만든다.
# ============================================================

refined_rows_v3 = []

for _, row in refined_group_df.iterrows():
    row_dict = row.to_dict()

    disease_kor = clean_text(row_dict.get("disease_kor", ""))
    intention = clean_text(row_dict.get("intention", ""))

    # 질문: representative_questions 기준 재선정
    original_questions = row_dict.get("representative_questions", [])
    refined_questions_v3 = refine_representative_questions_v3(
        original_questions,
        intention=intention,
        disease_kor=disease_kor,
        max_items=FINAL_MAX_QUESTION_EXAMPLES_V3
    )

    # 답변: refined_answers 기준 재선정
    original_answers = row_dict.get("refined_answers", [])
    if not isinstance(original_answers, list) or len(original_answers) == 0:
        original_answers = row_dict.get("representative_answers", [])

    refined_answers_v3 = refine_representative_answers_v3(
        original_answers,
        intention=intention,
        disease_kor=disease_kor,
        max_items=FINAL_MAX_ANSWER_TEXTS_V3
    )

    new_row = dict(row_dict)
    new_row["pre_refine_question_count_v3"] = len(original_questions) if isinstance(original_questions, list) else 0
    new_row["pre_refine_answer_count_v3"] = len(original_answers) if isinstance(original_answers, list) else 0
    new_row["refined_questions_v3"] = refined_questions_v3
    new_row["refined_answers_v3"] = refined_answers_v3
    new_row["refined_question_count_v3"] = len(refined_questions_v3)
    new_row["refined_answer_count_v3"] = len(refined_answers_v3)

    refined_rows_v3.append(new_row)

refined_group_df_v3 = pd.DataFrame(refined_rows_v3)

print("[INFO] refined_group_df_v3 shape:", refined_group_df_v3.shape)

display(
    refined_group_df_v3[
        [
            "disease_kor", "intention",
            "pre_refine_question_count_v3", "refined_question_count_v3",
            "pre_refine_answer_count_v3", "refined_answer_count_v3"
        ]
    ].head(10)
)

[INFO] refined_group_df_v3 shape: (139, 28)


,disease_kor,intention,pre_refine_question_count_v3,refined_question_count_v3,pre_refine_answer_count_v3,refined_answer_count_v3
0,가와사키병,약물,12,8,8,6
1,가와사키병,예방,12,8,8,6
2,가와사키병,원인,12,8,8,6
3,가와사키병,정의,12,8,8,6
4,가와사키병,증상,12,8,8,6
5,가와사키병,진단,12,8,8,6
6,가와사키병,치료,12,8,8,6
7,과숙아,원인,12,8,8,6
8,과숙아,정의,12,8,8,6
9,과숙아,증상,12,8,8,6


In [122]:
# ============================================================
# [STEP 3.7 - CELL 7] v3 정제 결과 점검
# ------------------------------------------------------------
# 질문/답변이 intention에 더 맞게 정제되었는지 샘플 확인한다.
# ============================================================

print("[INFO] refined_question_count_v3 통계]")
print(refined_group_df_v3["refined_question_count_v3"].describe())

print("\n[INFO] refined_answer_count_v3 통계]")
print(refined_group_df_v3["refined_answer_count_v3"].describe())

N_SHOW = 4

for i in range(min(N_SHOW, len(refined_group_df_v3))):
    row = refined_group_df_v3.iloc[i]

    print("=" * 120)
    print(f"[샘플 그룹 {i+1}] {row['disease_kor']} / {row['intention']}")
    print("-" * 120)
    print(f"질문: {row['pre_refine_question_count_v3']} -> {row['refined_question_count_v3']}")
    print(f"답변: {row['pre_refine_answer_count_v3']} -> {row['refined_answer_count_v3']}")

    print("\n[v3 질문]")
    for q in row["refined_questions_v3"][:8]:
        print(" -", q)

    print("\n[v3 답변]")
    for j, a in enumerate(row["refined_answers_v3"][:4], start=1):
        print(f" ({j}) {textwrap.shorten(a, width=320, placeholder=' ...')}")
    print()

[INFO] refined_question_count_v3 통계]
count    139.0
mean       8.0
std        0.0
min        8.0
25%        8.0
50%        8.0
75%        8.0
max        8.0
Name: refined_question_count_v3, dtype: float64

[INFO] refined_answer_count_v3 통계]
count    139.0
mean       6.0
std        0.0
min        6.0
25%        6.0
50%        6.0
75%        6.0
max        6.0
Name: refined_answer_count_v3, dtype: float64
[샘플 그룹 1] 가와사키병 / 약물
------------------------------------------------------------------------------------------------------------------------
질문: 12 -> 8
답변: 8 -> 6

[v3 질문]
 - 가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다.
 - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 약물을 처방받으면 좋을까요?
 - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 종류의 약물을 처방받으면 좋을까요?
 - 약물치료를 받을 때 주의해야 할 부작용이 있을까요?
 - 가와사키병의 약물치료가 효과적인가요?
 - 가와사키병 치료를 위해 어떤 종류의 약물이 효과적일까요?
 - 가와사키병 치료에 도움이 되는 약물은 어떤 것들이 있나요?
 - 가와사키병의 치료에 사용되는 약물의 종류와 특징은 무엇인가요?

[v3 답변]
 (1) 가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및 대증 치료로 이뤄집니다. 가와사키병의 치료에는 대증요법이 일반적으로 사용됩니다. 대부분의 소아

In [124]:
# ============================================================
# [STEP 3.7 - CELL 8] v3용 문서 생성 함수
# ------------------------------------------------------------
# refined_questions_v3 / refined_answers_v3 를 우선 사용하여
# 최종 문서셋(v3)을 만든다.
# ============================================================

def build_rag_document_v3(row):
    disease_category = clean_text(row.get("disease_category", ""))
    disease_kor = clean_text(row.get("disease_kor", ""))
    disease_eng = clean_text(row.get("disease_eng", ""))
    intention = clean_text(row.get("intention", ""))
    departments = row.get("departments", []) if isinstance(row.get("departments", []), list) else []

    question_examples = row.get("refined_questions_v3", [])
    if not isinstance(question_examples, list) or len(question_examples) == 0:
        question_examples = row.get("refined_questions", [])
    if not isinstance(question_examples, list) or len(question_examples) == 0:
        question_examples = row.get("representative_questions", [])

    answer_texts = row.get("refined_answers_v3", [])
    if not isinstance(answer_texts, list) or len(answer_texts) == 0:
        answer_texts = row.get("refined_answers", [])
    if not isinstance(answer_texts, list) or len(answer_texts) == 0:
        answer_texts = row.get("representative_answers", [])

    department_text = ", ".join([clean_text(x) for x in departments if clean_text(x)])
    question_block = "\n".join([f"- {clean_text(q)}" for q in question_examples if clean_text(q)])
    answer_block = "\n\n".join([clean_text(a) for a in answer_texts if clean_text(a)])

    full_text = join_nonempty([
        f"[질환분류] {disease_category}" if disease_category else "",
        f"[질환명] {disease_kor}" if disease_kor else "",
        f"[영문명] {disease_eng}" if disease_eng else "",
        f"[진료과] {department_text}" if department_text else "",
        f"[상담주제] {intention}" if intention else "",
        "[대표 질문 예시]",
        question_block,
        "[의료 답변]",
        answer_block,
    ], sep="\n")

    doc_id = f"{disease_kor}__{intention}"
    doc_id = re.sub(r"\s+", "_", doc_id)

    return {
        "doc_id": doc_id,
        "disease_category": disease_category,
        "disease_kor": disease_kor,
        "disease_eng": disease_eng,
        "intention": intention,
        "departments": departments,
        "question_examples": question_examples,
        "answer_texts": answer_texts,
        "question_count": len(question_examples),
        "answer_count": len(answer_texts),
        "full_text": full_text,
        "char_len": len(full_text),
        "question_example_count": len(question_examples),
        "answer_text_count": len(answer_texts),
        "raw_question_count": int(row.get("raw_question_count", 0)),
        "raw_answer_count": int(row.get("raw_answer_count", 0)),
    }

print("[INFO] build_rag_document_v3 준비 완료")

[INFO] build_rag_document_v3 준비 완료


In [126]:
# ============================================================
# [STEP 3.7 - CELL 9] v3 문서셋 생성
# ------------------------------------------------------------
# refined_group_df_v3를 기준으로 rag_docs_v3를 생성한다.
# ============================================================

rag_docs_v3 = [build_rag_document_v3(row) for _, row in refined_group_df_v3.iterrows()]
rag_docs_v3_df = pd.DataFrame(rag_docs_v3)

print("[INFO] rag_docs_v3 개수:", len(rag_docs_v3))
print("[INFO] rag_docs_v3_df shape:", rag_docs_v3_df.shape)

display(rag_docs_v3_df.head(10))

[INFO] rag_docs_v3 개수: 139
[INFO] rag_docs_v3_df shape: (139, 16)


,doc_id,disease_category,disease_kor,disease_eng,intention,departments,question_examples,answer_texts,question_count,answer_count,full_text,char_len,question_example_count,answer_text_count,raw_question_count,raw_answer_count
0,가와사키병__약물,소아청소년질환,가와사키병,Kawasaki Disease,약물,[소아청소년과],"[가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다., 가와사키...","[가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 ...",8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,4698,8,6,406,1358
1,가와사키병__예방,소아청소년질환,가와사키병,Kawasaki Disease,예방,[소아청소년과],"[가와사키병에 대한 예방법을 알고 싶어요., 가와사키병을 예방하기 위한 생활 습관이...",[가와사키병은 호흡기 감염 바이러스의 유전적 변이로 인해 발생하는 질병입니다. 이 ...,8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,4268,8,6,221,160
2,가와사키병__원인,소아청소년질환,가와사키병,Kawasaki Disease,원인,[소아청소년과],"[가와사키병이 왜 유전적 요인에 의해 발생하는지 알려주세요., 가와사키병의 원인에 ...","[가와사키병은 소아에서 발생하는 흔한 질환으로 알려져 있으며, 주로 소아에서 발열과...",8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,4822,8,6,340,1466
3,가와사키병__정의,소아청소년질환,가와사키병,Kawasaki Disease,정의,[소아청소년과],"[가와사키병이란 무엇인가요?, 가와사키병이란 정확한 정의를 알려주세요., 가와사키병...",[가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 ...,8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,5663,8,6,275,1415
4,가와사키병__증상,소아청소년질환,가와사키병,Kawasaki Disease,증상,[소아청소년과],"[가와사키병의 증상이 얼마나 심각해질 수 있나요?, 가와사키병으로 인해 나타나는 주...","[가와사키병은 5일 이상 지속되는 고열, 손과 발 부종, 다양한 형태의 피부 발진 ...",8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,4622,8,6,335,160
5,가와사키병__진단,소아청소년질환,가와사키병,Kawasaki Disease,진단,[소아청소년과],"[가와사키병 진단을 받으려면 어떤 검사나 진료 과정을 거쳐야 하나요?, 가와사키병의...","[가와사키병은 원인불명으로 발생하는 심한 열, 안구 결막 충혈, 입술 홍조 및 균열...",8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,3761,8,6,403,160
6,가와사키병__치료,소아청소년질환,가와사키병,Kawasaki Disease,치료,[소아청소년과],"[가와사키병을 치료하기 위해 어떤 의약품이나 치료법이 사용되나요?, 가와사키병의 고...",[가와사키병은 초기에는 관상동맥이 수축하고 혈류장애가 생겨 급사의 위험을 초래할 수...,8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,4535,8,6,337,160
7,과숙아__원인,소아청소년질환,과숙아,Post-term infant,원인,[소아청소년과],"[과숙아의 정의와 발생 원인을 자세히 알고 싶어요., 과숙아의 원인이 무엇인가요?,...",[과숙아는 여러 가지 원인으로 인해 발생할 수 있습니다. 과숙아가 발생하는 원인은 ...,8,6,[질환분류] 소아청소년질환\n[질환명] 과숙아\n[영문명] Post-term inf...,3613,8,6,267,160
8,과숙아__정의,소아청소년질환,과숙아,Post-term infant,정의,[소아청소년과],"[과숙아란 무엇인가요? 과숙아와 일반적인 과숙아의 기준은 어떻게 되나요?, 과숙아의...",[과숙아는 임신 기간이 42주(294일) 이상인 아기를 의미합니다. 이는 산모의 마...,8,6,[질환분류] 소아청소년질환\n[질환명] 과숙아\n[영문명] Post-term inf...,3895,8,6,366,160
9,과숙아__증상,소아청소년질환,과숙아,Post-term infant,증상,[소아청소년과],"[과숙아의 증상에 대해 알려주세요., 과숙아라는 질병의 특징과 증상을 알려주세요.,...",[과숙아와 정상 만삭아를 구분하는 것은 어렵습니다. 일반적으로 과숙아는 태어나기 전...,8,6,[질환분류] 소아청소년질환\n[질환명] 과숙아\n[영문명] Post-term inf...,4540,8,6,274,160


In [128]:
# ============================================================
# [STEP 3.7 - CELL 10] v3 문서 길이 및 샘플 점검
# ------------------------------------------------------------
# v2보다 길이가 줄었는지, 내용이 intention에 더 잘 맞는지 확인한다.
# ============================================================

print("[INFO] rag_docs_v3 char_len 통계]")
print(rag_docs_v3_df["char_len"].describe())

print("\n[INFO] 가장 긴 문서 TOP 15]")
display(
    rag_docs_v3_df[
        ["doc_id", "disease_kor", "intention", "char_len", "answer_text_count", "question_example_count"]
    ]
    .sort_values("char_len", ascending=False)
    .head(15)
)

N_SHOW = 4
for i, doc in enumerate(rag_docs_v3[:N_SHOW], start=1):
    print("=" * 120)
    print(f"[문서 {i}] doc_id = {doc['doc_id']}")
    print("-" * 120)
    print(textwrap.shorten(doc["full_text"], width=1800, placeholder=" ... [생략]"))
    print()

[INFO] rag_docs_v3 char_len 통계]
count     139.000000
mean     4218.553957
std       627.738189
min      3202.000000
25%      3794.500000
50%      4089.000000
75%      4604.500000
max      6611.000000
Name: char_len, dtype: float64

[INFO] 가장 긴 문서 TOP 15]


,doc_id,disease_kor,intention,char_len,answer_text_count,question_example_count
48,성조숙증__원인,성조숙증,원인,6611,6,8
49,성조숙증__정의,성조숙증,정의,5828,6,8
3,가와사키병__정의,가와사키병,정의,5663,6,8
32,발달_장애__원인,발달 장애,원인,5632,6,8
94,아스퍼거_증후군__약물,아스퍼거 증후군,약물,5553,6,8
110,"유당분해효소결핍증__식이,_생활",유당분해효소결핍증,"식이, 생활",5551,6,8
36,발달_장애__치료,발달 장애,치료,5440,6,8
128,저신장증__치료,저신장증,치료,5427,6,8
121,유행성_이하선염__증상,유행성 이하선염,증상,5311,6,8
112,유당분해효소결핍증__재활,유당분해효소결핍증,재활,5265,6,8


[문서 1] doc_id = 가와사키병__약물
------------------------------------------------------------------------------------------------------------------------
[질환분류] 소아청소년질환 [질환명] 가와사키병 [영문명] Kawasaki Disease [진료과] 소아청소년과 [상담주제] 약물 [대표 질문 예시] - 가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다. - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 약물을 처방받으면 좋을까요? - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 종류의 약물을 처방받으면 좋을까요? - 약물치료를 받을 때 주의해야 할 부작용이 있을까요? - 가와사키병의 약물치료가 효과적인가요? - 가와사키병 치료를 위해 어떤 종류의 약물이 효과적일까요? - 가와사키병 치료에 도움이 되는 약물은 어떤 것들이 있나요? - 가와사키병의 치료에 사용되는 약물의 종류와 특징은 무엇인가요? [의료 답변] 가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및 대증 치료로 이뤄집니다. 가와사키병의 치료에는 대증요법이 일반적으로 사용됩니다. 대부분의 소아는 열이나 구토와 같은 증상으로 인해 체온을 떨어뜨리기 위해 아스피린이나 아세트아미노펜과 같은 비스테로이드성 소염제를 사용합니다. 그러나 심한 증상이 나타나면 소염제를 복용할 수 없으므로, 병의 증상이 심각해지기 전까지는 입원을 통해 적절한 치료를 받아야 합니다. 고열이 있는 환자는 아스피린을 복용해야 하며, 이 경우 증상이 더 심해질 수 있습니다. 따라서 체온이 38℃ 이상인 환자에게는 아세트아미노펜이나 해열제를 투여합니다. 만약 발열이 심한 경우 타이레놀이나 부루펜과 같은 소염진통제를 사용할 수 있으며, 심한 환자에게는 부루펜이 적합합니다. 또한 소염진통제만으로 조절이 되지 않는 심한 증상이 있는 경우에는 의사의 지시에 따라 항생제를 사용해

In [130]:
# ============================================================
# [STEP 3.7 - CELL 11] v3 문서셋 저장
# ------------------------------------------------------------
# 이후 평가셋 구축과 RAG 실험은 v3 문서셋을 기준으로 진행한다.
# ============================================================

RAG_DOCS_V3_JSONL_PATH = PROCESSED_DIR / "rag_docs_v3.jsonl"
RAG_DOCS_V3_JSON_PATH  = PROCESSED_DIR / "rag_docs_v3.json"
RAG_DOCS_V3_CSV_PATH   = PROCESSED_DIR / "rag_docs_v3.csv"

save_jsonl(rag_docs_v3, RAG_DOCS_V3_JSONL_PATH)
save_json(rag_docs_v3, RAG_DOCS_V3_JSON_PATH)
rag_docs_v3_df.to_csv(RAG_DOCS_V3_CSV_PATH, index=False, encoding="utf-8-sig")

print("[INFO] 저장 완료")
print(" -", RAG_DOCS_V3_JSONL_PATH)
print(" -", RAG_DOCS_V3_JSON_PATH)
print(" -", RAG_DOCS_V3_CSV_PATH)

[INFO] 저장 완료
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\processed\rag_docs_v3.jsonl
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\processed\rag_docs_v3.json
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\processed\rag_docs_v3.csv


In [132]:
# ============================================================
# [STEP 3.7 - CELL 12] STEP 3.7 요약
# ------------------------------------------------------------
# 여기까지 완료되면 intention 정합성을 한 번 더 높인 v3 문서셋이 준비된다.
# 이후 STEP 4 평가셋 구축은 rag_docs_v3 기준으로 진행한다.
# ============================================================

step37_summary = {
    "rag_doc_count_v3": len(rag_docs_v3),
    "rag_docs_v3_jsonl_path": str(RAG_DOCS_V3_JSONL_PATH),
    "avg_char_len_v3": float(rag_docs_v3_df["char_len"].mean()) if len(rag_docs_v3_df) > 0 else 0.0,
    "max_char_len_v3": int(rag_docs_v3_df["char_len"].max()) if len(rag_docs_v3_df) > 0 else 0,
    "min_char_len_v3": int(rag_docs_v3_df["char_len"].min()) if len(rag_docs_v3_df) > 0 else 0,
}

print(json.dumps(step37_summary, ensure_ascii=False, indent=2))

{
  "rag_doc_count_v3": 139,
  "rag_docs_v3_jsonl_path": "D:\\PyProject\\AIFFEL_AI\\LLM\\NLP\\RAG_proj\\data\\processed\\rag_docs_v3.jsonl",
  "avg_char_len_v3": 4218.553956834532,
  "max_char_len_v3": 6611,
  "min_char_len_v3": 3202
}


### STEP 3.8 = “고위험 노이즈 제거" 4차

In [135]:
# ============================================================
# [STEP 3.8 - CELL 1] 고위험 노이즈 패턴 정의
# ------------------------------------------------------------
# 이번 단계에서는 "명백히 이상하거나", "현재 질환/intention과 매우 어긋나는"
# 답변을 제거하기 위한 하드 필터 패턴을 정의한다.
#
# 주 목적:
# - 잘못된 병원체/질환 혼입
# - 임신/태반/산모 맥락 혼입
# - 성인 맥락 혼입
# - 과도하게 엉뚱한 일반론 제거
# ============================================================

HIGH_RISK_NOISE_PATTERNS_COMMON = [
    "Mycobacterium tuberculosis",
    "Candida albicans",
    "결핵균",
    "칸디다",
    "태반",
    "산모",
    "임신 20주",
    "임신 24주",
    "임신 28주",
    "임신 중독증",
    "자궁경부",
    "질 분비물",
    "성관계",
    "정자",
    "난자",
    "출산 직후 성인",
]

HIGH_RISK_NOISE_PATTERNS_BY_INTENTION = {
    "약물": [
        "예방접종을 통해 예방",
        "예방접종이 가장 중요",
        "원인은 아직 명확하지",
        "정의하면",
        "무엇인가",
        "유전적으로 발생",
    ],
    "예방": [
        "아스피린을 복용",
        "면역글로불린을 투여",
        "항생제를 투여",
        "스테로이드를 사용",
        "수술로 예방",
        "예방접종이 치료",
    ],
    "원인": [
        "치료를 위해",
        "약물을 복용",
        "면역글로불린을 투여",
        "예방접종으로 막을 수",
        "검사를 통해 치료",
    ],
    "진단": [
        "예방을 위해",
        "예방접종을 해야",
        "약물을 복용",
        "면역글로불린 투여",
        "원인은 유전적",
    ],
}

# 문장 전체는 아니더라도, 위험 신호가 매우 강한 표현
VERY_SUSPICIOUS_PHRASES = [
    "호흡기 감염 바이러스의 유전적 변이",
    "가와사키 바이러스",
    "예방접종으로 예방할 수 있습니다",
    "임신 중 또는 수유 중인 여성",
    "태반을 통해 감염",
]

print("[INFO] STEP 3.8 고위험 노이즈 패턴 준비 완료")

[INFO] STEP 3.8 고위험 노이즈 패턴 준비 완료


In [137]:
# ============================================================
# [STEP 3.8 - CELL 2] 고위험 노이즈 판별 함수
# ------------------------------------------------------------
# answer_text가 현재 intention 기준으로
# "제거할 만큼 위험한 노이즈인지"를 판단한다.
#
# True  -> 제거 대상
# False -> 유지 가능
# ============================================================

def contains_any_pattern(text, patterns):
    text = clean_text(text)
    return any(p in text for p in patterns)

def is_high_risk_noisy_answer(answer_text, intention, disease_kor=""):
    text = clean_text(answer_text)
    if not text:
        return True

    # 1) 공통 위험 패턴
    if contains_any_pattern(text, HIGH_RISK_NOISE_PATTERNS_COMMON):
        return True

    # 2) intention별 위험 패턴
    if contains_any_pattern(text, HIGH_RISK_NOISE_PATTERNS_BY_INTENTION.get(intention, [])):
        return True

    # 3) 매우 의심스러운 표현
    if contains_any_pattern(text, VERY_SUSPICIOUS_PHRASES):
        return True

    # 4) 질환명이 없고, 현재 intention 핵심 키워드도 거의 없으면 제거 후보
    priority_keywords = INTENTION_PRIORITY_KEYWORDS.get(intention, [])
    priority_hits = count_keyword_hits(text, priority_keywords)

    if disease_kor and disease_kor not in text and priority_hits == 0:
        return True

    # 5) 예방 intention인데 예방/위생/관리/예방법이 전혀 없으면 제거
    if intention == "예방":
        if ("예방" not in text and "예방법" not in text and "위생" not in text and "관리" not in text):
            return True

    # 6) 진단 intention인데 검사/진단/혈액/검진이 전혀 없으면 제거
    if intention == "진단":
        if ("진단" not in text and "검사" not in text and "혈액" not in text and "검진" not in text):
            return True

    # 7) 약물 intention인데 약물/복용/투여/아스피린/면역글로불린이 전혀 없으면 제거
    if intention == "약물":
        if ("약물" not in text and "복용" not in text and "투여" not in text and
            "아스피린" not in text and "면역글로불린" not in text):
            return True

    # 8) 원인 intention인데 원인/유전/감염/면역/발생이 전혀 없으면 제거
    if intention == "원인":
        if ("원인" not in text and "유전" not in text and "감염" not in text and
            "면역" not in text and "발생" not in text):
            return True

    return False

print("[INFO] 고위험 노이즈 판별 함수 준비 완료")

[INFO] 고위험 노이즈 판별 함수 준비 완료


In [139]:
# ============================================================
# [STEP 3.8 - CELL 3] v3 답변 하드 필터 적용
# ------------------------------------------------------------
# refined_answers_v3를 대상으로 고위험 노이즈 제거를 수행한다.
#
# 규칙:
# - noisy answer는 제거
# - 너무 많이 제거되어 0개가 되면, 원래 v3 답변 중 점수 상위 일부를 fallback으로 사용
# - 최종 답변 수는 최대 6개 유지
# ============================================================

FINAL_MAX_ANSWER_TEXTS_V4 = 6
FINAL_MIN_ANSWER_TEXTS_V4 = 3

def hard_filter_answers_v4(answer_list, intention, disease_kor="", max_items=6, min_items=3):
    if not isinstance(answer_list, list):
        return []

    cleaned = []
    seen = set()

    for a in answer_list:
        a = clean_text(a)
        if not a:
            continue
        if a not in seen:
            seen.add(a)
            cleaned.append(a)

    # 1차: 하드 필터 적용
    filtered = [
        a for a in cleaned
        if not is_high_risk_noisy_answer(a, intention=intention, disease_kor=disease_kor)
    ]

    # 2차: 너무 적으면 score 기반 fallback
    if len(filtered) < min_items:
        scored = []
        for a in cleaned:
            s = score_answer_for_intention_v2(a, intention=intention, disease_kor=disease_kor)
            scored.append((a, s))

        scored = sorted(scored, key=lambda x: x[1], reverse=True)
        fallback = [a for a, s in scored[:max_items]]

        # fallback에서도 공통 위험 패턴은 한 번 더 제거 시도
        fallback = [
            a for a in fallback
            if not contains_any_pattern(a, HIGH_RISK_NOISE_PATTERNS_COMMON)
        ]

        if len(fallback) >= min_items:
            filtered = fallback
        else:
            filtered = [a for a, s in scored[:max_items]]

    return filtered[:max_items]

print("[INFO] v4 하드 필터 함수 준비 완료")
print(f"[INFO] FINAL_MAX_ANSWER_TEXTS_V4 = {FINAL_MAX_ANSWER_TEXTS_V4}")
print(f"[INFO] FINAL_MIN_ANSWER_TEXTS_V4 = {FINAL_MIN_ANSWER_TEXTS_V4}")

[INFO] v4 하드 필터 함수 준비 완료
[INFO] FINAL_MAX_ANSWER_TEXTS_V4 = 6
[INFO] FINAL_MIN_ANSWER_TEXTS_V4 = 3


In [141]:
# ============================================================
# [STEP 3.8 - CELL 4] 질문 보수적 재점검
# ------------------------------------------------------------
# 질문도 intention과 너무 어긋난 것이 있으면 제거한다.
# 다만 질문은 답변보다 정보 손실이 적으므로 보수적으로 필터링한다.
#
# 최종 질문 수는 최대 8개 유지
# ============================================================

FINAL_MAX_QUESTION_EXAMPLES_V4 = 8
FINAL_MIN_QUESTION_EXAMPLES_V4 = 4

def is_noisy_question_v4(question_text, intention, disease_kor=""):
    text = clean_text(question_text)
    if not text:
        return True

    # 질환명 불포함 + 핵심 intention 키워드 없음
    priority_keywords = QUESTION_PRIORITY_KEYWORDS.get(intention, [])
    priority_hits = count_keyword_hits(text, priority_keywords)

    if disease_kor and disease_kor not in text and priority_hits == 0:
        return True

    # focus intention은 관련 키워드가 없으면 제거
    if intention in FOCUS_INTENTIONS and priority_hits == 0:
        return True

    # 약물 질문인데 예방/원인/정의 쪽이 더 강하면 제거
    penalty_keywords = QUESTION_PENALTY_KEYWORDS.get(intention, [])
    penalty_hits = count_keyword_hits(text, penalty_keywords)
    if penalty_hits >= 2:
        return True

    return False

def hard_filter_questions_v4(question_list, intention, disease_kor="", max_items=8, min_items=4):
    if not isinstance(question_list, list):
        return []

    cleaned = []
    seen = set()

    for q in question_list:
        q = clean_text(q)
        if not q:
            continue
        if q not in seen:
            seen.add(q)
            cleaned.append(q)

    filtered = [
        q for q in cleaned
        if not is_noisy_question_v4(q, intention=intention, disease_kor=disease_kor)
    ]

    if len(filtered) < min_items:
        scored = []
        for q in cleaned:
            s = score_question_for_intention(q, intention=intention, disease_kor=disease_kor)
            scored.append((q, s))
        scored = sorted(scored, key=lambda x: x[1], reverse=True)
        filtered = [q for q, s in scored[:max_items]]

    return filtered[:max_items]

print("[INFO] v4 질문 하드 필터 함수 준비 완료")

[INFO] v4 질문 하드 필터 함수 준비 완료


In [143]:
# ============================================================
# [STEP 3.8 - CELL 5] 그룹별 질문/답변 v4 정제 수행
# ------------------------------------------------------------
# refined_group_df_v3를 기반으로
# - 질문은 hard_filter_questions_v4
# - 답변은 hard_filter_answers_v4
# 를 적용해 refined_group_df_v4를 만든다.
# ============================================================

refined_rows_v4 = []

for _, row in refined_group_df_v3.iterrows():
    row_dict = row.to_dict()

    disease_kor = clean_text(row_dict.get("disease_kor", ""))
    intention = clean_text(row_dict.get("intention", ""))

    original_questions_v3 = row_dict.get("refined_questions_v3", [])
    original_answers_v3 = row_dict.get("refined_answers_v3", [])

    refined_questions_v4 = hard_filter_questions_v4(
        original_questions_v3,
        intention=intention,
        disease_kor=disease_kor,
        max_items=FINAL_MAX_QUESTION_EXAMPLES_V4,
        min_items=FINAL_MIN_QUESTION_EXAMPLES_V4
    )

    refined_answers_v4 = hard_filter_answers_v4(
        original_answers_v3,
        intention=intention,
        disease_kor=disease_kor,
        max_items=FINAL_MAX_ANSWER_TEXTS_V4,
        min_items=FINAL_MIN_ANSWER_TEXTS_V4
    )

    new_row = dict(row_dict)
    new_row["pre_refine_question_count_v4"] = len(original_questions_v3) if isinstance(original_questions_v3, list) else 0
    new_row["pre_refine_answer_count_v4"] = len(original_answers_v3) if isinstance(original_answers_v3, list) else 0
    new_row["refined_questions_v4"] = refined_questions_v4
    new_row["refined_answers_v4"] = refined_answers_v4
    new_row["refined_question_count_v4"] = len(refined_questions_v4)
    new_row["refined_answer_count_v4"] = len(refined_answers_v4)

    refined_rows_v4.append(new_row)

refined_group_df_v4 = pd.DataFrame(refined_rows_v4)

print("[INFO] refined_group_df_v4 shape:", refined_group_df_v4.shape)

display(
    refined_group_df_v4[
        [
            "disease_kor", "intention",
            "pre_refine_question_count_v4", "refined_question_count_v4",
            "pre_refine_answer_count_v4", "refined_answer_count_v4"
        ]
    ].head(10)
)

[INFO] refined_group_df_v4 shape: (139, 34)


,disease_kor,intention,pre_refine_question_count_v4,refined_question_count_v4,pre_refine_answer_count_v4,refined_answer_count_v4
0,가와사키병,약물,8,8,6,6
1,가와사키병,예방,8,8,6,4
2,가와사키병,원인,8,8,6,5
3,가와사키병,정의,8,8,6,6
4,가와사키병,증상,8,8,6,6
5,가와사키병,진단,8,8,6,6
6,가와사키병,치료,8,8,6,6
7,과숙아,원인,8,8,6,3
8,과숙아,정의,8,8,6,6
9,과숙아,증상,8,8,6,6


In [145]:
# ============================================================
# [STEP 3.8 - CELL 6] v4 정제 결과 샘플 점검
# ------------------------------------------------------------
# 특히 약물/예방/원인/진단에서 질문/답변이 더 정돈되었는지 확인한다.
# ============================================================

print("[INFO] refined_question_count_v4 통계]")
print(refined_group_df_v4["refined_question_count_v4"].describe())

print("\n[INFO] refined_answer_count_v4 통계]")
print(refined_group_df_v4["refined_answer_count_v4"].describe())

# 가와사키병 4개 intention 우선 확인
focus_view = refined_group_df_v4[
    (refined_group_df_v4["disease_kor"] == "가와사키병") &
    (refined_group_df_v4["intention"].isin(["약물", "예방", "원인", "진단"]))
].copy()

print("\n[INFO] 가와사키병 핵심 intention 샘플]")
display(
    focus_view[
        [
            "disease_kor", "intention",
            "refined_question_count_v4", "refined_answer_count_v4"
        ]
    ]
)

for _, row in focus_view.iterrows():
    print("=" * 120)
    print(f"[샘플] {row['disease_kor']} / {row['intention']}")
    print("-" * 120)

    print("[v4 질문]")
    for q in row["refined_questions_v4"][:8]:
        print(" -", q)

    print("\n[v4 답변]")
    for i, a in enumerate(row["refined_answers_v4"][:4], start=1):
        print(f" ({i}) {textwrap.shorten(a, width=320, placeholder=' ...')}")
    print()

[INFO] refined_question_count_v4 통계]
count    139.000000
mean       7.985612
std        0.119517
min        7.000000
25%        8.000000
50%        8.000000
75%        8.000000
max        8.000000
Name: refined_question_count_v4, dtype: float64

[INFO] refined_answer_count_v4 통계]
count    139.000000
mean       5.733813
std        0.620607
min        3.000000
25%        6.000000
50%        6.000000
75%        6.000000
max        6.000000
Name: refined_answer_count_v4, dtype: float64

[INFO] 가와사키병 핵심 intention 샘플]


,disease_kor,intention,refined_question_count_v4,refined_answer_count_v4
0,가와사키병,약물,8,6
1,가와사키병,예방,8,4
2,가와사키병,원인,8,5
5,가와사키병,진단,8,6


[샘플] 가와사키병 / 약물
------------------------------------------------------------------------------------------------------------------------
[v4 질문]
 - 가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다.
 - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 약물을 처방받으면 좋을까요?
 - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 종류의 약물을 처방받으면 좋을까요?
 - 약물치료를 받을 때 주의해야 할 부작용이 있을까요?
 - 가와사키병의 약물치료가 효과적인가요?
 - 가와사키병 치료를 위해 어떤 종류의 약물이 효과적일까요?
 - 가와사키병 치료에 도움이 되는 약물은 어떤 것들이 있나요?
 - 가와사키병의 치료에 사용되는 약물의 종류와 특징은 무엇인가요?

[v4 답변]
 (1) 가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및 대증 치료로 이뤄집니다. 가와사키병의 치료에는 대증요법이 일반적으로 사용됩니다. 대부분의 소아는 열이나 구토와 같은 증상으로 인해 체온을 떨어뜨리기 위해 아스피린이나 아세트아미노펜과 같은 비스테로이드성 소염제를 사용합니다. 그러나 심한 증상이 나타나면 소염제를 복용할 수 없으므로, 병의 증상이 심각해지기 전까지는 입원을 통해 적절한 치료를 받아야 합니다. 고열이 있는 환자는 아스피린을 복용해야 하며, 이 경우 증상이 더 심해질 수 있습니다. 따라서 체온이 38℃ 이상인 ...
 (2) 가와사키병은 최근에 많이 나타나는 신경계 및 순환기계의 이상증후군입니다. 아직까지 가와사키병의 원인은 명확히 알려져 있지 않습니다. 하지만, 혈액 내에는 다양한 면역반응 이상에 기인하는 면역글로불린이 생성됩니다. 이 면역글로불린은 염증을 일으키고 심장 및 혈관 질환을 유발할 수 있습니다. 가와사키병의 치료에는 다양한 약물이 사용됩니다. 혈액과 체액의 성분을 조절하여 감염을 치료하는

In [147]:
# ============================================================
# [STEP 3.8 - CELL 7] v4 문서 생성 함수
# ------------------------------------------------------------
# refined_questions_v4 / refined_answers_v4 를 우선 사용하여
# 최종 문서셋(v4)을 만든다.
# ============================================================

def build_rag_document_v4(row):
    disease_category = clean_text(row.get("disease_category", ""))
    disease_kor = clean_text(row.get("disease_kor", ""))
    disease_eng = clean_text(row.get("disease_eng", ""))
    intention = clean_text(row.get("intention", ""))
    departments = row.get("departments", []) if isinstance(row.get("departments", []), list) else []

    question_examples = row.get("refined_questions_v4", [])
    if not isinstance(question_examples, list) or len(question_examples) == 0:
        question_examples = row.get("refined_questions_v3", [])
    if not isinstance(question_examples, list) or len(question_examples) == 0:
        question_examples = row.get("representative_questions", [])

    answer_texts = row.get("refined_answers_v4", [])
    if not isinstance(answer_texts, list) or len(answer_texts) == 0:
        answer_texts = row.get("refined_answers_v3", [])
    if not isinstance(answer_texts, list) or len(answer_texts) == 0:
        answer_texts = row.get("refined_answers", [])
    if not isinstance(answer_texts, list) or len(answer_texts) == 0:
        answer_texts = row.get("representative_answers", [])

    department_text = ", ".join([clean_text(x) for x in departments if clean_text(x)])
    question_block = "\n".join([f"- {clean_text(q)}" for q in question_examples if clean_text(q)])
    answer_block = "\n\n".join([clean_text(a) for a in answer_texts if clean_text(a)])

    full_text = join_nonempty([
        f"[질환분류] {disease_category}" if disease_category else "",
        f"[질환명] {disease_kor}" if disease_kor else "",
        f"[영문명] {disease_eng}" if disease_eng else "",
        f"[진료과] {department_text}" if department_text else "",
        f"[상담주제] {intention}" if intention else "",
        "[대표 질문 예시]",
        question_block,
        "[의료 답변]",
        answer_block,
    ], sep="\n")

    doc_id = f"{disease_kor}__{intention}"
    doc_id = re.sub(r"\s+", "_", doc_id)

    return {
        "doc_id": doc_id,
        "disease_category": disease_category,
        "disease_kor": disease_kor,
        "disease_eng": disease_eng,
        "intention": intention,
        "departments": departments,
        "question_examples": question_examples,
        "answer_texts": answer_texts,
        "question_count": len(question_examples),
        "answer_count": len(answer_texts),
        "full_text": full_text,
        "char_len": len(full_text),
        "question_example_count": len(question_examples),
        "answer_text_count": len(answer_texts),
        "raw_question_count": int(row.get("raw_question_count", 0)),
        "raw_answer_count": int(row.get("raw_answer_count", 0)),
    }

print("[INFO] build_rag_document_v4 준비 완료")

[INFO] build_rag_document_v4 준비 완료


In [149]:
# ============================================================
# [STEP 3.8 - CELL 8] v4 문서셋 생성
# ------------------------------------------------------------
# refined_group_df_v4를 기준으로 rag_docs_v4를 생성한다.
# ============================================================

rag_docs_v4 = [build_rag_document_v4(row) for _, row in refined_group_df_v4.iterrows()]
rag_docs_v4_df = pd.DataFrame(rag_docs_v4)

print("[INFO] rag_docs_v4 개수:", len(rag_docs_v4))
print("[INFO] rag_docs_v4_df shape:", rag_docs_v4_df.shape)

display(rag_docs_v4_df.head(10))

[INFO] rag_docs_v4 개수: 139
[INFO] rag_docs_v4_df shape: (139, 16)


,doc_id,disease_category,disease_kor,disease_eng,intention,departments,question_examples,answer_texts,question_count,answer_count,full_text,char_len,question_example_count,answer_text_count,raw_question_count,raw_answer_count
0,가와사키병__약물,소아청소년질환,가와사키병,Kawasaki Disease,약물,[소아청소년과],"[가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다., 가와사키...","[가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 ...",8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,4698,8,6,406,1358
1,가와사키병__예방,소아청소년질환,가와사키병,Kawasaki Disease,예방,[소아청소년과],"[가와사키병에 대한 예방법을 알고 싶어요., 가와사키병을 예방하기 위한 생활 습관이...","[가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 ...",8,4,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,3081,8,4,221,160
2,가와사키병__원인,소아청소년질환,가와사키병,Kawasaki Disease,원인,[소아청소년과],"[가와사키병이 왜 유전적 요인에 의해 발생하는지 알려주세요., 가와사키병의 원인에 ...","[가와사키병은 몇 가지 가설과 연구 결과가 있지만, 아직까지 확실히 입증된 것은 없...",8,5,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,4067,8,5,340,1466
3,가와사키병__정의,소아청소년질환,가와사키병,Kawasaki Disease,정의,[소아청소년과],"[가와사키병이란 무엇인가요?, 가와사키병이란 정확한 정의를 알려주세요., 가와사키병...",[가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 ...,8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,5663,8,6,275,1415
4,가와사키병__증상,소아청소년질환,가와사키병,Kawasaki Disease,증상,[소아청소년과],"[가와사키병의 증상이 얼마나 심각해질 수 있나요?, 가와사키병으로 인해 나타나는 주...","[가와사키병은 5일 이상 지속되는 고열, 손과 발 부종, 다양한 형태의 피부 발진 ...",8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,4622,8,6,335,160
5,가와사키병__진단,소아청소년질환,가와사키병,Kawasaki Disease,진단,[소아청소년과],"[가와사키병 진단을 받으려면 어떤 검사나 진료 과정을 거쳐야 하나요?, 가와사키병의...","[가와사키병은 원인불명으로 발생하는 심한 열, 안구 결막 충혈, 입술 홍조 및 균열...",8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,3761,8,6,403,160
6,가와사키병__치료,소아청소년질환,가와사키병,Kawasaki Disease,치료,[소아청소년과],"[가와사키병을 치료하기 위해 어떤 의약품이나 치료법이 사용되나요?, 가와사키병의 고...",[가와사키병은 초기에는 관상동맥이 수축하고 혈류장애가 생겨 급사의 위험을 초래할 수...,8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,4535,8,6,337,160
7,과숙아__원인,소아청소년질환,과숙아,Post-term infant,원인,[소아청소년과],"[과숙아의 정의와 발생 원인을 자세히 알고 싶어요., 과숙아의 원인이 무엇인가요?,...",[과숙아는 여러 가지 원인으로 인해 발생할 수 있습니다. 과숙아가 발생하는 원인은 ...,8,3,[질환분류] 소아청소년질환\n[질환명] 과숙아\n[영문명] Post-term inf...,1978,8,3,267,160
8,과숙아__정의,소아청소년질환,과숙아,Post-term infant,정의,[소아청소년과],"[과숙아란 무엇인가요? 과숙아와 일반적인 과숙아의 기준은 어떻게 되나요?, 과숙아의...",[과숙아는 임신 기간이 42주(294일) 이상인 아기를 의미합니다. 이는 산모의 마...,8,6,[질환분류] 소아청소년질환\n[질환명] 과숙아\n[영문명] Post-term inf...,3895,8,6,366,160
9,과숙아__증상,소아청소년질환,과숙아,Post-term infant,증상,[소아청소년과],"[과숙아의 증상에 대해 알려주세요., 과숙아라는 질병의 특징과 증상을 알려주세요.,...",[과숙아와 정상 만삭아를 구분하는 것은 어렵습니다. 일반적으로 과숙아는 태어나기 전...,8,6,[질환분류] 소아청소년질환\n[질환명] 과숙아\n[영문명] Post-term inf...,4540,8,6,274,160


In [151]:
# ============================================================
# [STEP 3.8 - CELL 9] v4 문서 길이 및 핵심 샘플 확인
# ------------------------------------------------------------
# 길이 통계와 핵심 질환 샘플을 확인한다.
# ============================================================

print("[INFO] rag_docs_v4 char_len 통계]")
print(rag_docs_v4_df["char_len"].describe())

print("\n[INFO] 가장 긴 문서 TOP 15]")
display(
    rag_docs_v4_df[
        ["doc_id", "disease_kor", "intention", "char_len", "question_example_count", "answer_text_count"]
    ]
    .sort_values("char_len", ascending=False)
    .head(15)
)

focus_docs_v4 = rag_docs_v4_df[
    (rag_docs_v4_df["disease_kor"] == "가와사키병") &
    (rag_docs_v4_df["intention"].isin(["약물", "예방", "원인", "진단"]))
].copy()

print("\n[INFO] 가와사키병 v4 문서 요약]")
display(
    focus_docs_v4[
        ["doc_id", "char_len", "question_example_count", "answer_text_count"]
    ]
)

for _, row in focus_docs_v4.iterrows():
    print("=" * 120)
    print(f"[문서 샘플] {row['doc_id']}")
    print("-" * 120)
    full_text = row["full_text"]
    print(textwrap.shorten(full_text, width=1500, placeholder=" ... [생략]"))
    print()

[INFO] rag_docs_v4 char_len 통계]
count     139.000000
mean     4051.971223
std       744.259966
min      1978.000000
25%      3597.000000
50%      3940.000000
75%      4569.500000
max      6611.000000
Name: char_len, dtype: float64

[INFO] 가장 긴 문서 TOP 15]


,doc_id,disease_kor,intention,char_len,question_example_count,answer_text_count
48,성조숙증__원인,성조숙증,원인,6611,8,6
49,성조숙증__정의,성조숙증,정의,5828,8,6
3,가와사키병__정의,가와사키병,정의,5663,8,6
32,발달_장애__원인,발달 장애,원인,5632,8,6
94,아스퍼거_증후군__약물,아스퍼거 증후군,약물,5553,8,6
110,"유당분해효소결핍증__식이,_생활",유당분해효소결핍증,"식이, 생활",5551,8,6
36,발달_장애__치료,발달 장애,치료,5440,8,6
128,저신장증__치료,저신장증,치료,5427,8,6
121,유행성_이하선염__증상,유행성 이하선염,증상,5311,8,6
112,유당분해효소결핍증__재활,유당분해효소결핍증,재활,5265,8,6



[INFO] 가와사키병 v4 문서 요약]


,doc_id,char_len,question_example_count,answer_text_count
0,가와사키병__약물,4698,8,6
1,가와사키병__예방,3081,8,4
2,가와사키병__원인,4067,8,5
5,가와사키병__진단,3761,8,6


[문서 샘플] 가와사키병__약물
------------------------------------------------------------------------------------------------------------------------
[질환분류] 소아청소년질환 [질환명] 가와사키병 [영문명] Kawasaki Disease [진료과] 소아청소년과 [상담주제] 약물 [대표 질문 예시] - 가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다. - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 약물을 처방받으면 좋을까요? - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 종류의 약물을 처방받으면 좋을까요? - 약물치료를 받을 때 주의해야 할 부작용이 있을까요? - 가와사키병의 약물치료가 효과적인가요? - 가와사키병 치료를 위해 어떤 종류의 약물이 효과적일까요? - 가와사키병 치료에 도움이 되는 약물은 어떤 것들이 있나요? - 가와사키병의 치료에 사용되는 약물의 종류와 특징은 무엇인가요? [의료 답변] 가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및 대증 치료로 이뤄집니다. 가와사키병의 치료에는 대증요법이 일반적으로 사용됩니다. 대부분의 소아는 열이나 구토와 같은 증상으로 인해 체온을 떨어뜨리기 위해 아스피린이나 아세트아미노펜과 같은 비스테로이드성 소염제를 사용합니다. 그러나 심한 증상이 나타나면 소염제를 복용할 수 없으므로, 병의 증상이 심각해지기 전까지는 입원을 통해 적절한 치료를 받아야 합니다. 고열이 있는 환자는 아스피린을 복용해야 하며, 이 경우 증상이 더 심해질 수 있습니다. 따라서 체온이 38℃ 이상인 환자에게는 아세트아미노펜이나 해열제를 투여합니다. 만약 발열이 심한 경우 타이레놀이나 부루펜과 같은 소염진통제를 사용할 수 있으며, 심한 환자에게는 부루펜이 적합합니다. 또한 소염진통제만으로 조절이 되지 않는 심한 증상이 있는 경우에는 의사의 지시에 따라 항생제를 사용해야 합니다. 항

In [153]:
# ============================================================
# [STEP 3.8 - CELL 10] v4 문서셋 저장
# ------------------------------------------------------------
# 이후 평가셋 구축 및 RAG 실험은 v4 기준으로 진행한다.
# ============================================================

RAG_DOCS_V4_JSONL_PATH = PROCESSED_DIR / "rag_docs_v4.jsonl"
RAG_DOCS_V4_JSON_PATH  = PROCESSED_DIR / "rag_docs_v4.json"
RAG_DOCS_V4_CSV_PATH   = PROCESSED_DIR / "rag_docs_v4.csv"

save_jsonl(rag_docs_v4, RAG_DOCS_V4_JSONL_PATH)
save_json(rag_docs_v4, RAG_DOCS_V4_JSON_PATH)
rag_docs_v4_df.to_csv(RAG_DOCS_V4_CSV_PATH, index=False, encoding="utf-8-sig")

print("[INFO] 저장 완료")
print(" -", RAG_DOCS_V4_JSONL_PATH)
print(" -", RAG_DOCS_V4_JSON_PATH)
print(" -", RAG_DOCS_V4_CSV_PATH)

[INFO] 저장 완료
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\processed\rag_docs_v4.jsonl
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\processed\rag_docs_v4.json
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\processed\rag_docs_v4.csv


In [155]:
# ============================================================
# [STEP 3.8 - CELL 11] STEP 3.8 요약
# ------------------------------------------------------------
# 여기까지 완료되면 고위험 노이즈 제거까지 반영된 v4 문서셋이 준비된다.
# 이후 STEP 4 평가셋 구축은 rag_docs_v4 기준으로 진행한다.
# ============================================================

step38_summary = {
    "rag_doc_count_v4": len(rag_docs_v4),
    "rag_docs_v4_jsonl_path": str(RAG_DOCS_V4_JSONL_PATH),
    "avg_char_len_v4": float(rag_docs_v4_df["char_len"].mean()) if len(rag_docs_v4_df) > 0 else 0.0,
    "max_char_len_v4": int(rag_docs_v4_df["char_len"].max()) if len(rag_docs_v4_df) > 0 else 0,
    "min_char_len_v4": int(rag_docs_v4_df["char_len"].min()) if len(rag_docs_v4_df) > 0 else 0,
}

print(json.dumps(step38_summary, ensure_ascii=False, indent=2))

{
  "rag_doc_count_v4": 139,
  "rag_docs_v4_jsonl_path": "D:\\PyProject\\AIFFEL_AI\\LLM\\NLP\\RAG_proj\\data\\processed\\rag_docs_v4.jsonl",
  "avg_char_len_v4": 4051.971223021583,
  "max_char_len_v4": 6611,
  "min_char_len_v4": 1978
}


### STEP 4. 평가셋 구축
- 질문 데이터에서 일부를 평가용으로 분리

In [175]:
# ============================================================
# [STEP 4 - CELL 1] v4 문서셋 로드 및 doc_id 매핑 준비
# ------------------------------------------------------------
# STEP 3.8에서 만든 rag_docs_v4를 기준으로
# 평가셋의 ground truth 문서 ID를 연결하기 위한 매핑을 준비한다.
# ============================================================

# 이미 메모리에 있으면 그대로 사용
if "rag_docs_v4" not in globals():
    rag_docs_v4 = load_jsonl(PROCESSED_DIR / "rag_docs_v4.jsonl")

rag_docs_v4_df = pd.DataFrame(rag_docs_v4)

print("[INFO] rag_docs_v4_df shape:", rag_docs_v4_df.shape)
display(rag_docs_v4_df.head(3))

# (disease_kor, intention) -> doc_id / answer_texts / full_text 매핑
doc_lookup = {}

for _, row in rag_docs_v4_df.iterrows():
    key = (clean_text(row["disease_kor"]), clean_text(row["intention"]))
    doc_lookup[key] = {
        "doc_id": row["doc_id"],
        "answer_texts": row["answer_texts"],
        "full_text": row["full_text"],
        "disease_eng": row["disease_eng"],
        "departments": row["departments"],
    }

print("[INFO] doc_lookup size:", len(doc_lookup))

[INFO] rag_docs_v4_df shape: (139, 16)


,doc_id,disease_category,disease_kor,disease_eng,intention,departments,question_examples,answer_texts,question_count,answer_count,full_text,char_len,question_example_count,answer_text_count,raw_question_count,raw_answer_count
0,가와사키병__약물,소아청소년질환,가와사키병,Kawasaki Disease,약물,[소아청소년과],"[가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다., 가와사키...","[가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 ...",8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,4698,8,6,406,1358
1,가와사키병__예방,소아청소년질환,가와사키병,Kawasaki Disease,예방,[소아청소년과],"[가와사키병에 대한 예방법을 알고 싶어요., 가와사키병을 예방하기 위한 생활 습관이...","[가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 ...",8,4,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,3081,8,4,221,160
2,가와사키병__원인,소아청소년질환,가와사키병,Kawasaki Disease,원인,[소아청소년과],"[가와사키병이 왜 유전적 요인에 의해 발생하는지 알려주세요., 가와사키병의 원인에 ...","[가와사키병은 몇 가지 가설과 연구 결과가 있지만, 아직까지 확실히 입증된 것은 없...",8,5,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,4067,8,5,340,1466


[INFO] doc_lookup size: 139


In [177]:
# ============================================================
# [STEP 4 - CELL 2] 질문 데이터에 ground truth doc_id 연결
# ------------------------------------------------------------
# q_df의 각 질문이 어떤 정답 문서(rag_docs_v4)에 연결되는지 표시한다.
# 기준은 disease_kor + intention 이다.
# ============================================================

eval_source_df = q_df.copy()

def make_doc_id_from_row(row):
    key = (clean_text(row["disease_kor"]), clean_text(row["intention"]))
    if key in doc_lookup:
        return doc_lookup[key]["doc_id"]
    return None

eval_source_df["ground_truth_doc_id"] = eval_source_df.apply(make_doc_id_from_row, axis=1)

print("[INFO] eval_source_df shape:", eval_source_df.shape)
print("[INFO] ground_truth_doc_id null 개수:", eval_source_df["ground_truth_doc_id"].isna().sum())

display(
    eval_source_df[
        ["disease_kor", "intention", "question", "ground_truth_doc_id"]
    ].head(10)
)

[INFO] eval_source_df shape: (39587, 19)
[INFO] ground_truth_doc_id null 개수: 0


,disease_kor,intention,question,ground_truth_doc_id
0,가와사키병,약물,"가와사키병 약물 치료의 기간은 개인별로 차이가 있는지, 아니면 대부분의 환자에게 동...",가와사키병__약물
1,가와사키병,약물,가와사키병의 진단 후에는 어떤 약물 치료와 검사를 받아야 하는지 궁금합니다.,가와사키병__약물
2,가와사키병,약물,가와사키병을 치료하기 위해 아스피린 이외에 어떤 약물을 사용하는지 알려주세요.,가와사키병__약물
3,가와사키병,약물,가와사키병을 치료하기 위해 어떤 종류의 약물이 사용될 수 있을까요?,가와사키병__약물
4,가와사키병,약물,가와사키병에 대한 약물 치료의 기간과 효과에 대한 정보를 알려주세요.,가와사키병__약물
5,가와사키병,약물,가와사키병의 약물 투여 시기를 결정하는데 고려해야 할 사항이 있을까요?,가와사키병__약물
6,가와사키병,약물,가와사키병 진단 후에도 약물 처방이 가능한지 궁금합니다.,가와사키병__약물
7,가와사키병,약물,가와사키병 치료를 위해 어떤 종류의 운동이 도움이 될까요?,가와사키병__약물
8,가와사키병,약물,가와사키병의 진단 후에는 어떤 약물 치료와 검사를 받아야 하는지 자세히 알려주세요.,가와사키병__약물
9,가와사키병,약물,가와사키병 환자가 병원에서 낙상을 예방하기 위해 어떤 조치를 취해야 할까요?,가와사키병__약물


In [179]:
# ============================================================
# [STEP 4 - CELL 3] 평가 후보 질문 정리
# ------------------------------------------------------------
# 평가셋 후보로 사용할 질문을 정리한다.
#
# 기본 원칙:
# - question 비어 있으면 제거
# - ground_truth_doc_id 없는 경우 제거
# - 중복 질문 제거
# ============================================================

eval_candidate_df = eval_source_df.copy()

# 기본 정리
eval_candidate_df["question"] = eval_candidate_df["question"].fillna("").map(clean_text)
eval_candidate_df = eval_candidate_df[eval_candidate_df["question"] != ""].copy()
eval_candidate_df = eval_candidate_df[eval_candidate_df["ground_truth_doc_id"].notna()].copy()

# 같은 질환/의도에서 완전 중복 질문 제거
eval_candidate_df = eval_candidate_df.drop_duplicates(
    subset=["disease_kor", "intention", "question"]
).reset_index(drop=True)

print("[INFO] eval_candidate_df shape:", eval_candidate_df.shape)

display(
    eval_candidate_df[
        ["disease_kor", "intention", "question", "ground_truth_doc_id"]
    ].head(10)
)

[INFO] eval_candidate_df shape: (37487, 19)


,disease_kor,intention,question,ground_truth_doc_id
0,가와사키병,약물,"가와사키병 약물 치료의 기간은 개인별로 차이가 있는지, 아니면 대부분의 환자에게 동...",가와사키병__약물
1,가와사키병,약물,가와사키병의 진단 후에는 어떤 약물 치료와 검사를 받아야 하는지 궁금합니다.,가와사키병__약물
2,가와사키병,약물,가와사키병을 치료하기 위해 아스피린 이외에 어떤 약물을 사용하는지 알려주세요.,가와사키병__약물
3,가와사키병,약물,가와사키병을 치료하기 위해 어떤 종류의 약물이 사용될 수 있을까요?,가와사키병__약물
4,가와사키병,약물,가와사키병에 대한 약물 치료의 기간과 효과에 대한 정보를 알려주세요.,가와사키병__약물
5,가와사키병,약물,가와사키병의 약물 투여 시기를 결정하는데 고려해야 할 사항이 있을까요?,가와사키병__약물
6,가와사키병,약물,가와사키병 진단 후에도 약물 처방이 가능한지 궁금합니다.,가와사키병__약물
7,가와사키병,약물,가와사키병 치료를 위해 어떤 종류의 운동이 도움이 될까요?,가와사키병__약물
8,가와사키병,약물,가와사키병의 진단 후에는 어떤 약물 치료와 검사를 받아야 하는지 자세히 알려주세요.,가와사키병__약물
9,가와사키병,약물,가와사키병 환자가 병원에서 낙상을 예방하기 위해 어떤 조치를 취해야 할까요?,가와사키병__약물


In [181]:
# ============================================================
# [STEP 4 - CELL 3] 평가 후보 질문 정리
# ------------------------------------------------------------
# 평가셋 후보로 사용할 질문을 정리한다.
#
# 기본 원칙:
# - question 비어 있으면 제거
# - ground_truth_doc_id 없는 경우 제거
# - 중복 질문 제거
# ============================================================

eval_candidate_df = eval_source_df.copy()

# 기본 정리
eval_candidate_df["question"] = eval_candidate_df["question"].fillna("").map(clean_text)
eval_candidate_df = eval_candidate_df[eval_candidate_df["question"] != ""].copy()
eval_candidate_df = eval_candidate_df[eval_candidate_df["ground_truth_doc_id"].notna()].copy()

# 같은 질환/의도에서 완전 중복 질문 제거
eval_candidate_df = eval_candidate_df.drop_duplicates(
    subset=["disease_kor", "intention", "question"]
).reset_index(drop=True)

print("[INFO] eval_candidate_df shape:", eval_candidate_df.shape)

display(
    eval_candidate_df[
        ["disease_kor", "intention", "question", "ground_truth_doc_id"]
    ].head(10)
)

[INFO] eval_candidate_df shape: (37487, 19)


,disease_kor,intention,question,ground_truth_doc_id
0,가와사키병,약물,"가와사키병 약물 치료의 기간은 개인별로 차이가 있는지, 아니면 대부분의 환자에게 동...",가와사키병__약물
1,가와사키병,약물,가와사키병의 진단 후에는 어떤 약물 치료와 검사를 받아야 하는지 궁금합니다.,가와사키병__약물
2,가와사키병,약물,가와사키병을 치료하기 위해 아스피린 이외에 어떤 약물을 사용하는지 알려주세요.,가와사키병__약물
3,가와사키병,약물,가와사키병을 치료하기 위해 어떤 종류의 약물이 사용될 수 있을까요?,가와사키병__약물
4,가와사키병,약물,가와사키병에 대한 약물 치료의 기간과 효과에 대한 정보를 알려주세요.,가와사키병__약물
5,가와사키병,약물,가와사키병의 약물 투여 시기를 결정하는데 고려해야 할 사항이 있을까요?,가와사키병__약물
6,가와사키병,약물,가와사키병 진단 후에도 약물 처방이 가능한지 궁금합니다.,가와사키병__약물
7,가와사키병,약물,가와사키병 치료를 위해 어떤 종류의 운동이 도움이 될까요?,가와사키병__약물
8,가와사키병,약물,가와사키병의 진단 후에는 어떤 약물 치료와 검사를 받아야 하는지 자세히 알려주세요.,가와사키병__약물
9,가와사키병,약물,가와사키병 환자가 병원에서 낙상을 예방하기 위해 어떤 조치를 취해야 할까요?,가와사키병__약물


In [183]:
# ============================================================
# [STEP 4 - CELL 4] 질환/의도별 평가 샘플 수 설정
# ------------------------------------------------------------
# 평가셋은 질환/의도별 균형을 맞추는 것이 중요하다.
#
# 우선 baseline 평가셋으로
# - 각 (disease_kor, intention) 그룹당 N개
# 를 샘플링한다.
#
# 너무 큰 평가셋은 RAGAS/G-EVAL 비용과 시간이 커지므로
# 처음에는 적당한 크기로 시작한다.
# ============================================================

EVAL_SAMPLES_PER_GROUP = 3

group_sizes = (
    eval_candidate_df.groupby(["disease_kor", "intention"])
    .size()
    .reset_index(name="candidate_count")
)

print("[INFO] 그룹 수:", len(group_sizes))
print("[INFO] 그룹별 후보 수 통계]")
print(group_sizes["candidate_count"].describe())

display(group_sizes.head(10))

[INFO] 그룹 수: 139
[INFO] 그룹별 후보 수 통계]
count    139.000000
mean     269.690647
std       69.246027
min      110.000000
25%      221.000000
50%      271.000000
75%      319.500000
max      484.000000
Name: candidate_count, dtype: float64


,disease_kor,intention,candidate_count
0,가와사키병,약물,406
1,가와사키병,예방,221
2,가와사키병,원인,340
3,가와사키병,정의,275
4,가와사키병,증상,335
5,가와사키병,진단,403
6,가와사키병,치료,337
7,과숙아,원인,267
8,과숙아,정의,366
9,과숙아,증상,274


In [197]:
# ============================================================
# [STEP 4 - 복구 CELL 5] eval_sample_df 재생성
# ------------------------------------------------------------
# eval_sample_df가 없는 경우를 대비해
# STEP 4 - CELL 5를 다시 실행하는 복구 셀이다.
# ============================================================

EVAL_SAMPLES_PER_GROUP = 2   # 처음 실험은 2 추천

def sample_questions_from_group(group_df, n=3):
    group_df = group_df.copy()
    group_df["q_len"] = group_df["question"].map(len)
    group_df = group_df.sort_values("q_len").reset_index(drop=True)

    if len(group_df) <= n:
        return group_df.drop(columns=["q_len"])

    idxs = np.linspace(0, len(group_df) - 1, n, dtype=int)
    sampled = group_df.iloc[idxs].copy()
    return sampled.drop(columns=["q_len"])

sampled_eval_parts = []

for (disease_kor, intention), g in eval_candidate_df.groupby(["disease_kor", "intention"]):
    sampled_g = sample_questions_from_group(g, n=EVAL_SAMPLES_PER_GROUP)
    sampled_eval_parts.append(sampled_g)

eval_sample_df = pd.concat(sampled_eval_parts, axis=0).reset_index(drop=True)

print("[INFO] eval_sample_df shape:", eval_sample_df.shape)

display(
    eval_sample_df[
        ["disease_kor", "intention", "question", "ground_truth_doc_id"]
    ].head(20)
)

[INFO] eval_sample_df shape: (278, 19)


,disease_kor,intention,question,ground_truth_doc_id
0,가와사키병,약물,가와사키병의 약물치료가 효과적인가요?,가와사키병__약물
1,가와사키병,약물,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,가와사키병__약물
2,가와사키병,예방,가와사키병에 대한 예방법을 알고 싶어요.,가와사키병__예방
3,가와사키병,예방,가와사키병의 예방에 있어서 유전적 요인이 어떤 역할을 하는지 알려주세요. 예방에 효...,가와사키병__예방
4,가와사키병,원인,가와사키병의 원인은 무엇인가요?,가와사키병__원인
5,가와사키병,원인,가와사키병이 유전적 요인과 관련이 있다는 말을 듣고 있습니다. 저희 부부도 가와사키...,가와사키병__원인
6,가와사키병,정의,가와사키병이란 무엇인가요?,가와사키병__정의
7,가와사키병,정의,가와사키병이 어떤 의미를 가지는 질병인지 알고 싶어요. 가와사키병에 대한 정확한 정...,가와사키병__정의
8,가와사키병,증상,가와사키병이란 어떤 질병인가요?,가와사키병__증상
9,가와사키병,증상,가와사키병에 대해 자세히 알고 싶어요. 어린 아이의 열과 피부와 얼굴이 붉어지는 것...,가와사키병__증상


In [199]:
# ============================================================
# [STEP 4 - 복구 CELL 6] reference_answer 생성
# ------------------------------------------------------------
# eval_sample_df에 ground truth 문서의 answer_texts를 붙여
# reference_answer를 만든다.
# ============================================================

def build_reference_answer_from_doc_id(doc_id):
    matched = rag_docs_v4_df[rag_docs_v4_df["doc_id"] == doc_id]

    if len(matched) == 0:
        return ""

    row = matched.iloc[0]
    answer_texts = row["answer_texts"] if isinstance(row["answer_texts"], list) else []

    reference_answer = "\n\n".join(
        [clean_text(a) for a in answer_texts if clean_text(a)]
    )
    return clean_text(reference_answer)

eval_sample_df["reference_answer"] = eval_sample_df["ground_truth_doc_id"].map(
    build_reference_answer_from_doc_id
)

print("[INFO] reference_answer empty 개수:", (eval_sample_df["reference_answer"] == "").sum())

display(
    eval_sample_df[
        ["disease_kor", "intention", "question", "ground_truth_doc_id", "reference_answer"]
    ].head(5)
)

[INFO] reference_answer empty 개수: 0


,disease_kor,intention,question,ground_truth_doc_id,reference_answer
0,가와사키병,약물,가와사키병의 약물치료가 효과적인가요?,가와사키병__약물,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및..."
1,가와사키병,약물,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,가와사키병__약물,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및..."
2,가와사키병,예방,가와사키병에 대한 예방법을 알고 싶어요.,가와사키병__예방,"가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 개..."
3,가와사키병,예방,가와사키병의 예방에 있어서 유전적 요인이 어떤 역할을 하는지 알려주세요. 예방에 효...,가와사키병__예방,"가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 개..."
4,가와사키병,원인,가와사키병의 원인은 무엇인가요?,가와사키병__원인,"가와사키병은 몇 가지 가설과 연구 결과가 있지만, 아직까지 확실히 입증된 것은 없습..."


In [201]:
# ============================================================
# [STEP 4 - 복구 CELL 7] 최종 평가셋 생성
# ------------------------------------------------------------
# RAGAS / G-EVAL용 평가셋 DataFrame을 생성한다.
# ============================================================

def infer_question_type(intention):
    intention = clean_text(intention)
    mapping = {
        "정의": "definition",
        "원인": "cause",
        "증상": "symptom",
        "진단": "diagnosis",
        "치료": "treatment",
        "약물": "medication",
        "예방": "prevention",
        "식이, 생활": "lifestyle",
        "재활": "rehabilitation",
        "검진": "screening",
    }
    return mapping.get(intention, "general")

eval_sample_df["question_type"] = eval_sample_df["intention"].map(infer_question_type)
eval_sample_df["eval_id"] = [f"EVAL_{i:04d}" for i in range(1, len(eval_sample_df) + 1)]

eval_dataset_df = eval_sample_df[
    [
        "eval_id",
        "disease_category",
        "disease_kor",
        "disease_eng",
        "intention",
        "question_type",
        "question",
        "ground_truth_doc_id",
        "reference_answer",
    ]
].copy()

print("[INFO] eval_dataset_df shape:", eval_dataset_df.shape)
display(eval_dataset_df.head(10))

[INFO] eval_dataset_df shape: (278, 9)


,eval_id,disease_category,disease_kor,disease_eng,intention,question_type,question,ground_truth_doc_id,reference_answer
0,EVAL_0001,소아청소년질환,가와사키병,Kawasaki Disease,약물,medication,가와사키병의 약물치료가 효과적인가요?,가와사키병__약물,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및..."
1,EVAL_0002,소아청소년질환,가와사키병,Kawasaki Disease,약물,medication,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,가와사키병__약물,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및..."
2,EVAL_0003,소아청소년질환,가와사키병,Kawasaki Disease,예방,prevention,가와사키병에 대한 예방법을 알고 싶어요.,가와사키병__예방,"가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 개..."
3,EVAL_0004,소아청소년질환,가와사키병,Kawasaki Disease,예방,prevention,가와사키병의 예방에 있어서 유전적 요인이 어떤 역할을 하는지 알려주세요. 예방에 효...,가와사키병__예방,"가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 개..."
4,EVAL_0005,소아청소년질환,가와사키병,Kawasaki Disease,원인,cause,가와사키병의 원인은 무엇인가요?,가와사키병__원인,"가와사키병은 몇 가지 가설과 연구 결과가 있지만, 아직까지 확실히 입증된 것은 없습..."
5,EVAL_0006,소아청소년질환,가와사키병,Kawasaki Disease,원인,cause,가와사키병이 유전적 요인과 관련이 있다는 말을 듣고 있습니다. 저희 부부도 가와사키...,가와사키병__원인,"가와사키병은 몇 가지 가설과 연구 결과가 있지만, 아직까지 확실히 입증된 것은 없습..."
6,EVAL_0007,소아청소년질환,가와사키병,Kawasaki Disease,정의,definition,가와사키병이란 무엇인가요?,가와사키병__정의,가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 주...
7,EVAL_0008,소아청소년질환,가와사키병,Kawasaki Disease,정의,definition,가와사키병이 어떤 의미를 가지는 질병인지 알고 싶어요. 가와사키병에 대한 정확한 정...,가와사키병__정의,가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 주...
8,EVAL_0009,소아청소년질환,가와사키병,Kawasaki Disease,증상,symptom,가와사키병이란 어떤 질병인가요?,가와사키병__증상,"가와사키병은 5일 이상 지속되는 고열, 손과 발 부종, 다양한 형태의 피부 발진 등..."
9,EVAL_0010,소아청소년질환,가와사키병,Kawasaki Disease,증상,symptom,가와사키병에 대해 자세히 알고 싶어요. 어린 아이의 열과 피부와 얼굴이 붉어지는 것...,가와사키병__증상,"가와사키병은 5일 이상 지속되는 고열, 손과 발 부종, 다양한 형태의 피부 발진 등..."


In [203]:
# ============================================================
# [STEP 4 - CELL 8] 평가셋 분포 확인
# ------------------------------------------------------------
# 질환/의도별로 샘플 수가 고르게 뽑혔는지 확인한다.
# ============================================================

print("[INFO] intention 분포]")
display(eval_dataset_df["intention"].value_counts().to_frame("count"))

print("[INFO] question_type 분포]")
display(eval_dataset_df["question_type"].value_counts().to_frame("count"))

group_eval_counts = (
    eval_dataset_df.groupby(["disease_kor", "intention"])
    .size()
    .reset_index(name="eval_count")
)

print("[INFO] 그룹별 eval_count 통계]")
print(group_eval_counts["eval_count"].describe())

display(group_eval_counts.head(20))

[INFO] intention 분포]


,count
intention,
원인,48
정의,48
치료,48
증상,46
진단,44
예방,16
약물,14
"식이, 생활",10
검진,2


[INFO] question_type 분포]


,count
question_type,
cause,48
definition,48
treatment,48
symptom,46
diagnosis,44
prevention,16
medication,14
lifestyle,10
screening,2


[INFO] 그룹별 eval_count 통계]
count    139.0
mean       2.0
std        0.0
min        2.0
25%        2.0
50%        2.0
75%        2.0
max        2.0
Name: eval_count, dtype: float64


,disease_kor,intention,eval_count
0,가와사키병,약물,2
1,가와사키병,예방,2
2,가와사키병,원인,2
3,가와사키병,정의,2
4,가와사키병,증상,2
5,가와사키병,진단,2
6,가와사키병,치료,2
7,과숙아,원인,2
8,과숙아,정의,2
9,과숙아,증상,2


In [205]:
# ============================================================
# [STEP 4 - CELL 9] 샘플 질문/정답 육안 검토
# ------------------------------------------------------------
# 실제 평가셋이 의도대로 만들어졌는지 사람 눈으로 확인한다.
# ============================================================

N_SHOW = 5

for i in range(min(N_SHOW, len(eval_dataset_df))):
    row = eval_dataset_df.iloc[i]

    print("=" * 120)
    print(f"[샘플 {i+1}] eval_id = {row['eval_id']}")
    print("-" * 120)
    print("질환명      :", row["disease_kor"])
    print("intention   :", row["intention"])
    print("question_type:", row["question_type"])
    print("question    :", row["question"])
    print("ground_truth_doc_id:", row["ground_truth_doc_id"])
    print("reference_answer:")
    print(textwrap.shorten(row["reference_answer"], width=1000, placeholder=" ... [생략]"))
    print()

[샘플 1] eval_id = EVAL_0001
------------------------------------------------------------------------------------------------------------------------
질환명      : 가와사키병
intention   : 약물
question_type: medication
question    : 가와사키병의 약물치료가 효과적인가요?
ground_truth_doc_id: 가와사키병__약물
reference_answer:
가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및 대증 치료로 이뤄집니다. 가와사키병의 치료에는 대증요법이 일반적으로 사용됩니다. 대부분의 소아는 열이나 구토와 같은 증상으로 인해 체온을 떨어뜨리기 위해 아스피린이나 아세트아미노펜과 같은 비스테로이드성 소염제를 사용합니다. 그러나 심한 증상이 나타나면 소염제를 복용할 수 없으므로, 병의 증상이 심각해지기 전까지는 입원을 통해 적절한 치료를 받아야 합니다. 고열이 있는 환자는 아스피린을 복용해야 하며, 이 경우 증상이 더 심해질 수 있습니다. 따라서 체온이 38℃ 이상인 환자에게는 아세트아미노펜이나 해열제를 투여합니다. 만약 발열이 심한 경우 타이레놀이나 부루펜과 같은 소염진통제를 사용할 수 있으며, 심한 환자에게는 부루펜이 적합합니다. 또한 소염진통제만으로 조절이 되지 않는 심한 증상이 있는 경우에는 의사의 지시에 따라 항생제를 사용해야 합니다. 항생제는 증상이 시작되고 2일 이상이 경과한 후에 투약해야 합니다. 또한, 열이 나면서 경련 등의 이상 증상이 있는 경우에는 항경련제를 투약해야 합니다. 가와사키병의 치료에는 대증요법이 일반적으로 사용되지만, 소염진통제나 대증치료는 상황에 따라 선택되어야 할 수도 있습니다. 가와사키병을 진단하기 위해서는 일반적으로 혈액 검사를 시행합니다. 일반적으로 고열이나 구토와 같은 증상이 있는 경우 병원을 찾아 진단을 받아야 합니다.

In [207]:
# ============================================================
# [STEP 4 - CELL 10] 평가셋 저장
# ------------------------------------------------------------
# 이후 STEP 5~9에서 반복 사용하기 위해 JSONL / JSON / CSV로 저장한다.
# ============================================================

EVAL_DATASET_JSONL_PATH = EVAL_DIR / "eval_dataset_v1.jsonl"
EVAL_DATASET_JSON_PATH  = EVAL_DIR / "eval_dataset_v1.json"
EVAL_DATASET_CSV_PATH   = EVAL_DIR / "eval_dataset_v1.csv"

eval_dataset_records = eval_dataset_df.to_dict(orient="records")

save_jsonl(eval_dataset_records, EVAL_DATASET_JSONL_PATH)
save_json(eval_dataset_records, EVAL_DATASET_JSON_PATH)
eval_dataset_df.to_csv(EVAL_DATASET_CSV_PATH, index=False, encoding="utf-8-sig")

print("[INFO] 저장 완료")
print(" -", EVAL_DATASET_JSONL_PATH)
print(" -", EVAL_DATASET_JSON_PATH)
print(" -", EVAL_DATASET_CSV_PATH)

[INFO] 저장 완료
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\eval\eval_dataset_v1.jsonl
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\eval\eval_dataset_v1.json
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\eval\eval_dataset_v1.csv


In [209]:
# ============================================================
# [STEP 4 - CELL 11] STEP 4 요약
# ------------------------------------------------------------
# 여기까지 완료되면 RAGAS / G-EVAL용 평가셋이 준비된다.
# 이후 STEP 5에서는 Naive RAG baseline을 구축한다.
# ============================================================

step4_summary = {
    "eval_dataset_count": len(eval_dataset_df),
    "samples_per_group": EVAL_SAMPLES_PER_GROUP,
    "num_groups": int(group_eval_counts.shape[0]),
    "eval_dataset_jsonl_path": str(EVAL_DATASET_JSONL_PATH),
}

print(json.dumps(step4_summary, ensure_ascii=False, indent=2))

{
  "eval_dataset_count": 278,
  "samples_per_group": 2,
  "num_groups": 139,
  "eval_dataset_jsonl_path": "D:\\PyProject\\AIFFEL_AI\\LLM\\NLP\\RAG_proj\\data\\eval\\eval_dataset_v1.jsonl"
}


### STEP 5. Baseline Naive RAG 구축

In [212]:
# ============================================================
# [STEP 5 - CELL 1] Baseline Naive RAG용 라이브러리 import
# ------------------------------------------------------------
# 이번 baseline에서는:
# - 문서 임베딩: sentence-transformers
# - 유사도 계산: sklearn cosine_similarity
# - 생성 모델: OpenAI API
# 를 사용한다.
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

print("[INFO] STEP 5 기본 라이브러리 import 완료")

[INFO] STEP 5 기본 라이브러리 import 완료


In [213]:
# ============================================================
# [STEP 5 - CELL 2] 데이터 로드
# ------------------------------------------------------------
# rag_docs_v4 / eval_dataset_df가 메모리에 없으면 저장 파일에서 다시 로드한다.
# ============================================================

RAG_DOCS_V4_JSONL_PATH = PROCESSED_DIR / "rag_docs_v4.jsonl"
EVAL_DATASET_JSONL_PATH = EVAL_DIR / "eval_dataset_v1.jsonl"

if "rag_docs_v4" not in globals():
    rag_docs_v4 = load_jsonl(RAG_DOCS_V4_JSONL_PATH)

rag_docs_v4_df = pd.DataFrame(rag_docs_v4)

if "eval_dataset_df" not in globals():
    eval_dataset_records = load_jsonl(EVAL_DATASET_JSONL_PATH)
    eval_dataset_df = pd.DataFrame(eval_dataset_records)

print("[INFO] rag_docs_v4_df shape:", rag_docs_v4_df.shape)
print("[INFO] eval_dataset_df shape:", eval_dataset_df.shape)

display(rag_docs_v4_df.head(3))
display(eval_dataset_df.head(3))

[INFO] rag_docs_v4_df shape: (139, 16)
[INFO] eval_dataset_df shape: (278, 9)


,doc_id,disease_category,disease_kor,disease_eng,intention,departments,question_examples,answer_texts,question_count,answer_count,full_text,char_len,question_example_count,answer_text_count,raw_question_count,raw_answer_count
0,가와사키병__약물,소아청소년질환,가와사키병,Kawasaki Disease,약물,[소아청소년과],"[가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다., 가와사키...","[가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 ...",8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,4698,8,6,406,1358
1,가와사키병__예방,소아청소년질환,가와사키병,Kawasaki Disease,예방,[소아청소년과],"[가와사키병에 대한 예방법을 알고 싶어요., 가와사키병을 예방하기 위한 생활 습관이...","[가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 ...",8,4,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,3081,8,4,221,160
2,가와사키병__원인,소아청소년질환,가와사키병,Kawasaki Disease,원인,[소아청소년과],"[가와사키병이 왜 유전적 요인에 의해 발생하는지 알려주세요., 가와사키병의 원인에 ...","[가와사키병은 몇 가지 가설과 연구 결과가 있지만, 아직까지 확실히 입증된 것은 없...",8,5,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,4067,8,5,340,1466


,eval_id,disease_category,disease_kor,disease_eng,intention,question_type,question,ground_truth_doc_id,reference_answer
0,EVAL_0001,소아청소년질환,가와사키병,Kawasaki Disease,약물,medication,가와사키병의 약물치료가 효과적인가요?,가와사키병__약물,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및..."
1,EVAL_0002,소아청소년질환,가와사키병,Kawasaki Disease,약물,medication,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,가와사키병__약물,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및..."
2,EVAL_0003,소아청소년질환,가와사키병,Kawasaki Disease,예방,prevention,가와사키병에 대한 예방법을 알고 싶어요.,가와사키병__예방,"가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 개..."


In [214]:
# ============================================================
# [STEP 5 - CELL 3] 임베딩 입력 텍스트 준비
# ------------------------------------------------------------
# retrieval용 문서 텍스트는 full_text를 그대로 사용한다.
# 이후 naive baseline에서는 chunking 없이 문서 단위 검색을 수행한다.
# ============================================================

doc_ids = rag_docs_v4_df["doc_id"].tolist()
doc_texts = rag_docs_v4_df["full_text"].tolist()

print("[INFO] 문서 수:", len(doc_ids))
print("[INFO] 샘플 doc_id:", doc_ids[:5])
print("[INFO] 샘플 문서 길이:", [len(x) for x in doc_texts[:5]])

[INFO] 문서 수: 139
[INFO] 샘플 doc_id: ['가와사키병__약물', '가와사키병__예방', '가와사키병__원인', '가와사키병__정의', '가와사키병__증상']
[INFO] 샘플 문서 길이: [4698, 3081, 4067, 5663, 4622]


In [218]:
# ============================================================
# [STEP 5 - CELL 4] 임베딩 모델 로드
# ------------------------------------------------------------
# 한국어 질의/문서 검색 baseline용으로 sentence-transformers 모델을 사용한다.
# 우선 baseline에서는 범용 다국어 모델을 사용한다.
#
# 필요 시 추후 다른 임베딩 모델과 비교 가능하다.
# ============================================================

EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

embed_model = SentenceTransformer(EMBED_MODEL_NAME)

print("[INFO] 임베딩 모델 로드 완료:", EMBED_MODEL_NAME)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[INFO] 임베딩 모델 로드 완료: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


In [219]:
# ============================================================
# [STEP 5 - CELL 4] 임베딩 모델 로드
# ------------------------------------------------------------
# 한국어 질의/문서 검색 baseline용으로 sentence-transformers 모델을 사용한다.
# 우선 baseline에서는 범용 다국어 모델을 사용한다.
#
# 필요 시 추후 다른 임베딩 모델과 비교 가능하다.
# ============================================================

EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

embed_model = SentenceTransformer(EMBED_MODEL_NAME)

print("[INFO] 임베딩 모델 로드 완료:", EMBED_MODEL_NAME)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] 임베딩 모델 로드 완료: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


In [220]:
# ============================================================
# [STEP 5 - CELL 5] 문서 임베딩 생성
# ------------------------------------------------------------
# rag_docs_v4 전체 문서에 대해 dense embedding을 생성한다.
# 생성 결과는 numpy array로 저장한다.
# ============================================================

doc_embeddings = embed_model.encode(
    doc_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("[INFO] doc_embeddings shape:", doc_embeddings.shape)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

[INFO] doc_embeddings shape: (139, 768)


In [221]:
# ============================================================
# [STEP 5 - CELL 5] 문서 임베딩 생성
# ------------------------------------------------------------
# rag_docs_v4 전체 문서에 대해 dense embedding을 생성한다.
# 생성 결과는 numpy array로 저장한다.
# ============================================================

doc_embeddings = embed_model.encode(
    doc_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("[INFO] doc_embeddings shape:", doc_embeddings.shape)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

[INFO] doc_embeddings shape: (139, 768)


In [222]:
# ============================================================
# [STEP 5 - CELL 6] 검색 함수 정의
# ------------------------------------------------------------
# 질문(query)을 임베딩한 뒤,
# 문서 임베딩과 cosine similarity를 계산하여 상위 top-k 문서를 반환한다.
# ============================================================

def retrieve_top_k_naive(query, top_k=3):
    query = clean_text(query)

    query_emb = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    sims = cosine_similarity(query_emb, doc_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_idx, start=1):
        results.append({
            "rank": rank,
            "doc_id": doc_ids[idx],
            "score": float(sims[idx]),
            "full_text": doc_texts[idx],
            "disease_kor": rag_docs_v4_df.iloc[idx]["disease_kor"],
            "intention": rag_docs_v4_df.iloc[idx]["intention"],
        })
    return results

print("[INFO] retrieve_top_k_naive 함수 준비 완료")

[INFO] retrieve_top_k_naive 함수 준비 완료


In [223]:
# ============================================================
# [STEP 5 - CELL 7] 검색 결과 포맷 함수
# ------------------------------------------------------------
# 검색된 문서들을 LLM 프롬프트에 넣기 위해 context 문자열로 합친다.
# ============================================================

def build_context_from_retrieval_results(results):
    context_blocks = []

    for r in results:
        block = (
            f"[문서 {r['rank']}]\n"
            f"doc_id: {r['doc_id']}\n"
            f"질환명: {r['disease_kor']}\n"
            f"상담주제: {r['intention']}\n"
            f"유사도점수: {r['score']:.4f}\n"
            f"내용:\n{r['full_text']}"
        )
        context_blocks.append(block)

    return "\n\n" + ("=" * 80 + "\n\n").join(context_blocks)

print("[INFO] build_context_from_retrieval_results 함수 준비 완료")

[INFO] build_context_from_retrieval_results 함수 준비 완료


In [224]:
# ============================================================
# [STEP 5 - CELL 8] OpenAI API 준비
# ------------------------------------------------------------
# 생성 모델은 OpenAI를 사용한다.
# .env 파일에서 OPENAI_API_KEY를 불러온다.
# ============================================================

from dotenv import load_dotenv
from openai import OpenAI

ENV_PATH = r"D:\PyProject\env_keys\.env"
load_dotenv(ENV_PATH)

openai_api_key = os.getenv("OPENAI_API_KEY")
assert openai_api_key is not None and len(openai_api_key) > 0, "OPENAI_API_KEY를 확인하세요."

client = OpenAI(api_key=openai_api_key)

GEN_MODEL_NAME = "gpt-4o-mini"

print("[INFO] OpenAI client 준비 완료")
print("[INFO] GEN_MODEL_NAME:", GEN_MODEL_NAME)

[INFO] OpenAI client 준비 완료
[INFO] GEN_MODEL_NAME: gpt-4o-mini


In [226]:
# ============================================================
# [STEP 5 - CELL 9] Baseline Naive RAG 프롬프트 정의
# ------------------------------------------------------------
# 가장 단순한 grounded generation 프롬프트를 사용한다.
# 원칙:
# - 검색 문서 기반으로만 답변
# - 모르면 모른다고 말하기
# - 존댓말 유지
# - 의료적 확정 진단처럼 단정하지 않기
# ============================================================

SYSTEM_PROMPT_NAIVE_RAG = """
당신은 소아과 상담 보조 AI입니다.
반드시 제공된 검색 문서(context)에 근거해서만 답변하세요.

규칙:
1. 검색 문서에 있는 정보 중심으로 답변하세요.
2. 검색 문서에 없는 내용을 추측해서 단정하지 마세요.
3. 보호자가 이해하기 쉬운 한국어 존댓말로 답변하세요.
4. 진단을 확정하는 표현은 피하세요.
5. 위험하거나 증상이 심한 경우에는 소아청소년과 진료가 필요할 수 있다고 안내하세요.
6. 답변은 핵심 위주로 4~8문장 정도로 작성하세요.
""".strip()

def build_user_prompt_naive_rag(question, context):
    return f"""
[사용자 질문]
{question}

[검색 문서]
{context}

위 검색 문서를 바탕으로 사용자 질문에 답변해주세요.
""".strip()

print("[INFO] baseline prompt 준비 완료")

[INFO] baseline prompt 준비 완료


In [234]:
# ============================================================
# [STEP 5 - CELL 10] 단건 생성 함수 정의
# ------------------------------------------------------------
# 질문 1개에 대해:
# - top-k retrieval
# - context 생성
# - LLM 답변 생성
# 을 수행한다.
# ============================================================

def generate_answer_naive_rag(question, top_k=3, temperature=0.0):
    retrieval_results = retrieve_top_k_naive(question, top_k=top_k)
    context = build_context_from_retrieval_results(retrieval_results)

    user_prompt = build_user_prompt_naive_rag(question, context)

    response = client.chat.completions.create(
        model=GEN_MODEL_NAME,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_NAIVE_RAG},
            {"role": "user", "content": user_prompt},
        ]
    )

    answer = response.choices[0].message.content.strip()

    return {
        "question": question,
        "answer": answer,
        "retrieval_results": retrieval_results,
        "context": context,
        "top_k": top_k,
        "model_name": GEN_MODEL_NAME,
    }

print("[INFO] generate_answer_naive_rag 함수 준비 완료")

[INFO] generate_answer_naive_rag 함수 준비 완료


In [236]:
# ============================================================
# [STEP 5 - CELL 11] 단건 테스트
# ------------------------------------------------------------
# 먼저 샘플 질문 1개에 대해 naive RAG가 어떻게 동작하는지 확인한다.
# ============================================================

test_question = "가와사키병 치료에 사용되는 약물은 무엇인가요?"

test_result = generate_answer_naive_rag(
    question=test_question,
    top_k=3,
    temperature=0.0
)

print("[질문]")
print(test_result["question"])

print("\n[답변]")
print(test_result["answer"])

print("\n[검색 결과]")
for r in test_result["retrieval_results"]:
    print(f"- rank={r['rank']} | doc_id={r['doc_id']} | score={r['score']:.4f}")

[질문]
가와사키병 치료에 사용되는 약물은 무엇인가요?

[답변]
가와사키병의 치료에 사용되는 약물은 주로 저용량 아스피린과 면역글로불린입니다. 초기 치료로는 고용량 면역글로불린이 사용되며, 이는 관상동맥 합병증을 예방하는 데 효과적입니다. 또한, 증상 완화를 위해 비스테로이드성 소염제인 아스피린이나 아세트아미노펜이 사용될 수 있습니다. 심한 증상이 나타날 경우에는 항생제나 항경련제를 사용할 수 있으며, 필요에 따라 항응고제도 고려될 수 있습니다. 치료는 환자의 상태에 따라 달라질 수 있으므로, 전문의와 상담하여 적절한 치료 계획을 세우는 것이 중요합니다.

[검색 결과]
- rank=1 | doc_id=가와사키병__치료 | score=0.5395
- rank=2 | doc_id=가와사키병__약물 | score=0.5189
- rank=3 | doc_id=가와사키병__예방 | score=0.4712


In [238]:
# ============================================================
# [STEP 5 - CELL 12] 평가셋 일부 샘플 테스트
# ------------------------------------------------------------
# eval_dataset_df에서 앞부분 몇 개 질문만 골라 baseline 결과를 확인한다.
# 전체 실행 전에 retrieval 품질과 생성 답변을 점검한다.
# ============================================================

N_DEBUG = 5
debug_rows = eval_dataset_df.head(N_DEBUG)

debug_results = []

for _, row in debug_rows.iterrows():
    result = generate_answer_naive_rag(
        question=row["question"],
        top_k=3,
        temperature=0.0
    )

    debug_results.append({
        "eval_id": row["eval_id"],
        "disease_kor": row["disease_kor"],
        "intention": row["intention"],
        "question": row["question"],
        "ground_truth_doc_id": row["ground_truth_doc_id"],
        "pred_answer": result["answer"],
        "retrieved_doc_ids": [x["doc_id"] for x in result["retrieval_results"]],
        "retrieved_scores": [x["score"] for x in result["retrieval_results"]],
    })

debug_results_df = pd.DataFrame(debug_results)

print("[INFO] debug_results_df shape:", debug_results_df.shape)
display(debug_results_df)

[INFO] debug_results_df shape: (5, 8)


,eval_id,disease_kor,intention,question,ground_truth_doc_id,pred_answer,retrieved_doc_ids,retrieved_scores
0,EVAL_0001,가와사키병,약물,가와사키병의 약물치료가 효과적인가요?,가와사키병__약물,가와사키병의 약물 치료는 주로 증상 완화와 관상동맥 합병증 예방을 목표로 합니다. ...,"[가와사키병__치료, 가와사키병__예방, 가와사키병__약물]","[0.595293402671814, 0.5232875347137451, 0.5002..."
1,EVAL_0002,가와사키병,약물,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,가와사키병__약물,"죄송하지만, 제공된 검색 문서에는 가와사키병에 대한 약물 치료가 임신 또는 수유 중...","[과숙아__원인, 신생아_황달__치료, 신생아_경련__치료]","[0.6249063014984131, 0.6198438405990601, 0.607..."
2,EVAL_0003,가와사키병,예방,가와사키병에 대한 예방법을 알고 싶어요.,가와사키병__예방,가와사키병은 현재 예방 방법이 개발되지 않은 질환입니다. 하지만 개인 위생 관리를 ...,"[가와사키병__예방, 가와사키병__원인, 가와사키병__치료]","[0.6530215740203857, 0.5434562563896179, 0.524..."
3,EVAL_0004,가와사키병,예방,가와사키병의 예방에 있어서 유전적 요인이 어떤 역할을 하는지 알려주세요. 예방에 효...,가와사키병__예방,가와사키병의 예방에 있어 유전적 요인은 중요한 역할을 할 수 있습니다. 연구에 따르...,"[가와사키병__원인, 가와사키병__예방, 가와사키병__치료]","[0.6852616667747498, 0.6082280278205872, 0.478..."
4,EVAL_0005,가와사키병,원인,가와사키병의 원인은 무엇인가요?,가와사키병__원인,"가와사키병의 원인은 아직 명확히 밝혀지지 않았습니다. 여러 연구에 따르면, 유전적 ...","[가와사키병__원인, 성홍열__원인, 유행성_이하선염__원인]","[0.6906836628913879, 0.46721524000167847, 0.46..."


In [239]:
# ============================================================
# [STEP 5 - CELL 13] 전체 평가셋에 대해 baseline 추론 함수 정의
# ------------------------------------------------------------
# eval_dataset_df 전체에 대해 naive RAG baseline 결과를 생성한다.
# 실행 시간이 걸릴 수 있으므로 별도 함수로 만든다.
# ============================================================

def run_naive_rag_inference(eval_df, top_k=3, temperature=0.0, limit=None):
    rows = eval_df.copy()

    if limit is not None:
        rows = rows.head(limit).copy()

    output_rows = []

    for i, (_, row) in enumerate(rows.iterrows(), start=1):
        result = generate_answer_naive_rag(
            question=row["question"],
            top_k=top_k,
            temperature=temperature
        )

        output_rows.append({
            "eval_id": row["eval_id"],
            "disease_category": row["disease_category"],
            "disease_kor": row["disease_kor"],
            "disease_eng": row["disease_eng"],
            "intention": row["intention"],
            "question_type": row["question_type"],
            "question": row["question"],
            "ground_truth_doc_id": row["ground_truth_doc_id"],
            "reference_answer": row["reference_answer"],
            "pred_answer": result["answer"],
            "retrieved_doc_ids": [x["doc_id"] for x in result["retrieval_results"]],
            "retrieved_scores": [x["score"] for x in result["retrieval_results"]],
            "top_k": top_k,
            "model_name": GEN_MODEL_NAME,
        })

        if i % 20 == 0:
            print(f"[INFO] {i}/{len(rows)} 완료")

    return pd.DataFrame(output_rows)

print("[INFO] run_naive_rag_inference 함수 준비 완료")

[INFO] run_naive_rag_inference 함수 준비 완료


In [240]:
# ============================================================
# [STEP 5 - CELL 14] Baseline Naive RAG 실행
# ------------------------------------------------------------
# 처음에는 limit를 두고 작게 돌려본 뒤,
# 괜찮으면 전체 eval_dataset_df로 확장한다.
# ============================================================

NAIVE_TOP_K = 3
NAIVE_TEMPERATURE = 0.0
NAIVE_LIMIT = 30   # 처음엔 30개 정도로 테스트 권장

naive_rag_results_df = run_naive_rag_inference(
    eval_df=eval_dataset_df,
    top_k=NAIVE_TOP_K,
    temperature=NAIVE_TEMPERATURE,
    limit=NAIVE_LIMIT,
)

print("[INFO] naive_rag_results_df shape:", naive_rag_results_df.shape)
display(naive_rag_results_df.head(10))

[INFO] 20/30 완료
[INFO] naive_rag_results_df shape: (30, 14)


,eval_id,disease_category,disease_kor,disease_eng,intention,question_type,question,ground_truth_doc_id,reference_answer,pred_answer,retrieved_doc_ids,retrieved_scores,top_k,model_name
0,EVAL_0001,소아청소년질환,가와사키병,Kawasaki Disease,약물,medication,가와사키병의 약물치료가 효과적인가요?,가와사키병__약물,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및...",가와사키병의 약물 치료는 주로 증상 완화와 관상동맥 합병증 예방을 목표로 합니다. ...,"[가와사키병__치료, 가와사키병__예방, 가와사키병__약물]","[0.595293402671814, 0.5232875347137451, 0.5002...",3,gpt-4o-mini
1,EVAL_0002,소아청소년질환,가와사키병,Kawasaki Disease,약물,medication,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,가와사키병__약물,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및...","죄송하지만, 제공된 검색 문서에는 가와사키병에 대한 약물 치료가 임신 또는 수유 중...","[과숙아__원인, 신생아_황달__치료, 신생아_경련__치료]","[0.6249063014984131, 0.6198438405990601, 0.607...",3,gpt-4o-mini
2,EVAL_0003,소아청소년질환,가와사키병,Kawasaki Disease,예방,prevention,가와사키병에 대한 예방법을 알고 싶어요.,가와사키병__예방,"가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 개...",가와사키병은 현재 예방 방법이 개발되지 않은 질환입니다. 하지만 개인 위생 관리를 ...,"[가와사키병__예방, 가와사키병__원인, 가와사키병__치료]","[0.6530215740203857, 0.5434562563896179, 0.524...",3,gpt-4o-mini
3,EVAL_0004,소아청소년질환,가와사키병,Kawasaki Disease,예방,prevention,가와사키병의 예방에 있어서 유전적 요인이 어떤 역할을 하는지 알려주세요. 예방에 효...,가와사키병__예방,"가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 개...",가와사키병의 예방에 있어 유전적 요인은 중요한 역할을 할 수 있습니다. 이 질환은 ...,"[가와사키병__원인, 가와사키병__예방, 가와사키병__치료]","[0.6852616667747498, 0.6082280278205872, 0.478...",3,gpt-4o-mini
4,EVAL_0005,소아청소년질환,가와사키병,Kawasaki Disease,원인,cause,가와사키병의 원인은 무엇인가요?,가와사키병__원인,"가와사키병은 몇 가지 가설과 연구 결과가 있지만, 아직까지 확실히 입증된 것은 없습...","가와사키병의 원인은 아직 명확히 밝혀지지 않았습니다. 여러 연구에 따르면, 유전적 ...","[가와사키병__원인, 성홍열__원인, 유행성_이하선염__원인]","[0.6906836628913879, 0.46721524000167847, 0.46...",3,gpt-4o-mini
5,EVAL_0006,소아청소년질환,가와사키병,Kawasaki Disease,원인,cause,가와사키병이 유전적 요인과 관련이 있다는 말을 듣고 있습니다. 저희 부부도 가와사키...,가와사키병__원인,"가와사키병은 몇 가지 가설과 연구 결과가 있지만, 아직까지 확실히 입증된 것은 없습...","가와사키병의 원인은 아직 명확히 밝혀지지 않았지만, 유전적 요인과 환경적 요인이 함...","[가와사키병__원인, 영아_비대성_유문협착증__원인, 가와사키병__예방]","[0.7583474516868591, 0.48252302408218384, 0.47...",3,gpt-4o-mini
6,EVAL_0007,소아청소년질환,가와사키병,Kawasaki Disease,정의,definition,가와사키병이란 무엇인가요?,가와사키병__정의,가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 주...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...,"[가와사키병__원인, 가와사키병__치료, 가와사키병__진단]","[0.6395149230957031, 0.5364222526550293, 0.519...",3,gpt-4o-mini
7,EVAL_0008,소아청소년질환,가와사키병,Kawasaki Disease,정의,definition,가와사키병이 어떤 의미를 가지는 질병인지 알고 싶어요. 가와사키병에 대한 정확한 정...,가와사키병__정의,가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 주...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질병...,"[가와사키병__정의, 성홍열__정의, 가와사키병__진단]","[0.6755939722061157, 0.6407908797264099, 0.627...",3,gpt-4o-mini
8,EVAL_0009,소아청소년질환,가와사키병,Kawasaki Disease,증상,symptom,가와사키병이란 어떤 질병인가요?,가와사키병__증상,"가와사키병은 5일 이상 지속되는 고열, 손과 발 부종, 다양한 형태의 피부 발진 등...",가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질병...,"[가와사키병__원인, 가와사키병__정의, 가와사키병__진단]","[0.6572088003158569, 0.5309429168701172, 0.527...",3,gpt-4o-mini
9,EVAL_0010,소아청소년질환,가와사키병,Kawasaki Disease,증상,symptom,가와사키병에 대해 자세히 알고 싶어요. 어린 아이의 열과 피부와 얼굴이 붉어지는 것...,가와사키병__증상,"가와사키병은 5일 이상 지속되는 고열, 손과 발 부종, 다양한 형태의 피부 발진 등...","가와사키병은 주로 어린이에게 발생하는 급성 열성 혈관염으로, 초기 증상으로는 고열,...","[성홍열__원인, 가와사키병__약물, 가와사키병__예방]","[0.6958668231964111, 0.639078676700592, 0.6330...",3,gpt-4o-mini


In [241]:
# ============================================================
# [STEP 5 - CELL 15] 검색 hit 여부 간단 점검
# ------------------------------------------------------------
# ground_truth_doc_id가 retrieved_doc_ids 상위 결과에 포함되는지 간단히 본다.
# baseline retrieval 품질의 대략적인 감을 볼 수 있다.
# ============================================================

def check_hit(row):
    gt = row["ground_truth_doc_id"]
    retrieved = row["retrieved_doc_ids"]
    return gt in retrieved

naive_rag_results_df["hit_at_k"] = naive_rag_results_df.apply(check_hit, axis=1)

hit_rate = naive_rag_results_df["hit_at_k"].mean()

print(f"[INFO] Hit@{NAIVE_TOP_K}: {hit_rate:.4f}")

display(
    naive_rag_results_df[
        ["eval_id", "disease_kor", "intention", "ground_truth_doc_id", "retrieved_doc_ids", "hit_at_k"]
    ].head(10)
)

[INFO] Hit@3: 0.4667


,eval_id,disease_kor,intention,ground_truth_doc_id,retrieved_doc_ids,hit_at_k
0,EVAL_0001,가와사키병,약물,가와사키병__약물,"[가와사키병__치료, 가와사키병__예방, 가와사키병__약물]",True
1,EVAL_0002,가와사키병,약물,가와사키병__약물,"[과숙아__원인, 신생아_황달__치료, 신생아_경련__치료]",False
2,EVAL_0003,가와사키병,예방,가와사키병__예방,"[가와사키병__예방, 가와사키병__원인, 가와사키병__치료]",True
3,EVAL_0004,가와사키병,예방,가와사키병__예방,"[가와사키병__원인, 가와사키병__예방, 가와사키병__치료]",True
4,EVAL_0005,가와사키병,원인,가와사키병__원인,"[가와사키병__원인, 성홍열__원인, 유행성_이하선염__원인]",True
5,EVAL_0006,가와사키병,원인,가와사키병__원인,"[가와사키병__원인, 영아_비대성_유문협착증__원인, 가와사키병__예방]",True
6,EVAL_0007,가와사키병,정의,가와사키병__정의,"[가와사키병__원인, 가와사키병__치료, 가와사키병__진단]",False
7,EVAL_0008,가와사키병,정의,가와사키병__정의,"[가와사키병__정의, 성홍열__정의, 가와사키병__진단]",True
8,EVAL_0009,가와사키병,증상,가와사키병__증상,"[가와사키병__원인, 가와사키병__정의, 가와사키병__진단]",False
9,EVAL_0010,가와사키병,증상,가와사키병__증상,"[성홍열__원인, 가와사키병__약물, 가와사키병__예방]",False


In [244]:
# ============================================================
# [STEP 5 - CELL 16] 결과 저장
# ------------------------------------------------------------
# 이후 STEP 8(RAGAS), STEP 9(G-EVAL)에서 사용할 수 있도록 저장한다.
# ============================================================

NAIVE_RESULTS_JSONL_PATH = OUTPUT_DIR / "naive_rag_results_v1.jsonl"
NAIVE_RESULTS_CSV_PATH   = OUTPUT_DIR / "naive_rag_results_v1.csv"

naive_result_records = naive_rag_results_df.to_dict(orient="records")
save_jsonl(naive_result_records, NAIVE_RESULTS_JSONL_PATH)
naive_rag_results_df.to_csv(NAIVE_RESULTS_CSV_PATH, index=False, encoding="utf-8-sig")

print("[INFO] 저장 완료")
print(" -", NAIVE_RESULTS_JSONL_PATH)
print(" -", NAIVE_RESULTS_CSV_PATH)

[INFO] 저장 완료
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\outputs\naive_rag_results_v1.jsonl
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\outputs\naive_rag_results_v1.csv


In [245]:
# ============================================================
# [STEP 5 - CELL 17] STEP 5 요약
# ------------------------------------------------------------
# 여기까지 완료되면 baseline naive RAG 결과가 준비된다.
# 이후 STEP 6에서는 hybrid / reranker 기반 advanced RAG를 만든다.
# ============================================================

step5_summary = {
    "embed_model_name": EMBED_MODEL_NAME,
    "gen_model_name": GEN_MODEL_NAME,
    "top_k": NAIVE_TOP_K,
    "temperature": NAIVE_TEMPERATURE,
    "num_results": int(len(naive_rag_results_df)),
    "hit_at_k": float(hit_rate),
    "results_jsonl_path": str(NAIVE_RESULTS_JSONL_PATH),
}

print(json.dumps(step5_summary, ensure_ascii=False, indent=2))

{
  "embed_model_name": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
  "gen_model_name": "gpt-4o-mini",
  "top_k": 3,
  "temperature": 0.0,
  "num_results": 30,
  "hit_at_k": 0.4666666666666667,
  "results_jsonl_path": "D:\\PyProject\\AIFFEL_AI\\LLM\\NLP\\RAG_proj\\outputs\\naive_rag_results_v1.jsonl"
}


### STEP 6. Retrieval 고도화 실험
#### 6-1. Metadata filtering: 질환명, intention 추정이 가능하면 필터를 걸음
- 질문에 “가와사키병”이 있으면 해당 질환 우선
- 질문에 “약물/치료/예방” 의도가 보이면 해당 intention 우선
#### 6-2. Hybrid Retrieval
- dense retrieval
- BM25 retrieval
- 둘을 합친 hybrid
- 의료 QA는 용어 매칭이 중요해서 BM25가 꽤 강하므로 hybrid는 거의 필수 후보
#### 6-3. Reranker
- 1차 검색 결과를 reranker로 재정렬합니다.
- 질환명이 비슷한 문서, 치료/약물/예방이 헷갈리는 문서를 정리
#### 6-4. Context compression
- 가져온 문서 중 필요한 부분만 줄여서 LLM에 넣음
#### 6-5. Multi-query / Query rewrite
- 질문을 2~4개 형태로 다시 써서 검색합니다.

## 멈춰서 커널 재시작

In [1]:
# ============================================================
# [STEP 6 - 복구 CELL A] 기본 import / 유틸 복구
# ------------------------------------------------------------
# 커널 재시작 후 STEP 6부터 이어가기 위해 필요한 최소 import와
# 공통 유틸 함수를 다시 정의한다.
# ============================================================

import os
import re
import json
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

def clean_text(text):
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\u200b", " ").replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("[INFO] STEP 6 복구용 기본 import 완료")

[INFO] STEP 6 복구용 기본 import 완료


In [3]:
# ============================================================
# [STEP 6 - 복구 CELL A] 기본 import / 유틸 복구
# ------------------------------------------------------------
# 커널 재시작 후 STEP 6부터 이어가기 위해 필요한 최소 import와
# 공통 유틸 함수를 다시 정의한다.
# ============================================================

import os
import re
import json
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

def clean_text(text):
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\u200b", " ").replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("[INFO] STEP 6 복구용 기본 import 완료")

[INFO] STEP 6 복구용 기본 import 완료


In [8]:
# ============================================================
# [STEP 6 - 복구 CELL F] OpenAI client 복구
# ------------------------------------------------------------
# STEP 6 - CELL 10 이후 hybrid generation을 위해 OpenAI client를 다시 준비한다.
# ============================================================

from dotenv import load_dotenv
from openai import OpenAI

ENV_PATH = r"D:\PyProject\env_keys\.env"
load_dotenv(ENV_PATH)

openai_api_key = os.getenv("OPENAI_API_KEY")
assert openai_api_key is not None and len(openai_api_key) > 0, "OPENAI_API_KEY를 확인하세요."

client = OpenAI(api_key=openai_api_key)

GEN_MODEL_NAME = "gpt-4o-mini"

SYSTEM_PROMPT_NAIVE_RAG = """
당신은 소아과 상담 보조 AI입니다.
반드시 제공된 검색 문서(context)에 근거해서만 답변하세요.

규칙:
1. 검색 문서에 있는 정보 중심으로 답변하세요.
2. 검색 문서에 없는 내용을 추측해서 단정하지 마세요.
3. 보호자가 이해하기 쉬운 한국어 존댓말로 답변하세요.
4. 진단을 확정하는 표현은 피하세요.
5. 위험하거나 증상이 심한 경우에는 소아청소년과 진료가 필요할 수 있다고 안내하세요.
6. 답변은 핵심 위주로 4~8문장 정도로 작성하세요.
""".strip()

def build_user_prompt_naive_rag(question, context):
    return f"""
[사용자 질문]
{question}

[검색 문서]
{context}

위 검색 문서를 바탕으로 사용자 질문에 답변해주세요.
""".strip()

def build_context_from_retrieval_results(results):
    context_blocks = []

    for r in results:
        block = (
            f"[문서 {r['rank']}]\n"
            f"doc_id: {r['doc_id']}\n"
            f"질환명: {r['disease_kor']}\n"
            f"상담주제: {r['intention']}\n"
            f"유사도점수: {r['score']:.4f}\n"
            f"내용:\n{r['full_text']}"
        )
        context_blocks.append(block)

    return "\n\n" + ("=" * 80 + "\n\n").join(context_blocks)

print("[INFO] OpenAI client 복구 완료")
print("[INFO] GEN_MODEL_NAME:", GEN_MODEL_NAME)

[INFO] OpenAI client 복구 완료
[INFO] GEN_MODEL_NAME: gpt-4o-mini


In [12]:
# ============================================================
# [STEP 6 - 복구 CELL B] 경로 및 데이터 로드 복구 (수정 버전)
# ============================================================

from pathlib import Path
import pandas as pd
import json

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

PROJECT_ROOT = Path(r"D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
EVAL_DIR = PROJECT_ROOT / "data" / "eval"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

RAG_DOCS_V4_JSONL_PATH = PROCESSED_DIR / "rag_docs_v4.jsonl"
EVAL_DATASET_JSONL_PATH = EVAL_DIR / "eval_dataset_v1.jsonl"
NAIVE_RESULTS_JSONL_PATH = OUTPUT_DIR / "naive_rag_results_v1.jsonl"

print("[DEBUG] 파일 존재 확인")
print("rag_docs_v4:", RAG_DOCS_V4_JSONL_PATH.exists())
print("eval_dataset:", EVAL_DATASET_JSONL_PATH.exists())
print("naive_results:", NAIVE_RESULTS_JSONL_PATH.exists())

# 👉 여기서 하나라도 False 나오면 경로 문제

rag_docs_v4 = load_jsonl(RAG_DOCS_V4_JSONL_PATH)
rag_docs_v4_df = pd.DataFrame(rag_docs_v4)

eval_dataset_records = load_jsonl(EVAL_DATASET_JSONL_PATH)
eval_dataset_df = pd.DataFrame(eval_dataset_records)

naive_rag_results = load_jsonl(NAIVE_RESULTS_JSONL_PATH)
naive_rag_results_df = pd.DataFrame(naive_rag_results)

print("[INFO] rag_docs_v4_df shape:", rag_docs_v4_df.shape)
print("[INFO] eval_dataset_df shape:", eval_dataset_df.shape)
print("[INFO] naive_rag_results_df shape:", naive_rag_results_df.shape)

display(rag_docs_v4_df.head(2))

[DEBUG] 파일 존재 확인
rag_docs_v4: True
eval_dataset: True
naive_results: True
[INFO] rag_docs_v4_df shape: (139, 16)
[INFO] eval_dataset_df shape: (278, 9)
[INFO] naive_rag_results_df shape: (30, 15)


,doc_id,disease_category,disease_kor,disease_eng,intention,departments,question_examples,answer_texts,question_count,answer_count,full_text,char_len,question_example_count,answer_text_count,raw_question_count,raw_answer_count
0,가와사키병__약물,소아청소년질환,가와사키병,Kawasaki Disease,약물,[소아청소년과],"[가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다., 가와사키...","[가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 ...",8,6,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,4698,8,6,406,1358
1,가와사키병__예방,소아청소년질환,가와사키병,Kawasaki Disease,예방,[소아청소년과],"[가와사키병에 대한 예방법을 알고 싶어요., 가와사키병을 예방하기 위한 생활 습관이...","[가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 ...",8,4,[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki Di...,3081,8,4,221,160


In [14]:
# ============================================================
# [STEP 6 - 복구 CELL C] 문서 텍스트 / 임베딩 모델 / 문서 임베딩 복구
# ------------------------------------------------------------
# STEP 6 - CELL 6부터는 dense score 계산이 필요하므로
# 문서 리스트, 임베딩 모델, 문서 임베딩을 다시 준비한다.
# ============================================================

doc_ids = rag_docs_v4_df["doc_id"].tolist()
doc_texts = rag_docs_v4_df["full_text"].tolist()

EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

doc_embeddings = embed_model.encode(
    doc_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("[INFO] EMBED_MODEL_NAME:", EMBED_MODEL_NAME)
print("[INFO] doc_embeddings shape:", doc_embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

[INFO] EMBED_MODEL_NAME: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
[INFO] doc_embeddings shape: (139, 768)


In [16]:
# ============================================================
# [STEP 6 - 복구 CELL D] BM25 준비
# ------------------------------------------------------------
# STEP 6 - CELL 6부터 BM25 score 계산이 필요하므로
# 간단 토크나이저와 BM25 인덱스를 다시 만든다.
# ============================================================

def simple_ko_tokenize(text):
    text = clean_text(text).lower()
    text = re.sub(r"[^0-9a-zA-Z가-힣\s]", " ", text)
    tokens = text.split()
    return [t for t in tokens if len(t) >= 1]

bm25_corpus_tokens = [simple_ko_tokenize(text) for text in doc_texts]
bm25 = BM25Okapi(bm25_corpus_tokens)

print("[INFO] BM25 인덱스 구축 완료")
print("[INFO] corpus size:", len(bm25_corpus_tokens))

[INFO] BM25 인덱스 구축 완료
[INFO] corpus size: 139


In [18]:
# ============================================================
# [STEP 6 - 복구 CELL E] metadata boosting 함수 복구
# ------------------------------------------------------------
# STEP 6 - CELL 6부터 metadata score 계산이 필요하므로
# 질환명 / intention 추정 함수와 metadata boost 함수를 다시 만든다.
# ============================================================

ALL_DISEASES = sorted(rag_docs_v4_df["disease_kor"].dropna().unique().tolist())
ALL_INTENTIONS = sorted(rag_docs_v4_df["intention"].dropna().unique().tolist())

INTENTION_HINT_KEYWORDS = {
    "약물": ["약", "약물", "복용", "투여", "처방", "부작용", "아스피린", "면역글로불린"],
    "예방": ["예방", "예방법", "막", "위생", "생활 습관", "예방접종", "백신"],
    "원인": ["원인", "왜", "이유", "유전", "감염", "면역", "발생"],
    "정의": ["무엇", "이란", "란", "뜻", "정의", "의미"],
    "증상": ["증상", "징후", "발열", "발진", "통증", "기침", "구토", "설사"],
    "진단": ["진단", "검사", "확인", "혈액", "소변", "영상", "검진"],
    "치료": ["치료", "치료법", "관리", "수술", "입원", "완화"],
    "식이, 생활": ["식이", "생활", "식사", "음식", "영양", "운동", "수면"],
    "재활": ["재활", "훈련", "회복", "기능 회복"],
    "검진": ["검진", "정기 검진", "선별 검사"],
}

def detect_disease_from_query(query):
    query = clean_text(query)
    matched = [d for d in ALL_DISEASES if d in query]
    return matched

def detect_intentions_from_query(query):
    query = clean_text(query)
    matched_intentions = []

    for intent, kws in INTENTION_HINT_KEYWORDS.items():
        hits = sum(1 for kw in kws if kw in query)
        if hits > 0:
            matched_intentions.append((intent, hits))

    matched_intentions = sorted(matched_intentions, key=lambda x: x[1], reverse=True)
    return [x[0] for x in matched_intentions]

def metadata_boost_score(query, row):
    score = 0.0
    q = clean_text(query)

    detected_diseases = detect_disease_from_query(q)
    detected_intentions = detect_intentions_from_query(q)

    disease_kor = clean_text(row["disease_kor"])
    intention = clean_text(row["intention"])

    if disease_kor in detected_diseases:
        score += 1.0

    if len(detected_intentions) > 0:
        if intention == detected_intentions[0]:
            score += 0.8
        elif intention in detected_intentions:
            score += 0.4

    return score

print("[INFO] metadata boosting 함수 복구 완료")
print("[INFO] 질환 수:", len(ALL_DISEASES))
print("[INFO] intention 수:", len(ALL_INTENTIONS))

[INFO] metadata boosting 함수 복구 완료
[INFO] 질환 수: 24
[INFO] intention 수: 10


In [20]:
# ============================================================
# [STEP 6 - 복구 CELL F] OpenAI client 복구
# ------------------------------------------------------------
# STEP 6 - CELL 10 이후 hybrid generation을 위해 OpenAI client를 다시 준비한다.
# ============================================================

from dotenv import load_dotenv
from openai import OpenAI

ENV_PATH = r"D:\PyProject\env_keys\.env"
load_dotenv(ENV_PATH)

openai_api_key = os.getenv("OPENAI_API_KEY")
assert openai_api_key is not None and len(openai_api_key) > 0, "OPENAI_API_KEY를 확인하세요."

client = OpenAI(api_key=openai_api_key)

GEN_MODEL_NAME = "gpt-4o-mini"

SYSTEM_PROMPT_NAIVE_RAG = """
당신은 소아과 상담 보조 AI입니다.
반드시 제공된 검색 문서(context)에 근거해서만 답변하세요.

규칙:
1. 검색 문서에 있는 정보 중심으로 답변하세요.
2. 검색 문서에 없는 내용을 추측해서 단정하지 마세요.
3. 보호자가 이해하기 쉬운 한국어 존댓말로 답변하세요.
4. 진단을 확정하는 표현은 피하세요.
5. 위험하거나 증상이 심한 경우에는 소아청소년과 진료가 필요할 수 있다고 안내하세요.
6. 답변은 핵심 위주로 4~8문장 정도로 작성하세요.
""".strip()

def build_user_prompt_naive_rag(question, context):
    return f"""
[사용자 질문]
{question}

[검색 문서]
{context}

위 검색 문서를 바탕으로 사용자 질문에 답변해주세요.
""".strip()

def build_context_from_retrieval_results(results):
    context_blocks = []

    for r in results:
        block = (
            f"[문서 {r['rank']}]\n"
            f"doc_id: {r['doc_id']}\n"
            f"질환명: {r['disease_kor']}\n"
            f"상담주제: {r['intention']}\n"
            f"유사도점수: {r['score']:.4f}\n"
            f"내용:\n{r['full_text']}"
        )
        context_blocks.append(block)

    return "\n\n" + ("=" * 80 + "\n\n").join(context_blocks)

print("[INFO] OpenAI client 복구 완료")
print("[INFO] GEN_MODEL_NAME:", GEN_MODEL_NAME)

[INFO] OpenAI client 복구 완료
[INFO] GEN_MODEL_NAME: gpt-4o-mini


In [22]:
# ============================================================
# [STEP 6 - CELL 1] BM25 라이브러리 import
# ------------------------------------------------------------
# hybrid retrieval을 위해 BM25를 사용한다.
# 설치가 안 되어 있으면 아래 주석 해제 후 1회 설치한다.
# !pip install rank_bm25 -q
# ============================================================

from rank_bm25 import BM25Okapi

print("[INFO] rank_bm25 import 완료")

[INFO] rank_bm25 import 완료


In [24]:
# ============================================================
# [STEP 6 - CELL 2] Naive 결과 에러 버킷 분석 함수
# ------------------------------------------------------------
# 현재 naive retrieval 실패가
# - 다른 질환으로 튀는지
# - 같은 질환인데 intention이 틀리는지
# - 정답 문서가 top-k에 없는지
# 간단히 본다.
# ============================================================

def parse_doc_id(doc_id):
    parts = str(doc_id).split("__")
    if len(parts) == 2:
        return parts[0], parts[1]
    return None, None

def analyze_naive_error_bucket(results_df):
    bucket_rows = []

    for _, row in results_df.iterrows():
        gt_doc = row["ground_truth_doc_id"]
        retrieved = row["retrieved_doc_ids"]

        gt_disease, gt_intention = parse_doc_id(gt_doc)

        top1 = retrieved[0] if isinstance(retrieved, list) and len(retrieved) > 0 else None
        top1_disease, top1_intention = parse_doc_id(top1) if top1 else (None, None)

        if gt_doc in retrieved:
            bucket = "hit"
        else:
            if top1_disease != gt_disease:
                bucket = "wrong_disease"
            elif top1_intention != gt_intention:
                bucket = "wrong_intention_same_disease"
            else:
                bucket = "other_miss"

        bucket_rows.append({
            "eval_id": row["eval_id"],
            "disease_kor": row["disease_kor"],
            "intention": row["intention"],
            "ground_truth_doc_id": gt_doc,
            "top1_doc_id": top1,
            "bucket": bucket,
        })

    bucket_df = pd.DataFrame(bucket_rows)
    return bucket_df

if "naive_rag_results_df" in globals():
    naive_bucket_df = analyze_naive_error_bucket(naive_rag_results_df)
    print("[INFO] naive bucket 분포]")
    display(naive_bucket_df["bucket"].value_counts().to_frame("count"))
    display(naive_bucket_df.head(10))
else:
    print("[WARN] naive_rag_results_df가 없습니다. STEP 5 결과가 있으면 다시 실행하세요.")

[INFO] naive bucket 분포]


,count
bucket,
hit,14
wrong_disease,11
wrong_intention_same_disease,5


,eval_id,disease_kor,intention,ground_truth_doc_id,top1_doc_id,bucket
0,EVAL_0001,가와사키병,약물,가와사키병__약물,가와사키병__치료,hit
1,EVAL_0002,가와사키병,약물,가와사키병__약물,과숙아__원인,wrong_disease
2,EVAL_0003,가와사키병,예방,가와사키병__예방,가와사키병__예방,hit
3,EVAL_0004,가와사키병,예방,가와사키병__예방,가와사키병__원인,hit
4,EVAL_0005,가와사키병,원인,가와사키병__원인,가와사키병__원인,hit
5,EVAL_0006,가와사키병,원인,가와사키병__원인,가와사키병__원인,hit
6,EVAL_0007,가와사키병,정의,가와사키병__정의,가와사키병__원인,wrong_intention_same_disease
7,EVAL_0008,가와사키병,정의,가와사키병__정의,가와사키병__정의,hit
8,EVAL_0009,가와사키병,증상,가와사키병__증상,가와사키병__원인,wrong_intention_same_disease
9,EVAL_0010,가와사키병,증상,가와사키병__증상,성홍열__원인,wrong_disease


In [26]:
# ============================================================
# [STEP 6 - CELL 3] BM25용 간단 토크나이저
# ------------------------------------------------------------
# 한국어 형태소 분석기 없이도 빠르게 실험할 수 있도록
# 공백 + 기초 기호 정리 기반 토크나이저를 사용한다.
# ============================================================

import re

def simple_ko_tokenize(text):
    text = clean_text(text).lower()
    text = re.sub(r"[^0-9a-zA-Z가-힣\s]", " ", text)
    tokens = text.split()
    return [t for t in tokens if len(t) >= 1]

# 샘플
print(simple_ko_tokenize("가와사키병 치료에 사용되는 약물은 무엇인가요?"))

['가와사키병', '치료에', '사용되는', '약물은', '무엇인가요']


In [28]:
# ============================================================
# [STEP 6 - CELL 4] BM25 인덱스 구축
# ------------------------------------------------------------
# rag_docs_v4 full_text를 BM25 검색용으로 인덱싱한다.
# ============================================================

bm25_corpus_tokens = [simple_ko_tokenize(text) for text in doc_texts]
bm25 = BM25Okapi(bm25_corpus_tokens)

print("[INFO] BM25 인덱스 구축 완료")
print("[INFO] corpus size:", len(bm25_corpus_tokens))

[INFO] BM25 인덱스 구축 완료
[INFO] corpus size: 139


In [30]:
# ============================================================
# [STEP 6 - CELL 5] metadata filtering/boosting 규칙 정의
# ------------------------------------------------------------
# 질문에서 질환명 / intention 신호를 뽑아
# 해당 문서에 가점을 주는 방식으로 metadata boost를 적용한다.
# ------------------------------------------------------------
# 처음에는 hard filter가 아니라 soft boost로 시작한다.
# ============================================================

ALL_DISEASES = sorted(rag_docs_v4_df["disease_kor"].dropna().unique().tolist())
ALL_INTENTIONS = sorted(rag_docs_v4_df["intention"].dropna().unique().tolist())

INTENTION_HINT_KEYWORDS = {
    "약물": ["약", "약물", "복용", "투여", "처방", "부작용", "아스피린", "면역글로불린"],
    "예방": ["예방", "예방법", "막", "위생", "생활 습관", "예방접종", "백신"],
    "원인": ["원인", "왜", "이유", "유전", "감염", "면역", "발생"],
    "정의": ["무엇", "이란", "란", "뜻", "정의", "의미"],
    "증상": ["증상", "징후", "발열", "발진", "통증", "기침", "구토", "설사"],
    "진단": ["진단", "검사", "확인", "혈액", "소변", "영상", "검진"],
    "치료": ["치료", "치료법", "관리", "수술", "입원", "완화"],
    "식이, 생활": ["식이", "생활", "식사", "음식", "영양", "운동", "수면"],
    "재활": ["재활", "훈련", "회복", "기능 회복"],
    "검진": ["검진", "정기 검진", "선별 검사"],
}

def detect_disease_from_query(query):
    query = clean_text(query)
    matched = [d for d in ALL_DISEASES if d in query]
    return matched

def detect_intentions_from_query(query):
    query = clean_text(query)
    matched_intentions = []

    for intent, kws in INTENTION_HINT_KEYWORDS.items():
        hits = sum(1 for kw in kws if kw in query)
        if hits > 0:
            matched_intentions.append((intent, hits))

    matched_intentions = sorted(matched_intentions, key=lambda x: x[1], reverse=True)
    return [x[0] for x in matched_intentions]

def metadata_boost_score(query, row):
    score = 0.0
    q = clean_text(query)

    detected_diseases = detect_disease_from_query(q)
    detected_intentions = detect_intentions_from_query(q)

    disease_kor = clean_text(row["disease_kor"])
    intention = clean_text(row["intention"])

    # 질환명 boost
    if disease_kor in detected_diseases:
        score += 1.0

    # intention boost
    if len(detected_intentions) > 0:
        if intention == detected_intentions[0]:
            score += 0.8
        elif intention in detected_intentions:
            score += 0.4

    return score

print("[INFO] metadata boosting 함수 준비 완료")
print("[INFO] 질환 수:", len(ALL_DISEASES))
print("[INFO] intention 수:", len(ALL_INTENTIONS))

[INFO] metadata boosting 함수 준비 완료
[INFO] 질환 수: 24
[INFO] intention 수: 10


In [32]:
# ============================================================
# [STEP 6 - CELL 6] Dense / BM25 / metadata score 계산 함수
# ------------------------------------------------------------
# query 하나에 대해
# - dense similarity
# - BM25 score
# - metadata boost
# 를 각각 계산한다.
# ============================================================

def get_dense_scores(query):
    query_emb = embed_model.encode(
        [clean_text(query)],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    sims = cosine_similarity(query_emb, doc_embeddings)[0]
    return sims

def minmax_scale(arr):
    arr = np.array(arr, dtype=float)
    mn = arr.min()
    mx = arr.max()
    if mx - mn < 1e-12:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

def get_bm25_scores(query):
    q_tokens = simple_ko_tokenize(query)
    scores = bm25.get_scores(q_tokens)
    return np.array(scores, dtype=float)

def get_metadata_scores(query):
    scores = []
    for _, row in rag_docs_v4_df.iterrows():
        s = metadata_boost_score(query, row)
        scores.append(s)
    return np.array(scores, dtype=float)

print("[INFO] dense / bm25 / metadata score 함수 준비 완료")

[INFO] dense / bm25 / metadata score 함수 준비 완료


In [34]:
# ============================================================
# [STEP 6 - CELL 7] Hybrid retrieval 함수 정의
# ------------------------------------------------------------
# 최종 점수 = dense + BM25 + metadata boost 가중합
# ------------------------------------------------------------
# 초기 가중치:
# - dense     : 0.45
# - bm25      : 0.40
# - metadata  : 0.15
#
# 이후 필요 시 조정 가능
# ============================================================

HYBRID_W_DENSE = 0.45
HYBRID_W_BM25 = 0.40
HYBRID_W_META = 0.15

def retrieve_top_k_hybrid(query, top_k=3,
                          w_dense=HYBRID_W_DENSE,
                          w_bm25=HYBRID_W_BM25,
                          w_meta=HYBRID_W_META):
    dense_scores_raw = get_dense_scores(query)
    bm25_scores_raw = get_bm25_scores(query)
    meta_scores_raw = get_metadata_scores(query)

    dense_scores = minmax_scale(dense_scores_raw)
    bm25_scores = minmax_scale(bm25_scores_raw)
    meta_scores = minmax_scale(meta_scores_raw)

    final_scores = (
        w_dense * dense_scores +
        w_bm25 * bm25_scores +
        w_meta  * meta_scores
    )

    top_idx = np.argsort(final_scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_idx, start=1):
        row = rag_docs_v4_df.iloc[idx]
        results.append({
            "rank": rank,
            "doc_id": row["doc_id"],
            "disease_kor": row["disease_kor"],
            "intention": row["intention"],
            "score": float(final_scores[idx]),
            "dense_score": float(dense_scores[idx]),
            "bm25_score": float(bm25_scores[idx]),
            "meta_score": float(meta_scores[idx]),
            "full_text": row["full_text"],
        })
    return results

print("[INFO] retrieve_top_k_hybrid 함수 준비 완료")
print(f"[INFO] 가중치 dense={HYBRID_W_DENSE}, bm25={HYBRID_W_BM25}, meta={HYBRID_W_META}")

[INFO] retrieve_top_k_hybrid 함수 준비 완료
[INFO] 가중치 dense=0.45, bm25=0.4, meta=0.15


In [36]:
# ============================================================
# [STEP 6 - CELL 8] Hybrid retrieval 단건 테스트
# ------------------------------------------------------------
# naive와 비교하기 위해 같은 질문으로 hybrid 검색 결과를 확인한다.
# ============================================================

test_question = "가와사키병 치료에 사용되는 약물은 무엇인가요?"

hybrid_test_results = retrieve_top_k_hybrid(test_question, top_k=5)

print("[질문]")
print(test_question)

print("\n[Hybrid 검색 결과]")
for r in hybrid_test_results:
    print(
        f"- rank={r['rank']} | doc_id={r['doc_id']} | final={r['score']:.4f} "
        f"| dense={r['dense_score']:.4f} | bm25={r['bm25_score']:.4f} | meta={r['meta_score']:.4f}"
    )

[질문]
가와사키병 치료에 사용되는 약물은 무엇인가요?

[Hybrid 검색 결과]
- rank=1 | doc_id=가와사키병__약물 | final=0.9800 | dense=0.9555 | bm25=1.0000 | meta=1.0000
- rank=2 | doc_id=가와사키병__치료 | final=0.7932 | dense=1.0000 | bm25=0.5663 | meta=0.7778
- rank=3 | doc_id=가와사키병__예방 | final=0.7876 | dense=0.8528 | bm25=0.8013 | meta=0.5556
- rank=4 | doc_id=가와사키병__원인 | final=0.6368 | dense=0.8066 | bm25=0.4762 | meta=0.5556
- rank=5 | doc_id=유행성_이하선염__약물 | final=0.6344 | dense=0.6383 | bm25=0.7013 | meta=0.4444


In [40]:
# ============================================================
# [STEP 6 - 복구 CELL G] naive retrieval 함수 복구
# ------------------------------------------------------------
# compare_naive_vs_hybrid()에서 사용하는 retrieve_top_k_naive를
# 커널 재시작 후 다시 정의한다.
# ============================================================

def retrieve_top_k_naive(query, top_k=3):
    query = clean_text(query)

    query_emb = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    sims = cosine_similarity(query_emb, doc_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_idx, start=1):
        results.append({
            "rank": rank,
            "doc_id": doc_ids[idx],
            "score": float(sims[idx]),
            "full_text": doc_texts[idx],
            "disease_kor": rag_docs_v4_df.iloc[idx]["disease_kor"],
            "intention": rag_docs_v4_df.iloc[idx]["intention"],
        })
    return results

print("[INFO] retrieve_top_k_naive 함수 복구 완료")

[INFO] retrieve_top_k_naive 함수 복구 완료


In [42]:
# ============================================================
# [STEP 6 - CELL 9] Naive vs Hybrid 검색 비교 함수
# ------------------------------------------------------------
# 같은 질문에 대해 naive와 hybrid의 top-k 결과를 나란히 본다.
# ============================================================

def compare_naive_vs_hybrid(query, top_k=5):
    naive_results = retrieve_top_k_naive(query, top_k=top_k)
    hybrid_results = retrieve_top_k_hybrid(query, top_k=top_k)

    print("=" * 120)
    print("[질문]")
    print(query)

    print("\n[Naive]")
    for r in naive_results:
        print(f"- rank={r['rank']} | doc_id={r['doc_id']} | score={r['score']:.4f}")

    print("\n[Hybrid]")
    for r in hybrid_results:
        print(
            f"- rank={r['rank']} | doc_id={r['doc_id']} | final={r['score']:.4f} "
            f"| dense={r['dense_score']:.4f} | bm25={r['bm25_score']:.4f} | meta={r['meta_score']:.4f}"
        )

compare_naive_vs_hybrid("가와사키병 치료에 사용되는 약물은 무엇인가요?", top_k=5)
compare_naive_vs_hybrid("가와사키병의 원인은 무엇인가요?", top_k=5)
compare_naive_vs_hybrid("가와사키병 진단을 위해 어떤 검사를 하나요?", top_k=5)

[질문]
가와사키병 치료에 사용되는 약물은 무엇인가요?

[Naive]
- rank=1 | doc_id=가와사키병__치료 | score=0.5395
- rank=2 | doc_id=가와사키병__약물 | score=0.5189
- rank=3 | doc_id=가와사키병__예방 | score=0.4712
- rank=4 | doc_id=가와사키병__원인 | score=0.4498
- rank=5 | doc_id=가와사키병__진단 | score=0.4020

[Hybrid]
- rank=1 | doc_id=가와사키병__약물 | final=0.9800 | dense=0.9555 | bm25=1.0000 | meta=1.0000
- rank=2 | doc_id=가와사키병__치료 | final=0.7932 | dense=1.0000 | bm25=0.5663 | meta=0.7778
- rank=3 | doc_id=가와사키병__예방 | final=0.7876 | dense=0.8528 | bm25=0.8013 | meta=0.5556
- rank=4 | doc_id=가와사키병__원인 | final=0.6368 | dense=0.8066 | bm25=0.4762 | meta=0.5556
- rank=5 | doc_id=유행성_이하선염__약물 | final=0.6344 | dense=0.6383 | bm25=0.7013 | meta=0.4444
[질문]
가와사키병의 원인은 무엇인가요?

[Naive]
- rank=1 | doc_id=가와사키병__원인 | score=0.6907
- rank=2 | doc_id=성홍열__원인 | score=0.4672
- rank=3 | doc_id=유행성_이하선염__원인 | score=0.4640
- rank=4 | doc_id=가와사키병__예방 | score=0.4603
- rank=5 | doc_id=가와사키병__치료 | score=0.4542

[Hybrid]
- rank=1 | doc_id=가와사키병__원인 | final=1.0000 |

In [44]:
# ============================================================
# [STEP 6 - CELL 10] Hybrid generation 함수 정의
# ------------------------------------------------------------
# retrieval만 hybrid로 바꾸고, 생성 프롬프트는 baseline과 동일하게 사용한다.
# ============================================================

def generate_answer_hybrid_rag(query, top_k=3, temperature=0.0):
    retrieval_results = retrieve_top_k_hybrid(query, top_k=top_k)
    context = build_context_from_retrieval_results(retrieval_results)
    user_prompt = build_user_prompt_naive_rag(query, context)

    response = client.chat.completions.create(
        model=GEN_MODEL_NAME,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_NAIVE_RAG},
            {"role": "user", "content": user_prompt},
        ]
    )

    answer = response.choices[0].message.content.strip()

    return {
        "question": query,
        "answer": answer,
        "retrieval_results": retrieval_results,
        "context": context,
        "top_k": top_k,
        "model_name": GEN_MODEL_NAME,
    }

print("[INFO] generate_answer_hybrid_rag 함수 준비 완료")

[INFO] generate_answer_hybrid_rag 함수 준비 완료


In [46]:
# ============================================================
# [STEP 6 - CELL 11] 평가셋 일부에 대해 Hybrid 테스트
# ------------------------------------------------------------
# 먼저 소규모 샘플로 naive와 비교한다.
# ============================================================

N_DEBUG = 10
debug_rows = eval_dataset_df.head(N_DEBUG)

hybrid_debug_rows = []

for _, row in debug_rows.iterrows():
    result = generate_answer_hybrid_rag(
        query=row["question"],
        top_k=3,
        temperature=0.0
    )

    hybrid_debug_rows.append({
        "eval_id": row["eval_id"],
        "disease_kor": row["disease_kor"],
        "intention": row["intention"],
        "question": row["question"],
        "ground_truth_doc_id": row["ground_truth_doc_id"],
        "pred_answer": result["answer"],
        "retrieved_doc_ids": [x["doc_id"] for x in result["retrieval_results"]],
        "retrieved_scores": [x["score"] for x in result["retrieval_results"]],
    })

hybrid_debug_df = pd.DataFrame(hybrid_debug_rows)

print("[INFO] hybrid_debug_df shape:", hybrid_debug_df.shape)
display(hybrid_debug_df)

[INFO] hybrid_debug_df shape: (10, 8)


,eval_id,disease_kor,intention,question,ground_truth_doc_id,pred_answer,retrieved_doc_ids,retrieved_scores
0,EVAL_0001,가와사키병,약물,가와사키병의 약물치료가 효과적인가요?,가와사키병__약물,가와사키병의 약물 치료는 효과적일 수 있습니다. 주로 사용되는 약물로는 면역글로불린...,"[가와사키병__약물, 가와사키병__치료, 가와사키병__원인]","[0.9207343946860459, 0.7524539323041546, 0.637..."
1,EVAL_0002,가와사키병,약물,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,가와사키병__약물,가와사키병의 약물 치료가 임신 또는 수유 중인 여성에게 미치는 영향에 대한 구체적인...,"[가와사키병__약물, 신생아_황달__치료, 신생아_경련__정의]","[0.7835865565191377, 0.7741484076552562, 0.680..."
2,EVAL_0003,가와사키병,예방,가와사키병에 대한 예방법을 알고 싶어요.,가와사키병__예방,가와사키병은 현재로서는 예방 방법이 개발되지 않았습니다. 하지만 개인 위생 관리를 ...,"[가와사키병__예방, 가와사키병__약물, 가와사키병__원인]","[1.0, 0.6842056455611938, 0.6584282819333849]"
3,EVAL_0004,가와사키병,예방,가와사키병의 예방에 있어서 유전적 요인이 어떤 역할을 하는지 알려주세요. 예방에 효...,가와사키병__예방,가와사키병의 예방에 있어 유전적 요인은 중요한 역할을 할 수 있습니다. 연구에 따르...,"[가와사키병__원인, 가와사키병__예방, 가와사키병__치료]","[0.9666666666666668, 0.8233956498890908, 0.732..."
4,EVAL_0005,가와사키병,원인,가와사키병의 원인은 무엇인가요?,가와사키병__원인,가와사키병의 원인은 아직 명확히 밝혀지지 않았습니다. 그러나 유전적 요인과 환경적 ...,"[가와사키병__원인, 가와사키병__정의, 가와사키병__치료]","[1.0, 0.7451022934415097, 0.7171297591888833]"
5,EVAL_0006,가와사키병,원인,가와사키병이 유전적 요인과 관련이 있다는 말을 듣고 있습니다. 저희 부부도 가와사키...,가와사키병__원인,"가와사키병의 원인은 아직 명확히 밝혀지지 않았지만, 유전적 요인과 환경적 요인이 함...","[가와사키병__원인, 영아_비대성_유문협착증__원인, 가와사키병__진단]","[1.0, 0.5005331338439302, 0.412682136329142]"
6,EVAL_0007,가와사키병,정의,가와사키병이란 무엇인가요?,가와사키병__정의,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...,"[가와사키병__정의, 가와사키병__원인, 가와사키병__증상]","[0.8940441877314097, 0.6169178803409109, 0.582..."
7,EVAL_0008,가와사키병,정의,가와사키병이 어떤 의미를 가지는 질병인지 알고 싶어요. 가와사키병에 대한 정확한 정...,가와사키병__정의,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질병...,"[가와사키병__정의, 가와사키병__원인, 라이_증후군__정의]","[0.9714514792634685, 0.7343909064395698, 0.651..."
8,EVAL_0009,가와사키병,증상,가와사키병이란 어떤 질병인가요?,가와사키병__증상,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질병...,"[가와사키병__정의, 가와사키병__증상, 가와사키병__원인]","[0.8905716024403058, 0.5552907567906413, 0.533..."
9,EVAL_0010,가와사키병,증상,가와사키병에 대해 자세히 알고 싶어요. 어린 아이의 열과 피부와 얼굴이 붉어지는 것...,가와사키병__증상,"가와사키병은 주로 어린이에게 발생하는 급성 열성 혈관염으로, 5세 이하의 아이들에게...","[가와사키병__예방, 가와사키병__증상, 가와사키병__정의]","[0.8630187217250941, 0.7488104858323924, 0.728..."


In [48]:
# ============================================================
# [STEP 6 - CELL 12] Hybrid inference 함수 정의
# ------------------------------------------------------------
# eval_dataset_df 전체 또는 일부에 대해 hybrid RAG 결과를 생성한다.
# ============================================================

def run_hybrid_rag_inference(eval_df, top_k=3, temperature=0.0, limit=None):
    rows = eval_df.copy()

    if limit is not None:
        rows = rows.head(limit).copy()

    output_rows = []

    for i, (_, row) in enumerate(rows.iterrows(), start=1):
        result = generate_answer_hybrid_rag(
            query=row["question"],
            top_k=top_k,
            temperature=temperature
        )

        output_rows.append({
            "eval_id": row["eval_id"],
            "disease_category": row["disease_category"],
            "disease_kor": row["disease_kor"],
            "disease_eng": row["disease_eng"],
            "intention": row["intention"],
            "question_type": row["question_type"],
            "question": row["question"],
            "ground_truth_doc_id": row["ground_truth_doc_id"],
            "reference_answer": row["reference_answer"],
            "pred_answer": result["answer"],
            "retrieved_doc_ids": [x["doc_id"] for x in result["retrieval_results"]],
            "retrieved_scores": [x["score"] for x in result["retrieval_results"]],
            "top_k": top_k,
            "model_name": GEN_MODEL_NAME,
        })

        if i % 20 == 0:
            print(f"[INFO] {i}/{len(rows)} 완료")

    return pd.DataFrame(output_rows)

print("[INFO] run_hybrid_rag_inference 함수 준비 완료")

[INFO] run_hybrid_rag_inference 함수 준비 완료


In [50]:
# ============================================================
# [STEP 6 - CELL 13] Hybrid RAG 실행
# ------------------------------------------------------------
# 처음에는 limit를 작게 두고 hit@k 개선 여부를 확인한다.
# ============================================================

HYBRID_TOP_K = 3
HYBRID_TEMPERATURE = 0.0
HYBRID_LIMIT = 30

hybrid_rag_results_df = run_hybrid_rag_inference(
    eval_df=eval_dataset_df,
    top_k=HYBRID_TOP_K,
    temperature=HYBRID_TEMPERATURE,
    limit=HYBRID_LIMIT
)

print("[INFO] hybrid_rag_results_df shape:", hybrid_rag_results_df.shape)
display(hybrid_rag_results_df.head(10))

[INFO] 20/30 완료
[INFO] hybrid_rag_results_df shape: (30, 14)


,eval_id,disease_category,disease_kor,disease_eng,intention,question_type,question,ground_truth_doc_id,reference_answer,pred_answer,retrieved_doc_ids,retrieved_scores,top_k,model_name
0,EVAL_0001,소아청소년질환,가와사키병,Kawasaki Disease,약물,medication,가와사키병의 약물치료가 효과적인가요?,가와사키병__약물,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및...",가와사키병의 약물치료는 효과적일 수 있습니다. 주로 사용되는 약물로는 면역글로불린과...,"[가와사키병__약물, 가와사키병__치료, 가와사키병__원인]","[0.9207343946860459, 0.7524539323041546, 0.637...",3,gpt-4o-mini
1,EVAL_0002,소아청소년질환,가와사키병,Kawasaki Disease,약물,medication,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,가와사키병__약물,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및...",가와사키병의 약물 치료가 임신 또는 수유 중인 여성에게 미치는 영향에 대한 구체적인...,"[가와사키병__약물, 신생아_황달__치료, 신생아_경련__정의]","[0.7835865565191377, 0.7741484076552562, 0.680...",3,gpt-4o-mini
2,EVAL_0003,소아청소년질환,가와사키병,Kawasaki Disease,예방,prevention,가와사키병에 대한 예방법을 알고 싶어요.,가와사키병__예방,"가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 개...",가와사키병은 현재로서는 예방 방법이 확립되어 있지 않습니다. 하지만 개인 위생 관리...,"[가와사키병__예방, 가와사키병__약물, 가와사키병__원인]","[1.0, 0.6842056455611938, 0.6584282819333849]",3,gpt-4o-mini
3,EVAL_0004,소아청소년질환,가와사키병,Kawasaki Disease,예방,prevention,가와사키병의 예방에 있어서 유전적 요인이 어떤 역할을 하는지 알려주세요. 예방에 효...,가와사키병__예방,"가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 개...",가와사키병의 예방에 있어 유전적 요인은 중요한 역할을 할 수 있습니다. 연구에 따르...,"[가와사키병__원인, 가와사키병__예방, 가와사키병__치료]","[0.9666666666666668, 0.8233956498890908, 0.732...",3,gpt-4o-mini
4,EVAL_0005,소아청소년질환,가와사키병,Kawasaki Disease,원인,cause,가와사키병의 원인은 무엇인가요?,가와사키병__원인,"가와사키병은 몇 가지 가설과 연구 결과가 있지만, 아직까지 확실히 입증된 것은 없습...",가와사키병의 원인은 아직 명확히 밝혀지지 않았습니다. 그러나 유전적 요인과 환경적 ...,"[가와사키병__원인, 가와사키병__정의, 가와사키병__치료]","[1.0, 0.7451022934415097, 0.7171297591888833]",3,gpt-4o-mini
5,EVAL_0006,소아청소년질환,가와사키병,Kawasaki Disease,원인,cause,가와사키병이 유전적 요인과 관련이 있다는 말을 듣고 있습니다. 저희 부부도 가와사키...,가와사키병__원인,"가와사키병은 몇 가지 가설과 연구 결과가 있지만, 아직까지 확실히 입증된 것은 없습...","가와사키병의 원인은 아직 명확히 밝혀지지 않았지만, 유전적 요인과 환경적 요인이 함...","[가와사키병__원인, 영아_비대성_유문협착증__원인, 가와사키병__진단]","[1.0, 0.5005331338439302, 0.412682136329142]",3,gpt-4o-mini
6,EVAL_0007,소아청소년질환,가와사키병,Kawasaki Disease,정의,definition,가와사키병이란 무엇인가요?,가와사키병__정의,가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 주...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...,"[가와사키병__정의, 가와사키병__원인, 가와사키병__증상]","[0.8940441877314097, 0.6169178803409109, 0.582...",3,gpt-4o-mini
7,EVAL_0008,소아청소년질환,가와사키병,Kawasaki Disease,정의,definition,가와사키병이 어떤 의미를 가지는 질병인지 알고 싶어요. 가와사키병에 대한 정확한 정...,가와사키병__정의,가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 주...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질병...,"[가와사키병__정의, 가와사키병__원인, 라이_증후군__정의]","[0.9714514792634685, 0.7343909064395698, 0.651...",3,gpt-4o-mini
8,EVAL_0009,소아청소년질환,가와사키병,Kawasaki Disease,증상,symptom,가와사키병이란 어떤 질병인가요?,가와사키병__증상,"가와사키병은 5일 이상 지속되는 고열, 손과 발 부종, 다양한 형태의 피부 발진 등...",가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질병...,"[가와사키병__정의, 가와사키병__증상, 가와사키병__원인]","[0.8905716024403058, 0.5552907567906413, 0.533...",3,gpt-4o-mini
9,EVAL_0010,소아청소년질환,가와사키병,Kawasaki Disease,증상,symptom,가와사키병에 대해 자세히 알고 싶어요. 어린 아이의 열과 피부와 얼굴이 붉어지는 것...,가와사키병__증상,"가와사키병은 5일 이상 지속되는 고열, 손과 발 부종, 다양한 형태의 피부 발진 등...","가와사키병은 주로 어린이에게 발생하는 급성 열성 혈관염으로, 5세 이하의 아이들에게...","[가와사키병__예방, 가와사키병__증상, 가와사키병__정의]","[0.8630187217250941, 0.7488104858323924, 0.728...",3,gpt-4o-mini


In [51]:
# ============================================================
# [STEP 6 - CELL 14] Hybrid Hit@K 계산 및 Naive와 비교
# ============================================================

def check_hit(row):
    gt = row["ground_truth_doc_id"]
    retrieved = row["retrieved_doc_ids"]
    return gt in retrieved

hybrid_rag_results_df["hit_at_k"] = hybrid_rag_results_df.apply(check_hit, axis=1)
hybrid_hit_rate = hybrid_rag_results_df["hit_at_k"].mean()

print(f"[INFO] Hybrid Hit@{HYBRID_TOP_K}: {hybrid_hit_rate:.4f}")

if "naive_rag_results_df" in globals():
    naive_compare = naive_rag_results_df.copy()
    if "hit_at_k" not in naive_compare.columns:
        naive_compare["hit_at_k"] = naive_compare.apply(check_hit, axis=1)
    naive_hit_rate = naive_compare["hit_at_k"].mean()

    compare_summary_df = pd.DataFrame([
        {"system": "naive_dense_only", "hit_at_k": naive_hit_rate, "n": len(naive_compare)},
        {"system": "hybrid_dense_bm25_meta", "hit_at_k": hybrid_hit_rate, "n": len(hybrid_rag_results_df)},
    ])
    display(compare_summary_df)
else:
    print("[WARN] naive_rag_results_df가 없어 naive 비교는 생략합니다.")

[INFO] Hybrid Hit@3: 0.8333


,system,hit_at_k,n
0,naive_dense_only,0.466667,30
1,hybrid_dense_bm25_meta,0.833333,30


In [54]:
# ============================================================
# [STEP 6 - CELL 15] Hybrid 결과 저장
# ============================================================

HYBRID_RESULTS_JSONL_PATH = OUTPUT_DIR / "hybrid_rag_results_v1.jsonl"
HYBRID_RESULTS_CSV_PATH   = OUTPUT_DIR / "hybrid_rag_results_v1.csv"

hybrid_result_records = hybrid_rag_results_df.to_dict(orient="records")
save_jsonl(hybrid_result_records, HYBRID_RESULTS_JSONL_PATH)
hybrid_rag_results_df.to_csv(HYBRID_RESULTS_CSV_PATH, index=False, encoding="utf-8-sig")

print("[INFO] 저장 완료")
print(" -", HYBRID_RESULTS_JSONL_PATH)
print(" -", HYBRID_RESULTS_CSV_PATH)

[INFO] 저장 완료
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\outputs\hybrid_rag_results_v1.jsonl
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\outputs\hybrid_rag_results_v1.csv


In [56]:
# ============================================================
# [STEP 6 - CELL 16] STEP 6-1 / 6-2 요약
# ============================================================

step6_summary = {
    "dense_weight": HYBRID_W_DENSE,
    "bm25_weight": HYBRID_W_BM25,
    "meta_weight": HYBRID_W_META,
    "top_k": HYBRID_TOP_K,
    "num_results": int(len(hybrid_rag_results_df)),
    "hybrid_hit_at_k": float(hybrid_hit_rate),
    "results_jsonl_path": str(HYBRID_RESULTS_JSONL_PATH),
}

print(json.dumps(step6_summary, ensure_ascii=False, indent=2))

{
  "dense_weight": 0.45,
  "bm25_weight": 0.4,
  "meta_weight": 0.15,
  "top_k": 3,
  "num_results": 30,
  "hybrid_hit_at_k": 0.8333333333333334,
  "results_jsonl_path": "D:\\PyProject\\AIFFEL_AI\\LLM\\NLP\\RAG_proj\\outputs\\hybrid_rag_results_v1.jsonl"
}


### [STEP 6-3 - CELL 1] reranker

In [59]:
# ============================================================
# [STEP 6-3 - CELL 1] reranker 라이브러리 import
# ------------------------------------------------------------
# sentence-transformers의 CrossEncoder를 사용한다.
# 설치가 안 되어 있으면 아래 주석 해제 후 1회 설치한다.
# !pip install sentence-transformers -q
# ============================================================

from sentence_transformers import CrossEncoder

print("[INFO] CrossEncoder import 완료")

[INFO] CrossEncoder import 완료


In [61]:
# ============================================================
# [STEP 6-3 - CELL 2] reranker 모델 로드
# ------------------------------------------------------------
# 다국어 계열 cross-encoder reranker를 사용한다.
# 너무 무거우면 나중에 더 작은 모델로 바꿀 수 있다.
# ============================================================

RERANKER_MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

reranker_model = CrossEncoder(RERANKER_MODEL_NAME, max_length=512)

print("[INFO] reranker 모델 로드 완료:", RERANKER_MODEL_NAME)

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

[INFO] reranker 모델 로드 완료: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


In [62]:
# ============================================================
# [STEP 6-3 - CELL 2] reranker 모델 로드
# ------------------------------------------------------------
# 다국어 계열 cross-encoder reranker를 사용한다.
# 너무 무거우면 나중에 더 작은 모델로 바꿀 수 있다.
# ============================================================

RERANKER_MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

reranker_model = CrossEncoder(RERANKER_MODEL_NAME, max_length=512)

print("[INFO] reranker 모델 로드 완료:", RERANKER_MODEL_NAME)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] reranker 모델 로드 완료: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


In [65]:
# ============================================================
# [STEP 6-3 - CELL 3] hybrid 후보 재정렬 함수
# ------------------------------------------------------------
# 1차 후보는 hybrid retrieval로 뽑고,
# query-document pair를 cross-encoder로 다시 점수화하여 재정렬한다.
# ============================================================

def rerank_hybrid_candidates(query, initial_top_n=8, final_top_k=3):
    # 1차 후보: hybrid
    hybrid_candidates = retrieve_top_k_hybrid(query, top_k=initial_top_n)

    # reranker 입력 pair
    pairs = [(clean_text(query), clean_text(c["full_text"])) for c in hybrid_candidates]

    rerank_scores = reranker_model.predict(pairs)

    reranked = []
    for cand, rscore in zip(hybrid_candidates, rerank_scores):
        item = dict(cand)
        item["rerank_score"] = float(rscore)
        reranked.append(item)

    reranked = sorted(reranked, key=lambda x: x["rerank_score"], reverse=True)
    reranked = reranked[:final_top_k]

    # rank 다시 부여
    for i, item in enumerate(reranked, start=1):
        item["rank"] = i

    return reranked

print("[INFO] rerank_hybrid_candidates 함수 준비 완료")

[INFO] rerank_hybrid_candidates 함수 준비 완료


In [67]:
# ============================================================
# [STEP 6-3 - CELL 3] hybrid 후보 재정렬 함수
# ------------------------------------------------------------
# 1차 후보는 hybrid retrieval로 뽑고,
# query-document pair를 cross-encoder로 다시 점수화하여 재정렬한다.
# ============================================================

def rerank_hybrid_candidates(query, initial_top_n=8, final_top_k=3):
    # 1차 후보: hybrid
    hybrid_candidates = retrieve_top_k_hybrid(query, top_k=initial_top_n)

    # reranker 입력 pair
    pairs = [(clean_text(query), clean_text(c["full_text"])) for c in hybrid_candidates]

    rerank_scores = reranker_model.predict(pairs)

    reranked = []
    for cand, rscore in zip(hybrid_candidates, rerank_scores):
        item = dict(cand)
        item["rerank_score"] = float(rscore)
        reranked.append(item)

    reranked = sorted(reranked, key=lambda x: x["rerank_score"], reverse=True)
    reranked = reranked[:final_top_k]

    # rank 다시 부여
    for i, item in enumerate(reranked, start=1):
        item["rank"] = i

    return reranked

print("[INFO] rerank_hybrid_candidates 함수 준비 완료")

[INFO] rerank_hybrid_candidates 함수 준비 완료


In [69]:
# ============================================================
# [STEP 6-4 - CELL 5] passage 분할 및 간단 점수 함수
# ------------------------------------------------------------
# 문서 full_text를 passage 단위로 나누고,
# query와의 lexical overlap + metadata 힌트로 간단 점수화한다.
# ============================================================

def simple_ko_tokenize(text):
    text = clean_text(text).lower()
    text = re.sub(r"[^0-9a-zA-Z가-힣\s]", " ", text)
    tokens = text.split()
    return [t for t in tokens if len(t) >= 1]

def split_into_passages(full_text):
    full_text = clean_text(full_text)

    # 1차: 빈 줄 기준
    chunks = re.split(r"\n\s*\n", full_text)

    passages = []
    for ch in chunks:
        ch = clean_text(ch)
        if not ch:
            continue

        # 너무 길면 줄 단위로 한 번 더 분할
        if len(ch) > 700:
            lines = [clean_text(x) for x in ch.split("\n") if clean_text(x)]
            tmp = []
            cur = ""
            for line in lines:
                if len(cur) + len(line) <= 500:
                    cur = (cur + "\n" + line).strip()
                else:
                    if cur:
                        passages.append(cur)
                    cur = line
            if cur:
                passages.append(cur)
        else:
            passages.append(ch)

    return passages

def lexical_overlap_score(query, passage):
    q_tokens = set(simple_ko_tokenize(query))
    p_tokens = set(simple_ko_tokenize(passage))

    if len(q_tokens) == 0:
        return 0.0

    overlap = len(q_tokens & p_tokens) / len(q_tokens)
    return overlap

print("[INFO] passage 분할 / lexical overlap 함수 준비 완료")

[INFO] passage 분할 / lexical overlap 함수 준비 완료


###  [STEP 6-4 - CELL 6] reranked 문서 기반 context compression

In [72]:
# ============================================================
# [STEP 6-4 - CELL 6] reranked 문서 기반 context compression
# ------------------------------------------------------------
# reranked 상위 문서들에서 query와 관련성이 높은 passage만 추려
# 더 짧고 밀도 높은 context를 만든다.
# ============================================================

def compress_context_from_reranked_results(query, reranked_results, max_passages=5):
    scored_passages = []

    for doc in reranked_results:
        passages = split_into_passages(doc["full_text"])

        for p in passages:
            overlap = lexical_overlap_score(query, p)

            # 문서 rank / rerank score 반영
            doc_bonus = 0.0
            if doc["rank"] == 1:
                doc_bonus += 0.20
            elif doc["rank"] == 2:
                doc_bonus += 0.10

            score = overlap + doc_bonus

            scored_passages.append({
                "doc_id": doc["doc_id"],
                "disease_kor": doc["disease_kor"],
                "intention": doc["intention"],
                "passage": p,
                "score": score,
                "rerank_score": doc["rerank_score"],
            })

    scored_passages = sorted(scored_passages, key=lambda x: x["score"], reverse=True)
    selected = scored_passages[:max_passages]

    blocks = []
    for i, sp in enumerate(selected, start=1):
        block = (
            f"[압축문맥 {i}]\n"
            f"doc_id: {sp['doc_id']}\n"
            f"질환명: {sp['disease_kor']}\n"
            f"상담주제: {sp['intention']}\n"
            f"내용:\n{sp['passage']}"
        )
        blocks.append(block)

    compressed_context = "\n\n" + ("=" * 70 + "\n\n").join(blocks)

    return {
        "compressed_context": compressed_context,
        "selected_passages": selected,
    }

print("[INFO] compress_context_from_reranked_results 함수 준비 완료")

[INFO] compress_context_from_reranked_results 함수 준비 완료


In [74]:
# ============================================================
# [STEP 6-3/6-4 - CELL 7] reranker + context compression 생성 함수
# ============================================================

def generate_answer_reranked_rag(query, initial_top_n=8, final_top_k=3, max_passages=5, temperature=0.0):
    reranked_results = rerank_hybrid_candidates(
        query=query,
        initial_top_n=initial_top_n,
        final_top_k=final_top_k
    )

    compressed = compress_context_from_reranked_results(
        query=query,
        reranked_results=reranked_results,
        max_passages=max_passages
    )

    context = compressed["compressed_context"]
    user_prompt = build_user_prompt_naive_rag(query, context)

    response = client.chat.completions.create(
        model=GEN_MODEL_NAME,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_NAIVE_RAG},
            {"role": "user", "content": user_prompt},
        ]
    )

    answer = response.choices[0].message.content.strip()

    return {
        "question": query,
        "answer": answer,
        "retrieval_results": reranked_results,
        "compressed_context": context,
        "selected_passages": compressed["selected_passages"],
        "model_name": GEN_MODEL_NAME,
    }

print("[INFO] generate_answer_reranked_rag 함수 준비 완료")

[INFO] generate_answer_reranked_rag 함수 준비 완료


In [76]:
# ============================================================
# [STEP 6-3/6-4 - CELL 8] 단건 테스트
# ============================================================

test_query = "가와사키병 치료에 사용되는 약물은 무엇인가요?"

rerank_test_result = generate_answer_reranked_rag(
    query=test_query,
    initial_top_n=8,
    final_top_k=3,
    max_passages=5,
    temperature=0.0
)

print("[질문]")
print(test_query)

print("\n[답변]")
print(rerank_test_result["answer"])

print("\n[reranked retrieval]")
for r in rerank_test_result["retrieval_results"]:
    print(
        f"- rank={r['rank']} | doc_id={r['doc_id']} | rerank={r['rerank_score']:.4f} | hybrid={r['score']:.4f}"
    )

print("\n[selected passages]")
for sp in rerank_test_result["selected_passages"][:3]:
    print("-" * 80)
    print(f"doc_id={sp['doc_id']} | score={sp['score']:.4f}")
    print(textwrap.shorten(sp["passage"], width=300, placeholder=" ..."))

[질문]
가와사키병 치료에 사용되는 약물은 무엇인가요?

[답변]
가와사키병의 치료에는 여러 가지 약물이 사용됩니다. 주로 저용량 아스피린과 면역글로불린이 사용되며, 이들은 관상동맥 합병증을 예방하는 데 효과적입니다. 또한, 증상이 심한 경우에는 항응고제나 스테로이드제도 고려될 수 있습니다. 발열이 있는 경우에는 아세트아미노펜과 같은 해열제를 사용할 수 있습니다. 치료는 환자의 증상과 상태에 따라 달라질 수 있으므로, 반드시 전문의의 지시에 따라 진행하는 것이 중요합니다. 증상이 심하거나 우려되는 경우에는 소아청소년과 진료를 받는 것이 좋습니다.

[reranked retrieval]
- rank=1 | doc_id=가와사키병__약물 | rerank=5.7978 | hybrid=0.9800
- rank=2 | doc_id=가와사키병__치료 | rerank=3.8835 | hybrid=0.7932
- rank=3 | doc_id=가와사키병__진단 | rerank=-0.5297 | hybrid=0.5830

[selected passages]
--------------------------------------------------------------------------------
doc_id=가와사키병__약물 | score=1.2000
[질환분류] 소아청소년질환 [질환명] 가와사키병 [영문명] Kawasaki Disease [진료과] 소아청소년과 [상담주제] 약물 [대표 질문 예시] - 가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다. - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 약물을 처방받으면 좋을까요? - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 종류의 약물을 처방받으면 좋을까요? - 약물치료를 받을 때 주의해야 할 부작용이 있을까요? - 가와사키병의 약물치료가 효과적인가요? - 가와사키병 치료를 위해 어떤 ...
--------------------------------------------------------

In [78]:
# ============================================================
# [STEP 6-3/6-4 - CELL 8] 단건 테스트
# ============================================================

test_query = "가와사키병 치료에 사용되는 약물은 무엇인가요?"

rerank_test_result = generate_answer_reranked_rag(
    query=test_query,
    initial_top_n=8,
    final_top_k=3,
    max_passages=5,
    temperature=0.0
)

print("[질문]")
print(test_query)

print("\n[답변]")
print(rerank_test_result["answer"])

print("\n[reranked retrieval]")
for r in rerank_test_result["retrieval_results"]:
    print(
        f"- rank={r['rank']} | doc_id={r['doc_id']} | rerank={r['rerank_score']:.4f} | hybrid={r['score']:.4f}"
    )

print("\n[selected passages]")
for sp in rerank_test_result["selected_passages"][:3]:
    print("-" * 80)
    print(f"doc_id={sp['doc_id']} | score={sp['score']:.4f}")
    print(textwrap.shorten(sp["passage"], width=300, placeholder=" ..."))

[질문]
가와사키병 치료에 사용되는 약물은 무엇인가요?

[답변]
가와사키병의 치료에는 여러 가지 약물이 사용됩니다. 주로 저용량 아스피린과 면역글로불린이 사용되며, 이들은 관상동맥 합병증을 예방하는 데 효과적입니다. 또한, 증상이 심한 경우에는 항응고제나 스테로이드, 항생제 등의 추가적인 약물이 필요할 수 있습니다. 발열이 있는 경우에는 아세트아미노펜과 같은 해열제를 사용하여 체온을 조절할 수 있습니다. 치료는 환자의 상태에 따라 달라질 수 있으므로, 반드시 전문의의 지시에 따라 진행하는 것이 중요합니다. 증상이 심하거나 우려되는 경우에는 소아청소년과 진료를 받는 것이 좋습니다.

[reranked retrieval]
- rank=1 | doc_id=가와사키병__약물 | rerank=5.7978 | hybrid=0.9800
- rank=2 | doc_id=가와사키병__치료 | rerank=3.8835 | hybrid=0.7932
- rank=3 | doc_id=가와사키병__진단 | rerank=-0.5297 | hybrid=0.5830

[selected passages]
--------------------------------------------------------------------------------
doc_id=가와사키병__약물 | score=1.2000
[질환분류] 소아청소년질환 [질환명] 가와사키병 [영문명] Kawasaki Disease [진료과] 소아청소년과 [상담주제] 약물 [대표 질문 예시] - 가와사키병 진단 후 어떤 종류의 약물을 복용해야 하는지 알고 싶습니다. - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 약물을 처방받으면 좋을까요? - 가와사키병 환자의 발열과 결막염이 있을 때, 어떤 종류의 약물을 처방받으면 좋을까요? - 약물치료를 받을 때 주의해야 할 부작용이 있을까요? - 가와사키병의 약물치료가 효과적인가요? - 가와사키병 치료를 위해 어떤 ...
------------------------------------

In [80]:
# ============================================================
# [STEP 6-3/6-4 - CELL 9] debug sample 테스트
# ============================================================

N_DEBUG = 10
debug_rows = eval_dataset_df.head(N_DEBUG)

rerank_debug_rows = []

for _, row in debug_rows.iterrows():
    result = generate_answer_reranked_rag(
        query=row["question"],
        initial_top_n=8,
        final_top_k=3,
        max_passages=5,
        temperature=0.0
    )

    rerank_debug_rows.append({
        "eval_id": row["eval_id"],
        "disease_kor": row["disease_kor"],
        "intention": row["intention"],
        "question": row["question"],
        "ground_truth_doc_id": row["ground_truth_doc_id"],
        "pred_answer": result["answer"],
        "retrieved_doc_ids": [x["doc_id"] for x in result["retrieval_results"]],
        "rerank_scores": [x["rerank_score"] for x in result["retrieval_results"]],
    })

rerank_debug_df = pd.DataFrame(rerank_debug_rows)

print("[INFO] rerank_debug_df shape:", rerank_debug_df.shape)
display(rerank_debug_df)

[INFO] rerank_debug_df shape: (10, 8)


,eval_id,disease_kor,intention,question,ground_truth_doc_id,pred_answer,retrieved_doc_ids,rerank_scores
0,EVAL_0001,가와사키병,약물,가와사키병의 약물치료가 효과적인가요?,가와사키병__약물,가와사키병의 약물치료는 증상 완화와 합병증 예방에 효과적입니다. 주로 사용되는 약물...,"[가와사키병__약물, 가와사키병__치료, 가와사키병__예방]","[6.644120693206787, 4.863039493560791, 2.60981..."
1,EVAL_0002,가와사키병,약물,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,가와사키병__약물,가와사키병의 약물 치료가 임신 또는 수유 중인 여성에게 미치는 영향에 대한 구체적인...,"[가와사키병__약물, 가와사키병__원인, 가와사키병__예방]","[-0.13000823557376862, -1.481030821800232, -1...."
2,EVAL_0003,가와사키병,예방,가와사키병에 대한 예방법을 알고 싶어요.,가와사키병__예방,가와사키병은 현재로서는 예방 방법이 개발되지 않았습니다. 하지만 개인 위생 관리가 ...,"[가와사키병__예방, 가와사키병__약물, 가와사키병__치료]","[8.00125789642334, 2.4571104049682617, 2.25891..."
3,EVAL_0004,가와사키병,예방,가와사키병의 예방에 있어서 유전적 요인이 어떤 역할을 하는지 알려주세요. 예방에 효...,가와사키병__예방,가와사키병의 예방에 있어 유전적 요인은 중요한 역할을 할 수 있습니다. 이 질환은 ...,"[가와사키병__원인, 가와사키병__예방, 가와사키병__치료]","[4.324207782745361, 3.7527904510498047, -0.003..."
4,EVAL_0005,가와사키병,원인,가와사키병의 원인은 무엇인가요?,가와사키병__원인,"가와사키병의 원인은 아직 명확하게 밝혀지지 않았습니다. 현재까지의 연구에 따르면, ...","[가와사키병__원인, 가와사키병__정의, 가와사키병__증상]","[5.090737342834473, 3.610025644302368, 3.29750..."
5,EVAL_0006,가와사키병,원인,가와사키병이 유전적 요인과 관련이 있다는 말을 듣고 있습니다. 저희 부부도 가와사키...,가와사키병__원인,"가와사키병의 원인은 아직 명확히 밝혀지지 않았지만, 유전적 요인과 환경적 요인이 관...","[가와사키병__원인, 영아_비대성_유문협착증__원인, 저신장증__원인]","[9.015971183776855, 2.4214277267456055, 1.7766..."
6,EVAL_0007,가와사키병,정의,가와사키병이란 무엇인가요?,가와사키병__정의,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...,"[가와사키병__증상, 가와사키병__정의, 가와사키병__치료]","[5.929617404937744, 3.963275194168091, 3.48692..."
7,EVAL_0008,가와사키병,정의,가와사키병이 어떤 의미를 가지는 질병인지 알고 싶어요. 가와사키병에 대한 정확한 정...,가와사키병__정의,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질병...,"[가와사키병__증상, 가와사키병__정의, 가와사키병__진단]","[5.069990634918213, 4.133909702301025, 3.51937..."
8,EVAL_0009,가와사키병,증상,가와사키병이란 어떤 질병인가요?,가와사키병__증상,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질병...,"[가와사키병__증상, 가와사키병__정의, 가와사키병__치료]","[7.181919574737549, 5.388294696807861, 4.88189..."
9,EVAL_0010,가와사키병,증상,가와사키병에 대해 자세히 알고 싶어요. 어린 아이의 열과 피부와 얼굴이 붉어지는 것...,가와사키병__증상,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...,"[가와사키병__증상, 가와사키병__정의, 가와사키병__예방]","[3.496079206466675, 0.777578592300415, -0.9754..."


In [83]:
# ============================================================
# [STEP 6-3/6-4 - CELL 10] reranked RAG inference 함수
# ============================================================

def run_reranked_rag_inference(eval_df, initial_top_n=8, final_top_k=3, max_passages=5, temperature=0.0, limit=None):
    rows = eval_df.copy()

    if limit is not None:
        rows = rows.head(limit).copy()

    output_rows = []

    for i, (_, row) in enumerate(rows.iterrows(), start=1):
        result = generate_answer_reranked_rag(
            query=row["question"],
            initial_top_n=initial_top_n,
            final_top_k=final_top_k,
            max_passages=max_passages,
            temperature=temperature
        )

        output_rows.append({
            "eval_id": row["eval_id"],
            "disease_category": row["disease_category"],
            "disease_kor": row["disease_kor"],
            "disease_eng": row["disease_eng"],
            "intention": row["intention"],
            "question_type": row["question_type"],
            "question": row["question"],
            "ground_truth_doc_id": row["ground_truth_doc_id"],
            "reference_answer": row["reference_answer"],
            "pred_answer": result["answer"],
            "retrieved_doc_ids": [x["doc_id"] for x in result["retrieval_results"]],
            "rerank_scores": [x["rerank_score"] for x in result["retrieval_results"]],
            "top_k": final_top_k,
            "model_name": GEN_MODEL_NAME,
        })

        if i % 20 == 0:
            print(f"[INFO] {i}/{len(rows)} 완료")

    return pd.DataFrame(output_rows)

print("[INFO] run_reranked_rag_inference 함수 준비 완료")

[INFO] run_reranked_rag_inference 함수 준비 완료


In [85]:
# ============================================================
# [STEP 6-3/6-4 - CELL 11] reranked RAG 실행
# ============================================================

RERANK_INITIAL_TOP_N = 8
RERANK_FINAL_TOP_K = 3
RERANK_MAX_PASSAGES = 5
RERANK_LIMIT = 30

reranked_rag_results_df = run_reranked_rag_inference(
    eval_df=eval_dataset_df,
    initial_top_n=RERANK_INITIAL_TOP_N,
    final_top_k=RERANK_FINAL_TOP_K,
    max_passages=RERANK_MAX_PASSAGES,
    temperature=0.0,
    limit=RERANK_LIMIT
)

print("[INFO] reranked_rag_results_df shape:", reranked_rag_results_df.shape)
display(reranked_rag_results_df.head(10))

[INFO] 20/30 완료
[INFO] reranked_rag_results_df shape: (30, 14)


,eval_id,disease_category,disease_kor,disease_eng,intention,question_type,question,ground_truth_doc_id,reference_answer,pred_answer,retrieved_doc_ids,rerank_scores,top_k,model_name
0,EVAL_0001,소아청소년질환,가와사키병,Kawasaki Disease,약물,medication,가와사키병의 약물치료가 효과적인가요?,가와사키병__약물,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및...",가와사키병의 약물 치료는 효과적일 수 있습니다. 주로 사용되는 약물로는 면역글로불린...,"[가와사키병__약물, 가와사키병__치료, 가와사키병__예방]","[6.644120693206787, 4.863039493560791, 2.60981...",3,gpt-4o-mini
1,EVAL_0002,소아청소년질환,가와사키병,Kawasaki Disease,약물,medication,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,가와사키병__약물,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및...",가와사키병의 약물 치료가 임신 또는 수유 중인 여성에게 미치는 영향에 대한 구체적인...,"[가와사키병__약물, 가와사키병__원인, 가와사키병__예방]","[-0.13000823557376862, -1.481030821800232, -1....",3,gpt-4o-mini
2,EVAL_0003,소아청소년질환,가와사키병,Kawasaki Disease,예방,prevention,가와사키병에 대한 예방법을 알고 싶어요.,가와사키병__예방,"가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 개...",가와사키병은 현재 예방 방법이 개발되지 않은 질환입니다. 하지만 개인 위생 관리를 ...,"[가와사키병__예방, 가와사키병__약물, 가와사키병__치료]","[8.00125789642334, 2.4571104049682617, 2.25891...",3,gpt-4o-mini
3,EVAL_0004,소아청소년질환,가와사키병,Kawasaki Disease,예방,prevention,가와사키병의 예방에 있어서 유전적 요인이 어떤 역할을 하는지 알려주세요. 예방에 효...,가와사키병__예방,"가와사키병은 주로 어린이에게서 발생하는 급성 열성 혈관염으로, 예방 방법은 아직 개...",가와사키병의 예방에 있어 유전적 요인은 중요한 역할을 할 수 있습니다. 연구에 따르...,"[가와사키병__원인, 가와사키병__예방, 가와사키병__치료]","[4.324207782745361, 3.7527904510498047, -0.003...",3,gpt-4o-mini
4,EVAL_0005,소아청소년질환,가와사키병,Kawasaki Disease,원인,cause,가와사키병의 원인은 무엇인가요?,가와사키병__원인,"가와사키병은 몇 가지 가설과 연구 결과가 있지만, 아직까지 확실히 입증된 것은 없습...",가와사키병의 원인은 아직 명확히 밝혀지지 않았습니다. 그러나 유전적 요인과 환경적 ...,"[가와사키병__원인, 가와사키병__정의, 가와사키병__증상]","[5.090737342834473, 3.610025644302368, 3.29750...",3,gpt-4o-mini
5,EVAL_0006,소아청소년질환,가와사키병,Kawasaki Disease,원인,cause,가와사키병이 유전적 요인과 관련이 있다는 말을 듣고 있습니다. 저희 부부도 가와사키...,가와사키병__원인,"가와사키병은 몇 가지 가설과 연구 결과가 있지만, 아직까지 확실히 입증된 것은 없습...","가와사키병의 원인은 아직 명확히 밝혀지지 않았지만, 유전적 요인과 환경적 요인이 관...","[가와사키병__원인, 영아_비대성_유문협착증__원인, 저신장증__원인]","[9.015971183776855, 2.4214277267456055, 1.7766...",3,gpt-4o-mini
6,EVAL_0007,소아청소년질환,가와사키병,Kawasaki Disease,정의,definition,가와사키병이란 무엇인가요?,가와사키병__정의,가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 주...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...,"[가와사키병__증상, 가와사키병__정의, 가와사키병__치료]","[5.929617404937744, 3.963275194168091, 3.48692...",3,gpt-4o-mini
7,EVAL_0008,소아청소년질환,가와사키병,Kawasaki Disease,정의,definition,가와사키병이 어떤 의미를 가지는 질병인지 알고 싶어요. 가와사키병에 대한 정확한 정...,가와사키병__정의,가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 주...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질병...,"[가와사키병__증상, 가와사키병__정의, 가와사키병__진단]","[5.069990634918213, 4.133909702301025, 3.51937...",3,gpt-4o-mini
8,EVAL_0009,소아청소년질환,가와사키병,Kawasaki Disease,증상,symptom,가와사키병이란 어떤 질병인가요?,가와사키병__증상,"가와사키병은 5일 이상 지속되는 고열, 손과 발 부종, 다양한 형태의 피부 발진 등...",가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질병...,"[가와사키병__증상, 가와사키병__정의, 가와사키병__치료]","[7.181919574737549, 5.388294696807861, 4.88189...",3,gpt-4o-mini
9,EVAL_0010,소아청소년질환,가와사키병,Kawasaki Disease,증상,symptom,가와사키병에 대해 자세히 알고 싶어요. 어린 아이의 열과 피부와 얼굴이 붉어지는 것...,가와사키병__증상,"가와사키병은 5일 이상 지속되는 고열, 손과 발 부종, 다양한 형태의 피부 발진 등...",가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...,"[가와사키병__증상, 가와사키병__정의, 가와사키병__예방]","[3.496079206466675, 0.777578592300415, -0.9754...",3,gpt-4o-mini


In [87]:
# ============================================================
# [STEP 6-3/6-4 - CELL 12] Hit@K 비교
# ============================================================

def check_hit(row):
    gt = row["ground_truth_doc_id"]
    retrieved = row["retrieved_doc_ids"]
    return gt in retrieved

reranked_rag_results_df["hit_at_k"] = reranked_rag_results_df.apply(check_hit, axis=1)
reranked_hit_rate = reranked_rag_results_df["hit_at_k"].mean()

print(f"[INFO] Reranked Hit@{RERANK_FINAL_TOP_K}: {reranked_hit_rate:.4f}")

compare_rows = []

if "naive_rag_results_df" in globals():
    naive_tmp = naive_rag_results_df.copy()
    if "hit_at_k" not in naive_tmp.columns:
        naive_tmp["hit_at_k"] = naive_tmp.apply(check_hit, axis=1)
    compare_rows.append({
        "system": "naive_dense_only",
        "hit_at_k": float(naive_tmp["hit_at_k"].mean()),
        "n": len(naive_tmp)
    })

if "hybrid_rag_results_df" in globals():
    hybrid_tmp = hybrid_rag_results_df.copy()
    if "hit_at_k" not in hybrid_tmp.columns:
        hybrid_tmp["hit_at_k"] = hybrid_tmp.apply(check_hit, axis=1)
    compare_rows.append({
        "system": "hybrid_dense_bm25_meta",
        "hit_at_k": float(hybrid_tmp["hit_at_k"].mean()),
        "n": len(hybrid_tmp)
    })

compare_rows.append({
    "system": "reranked_hybrid_compressed",
    "hit_at_k": float(reranked_hit_rate),
    "n": len(reranked_rag_results_df)
})

compare_summary_df = pd.DataFrame(compare_rows)
display(compare_summary_df)

[INFO] Reranked Hit@3: 0.9000


,system,hit_at_k,n
0,naive_dense_only,0.466667,30
1,hybrid_dense_bm25_meta,0.833333,30
2,reranked_hybrid_compressed,0.900000,30


In [89]:
# ============================================================
# [STEP 6-3/6-4 - CELL 13] 결과 저장
# ============================================================

RERANK_RESULTS_JSONL_PATH = OUTPUT_DIR / "reranked_rag_results_v1.jsonl"
RERANK_RESULTS_CSV_PATH   = OUTPUT_DIR / "reranked_rag_results_v1.csv"

rerank_result_records = reranked_rag_results_df.to_dict(orient="records")
save_jsonl(rerank_result_records, RERANK_RESULTS_JSONL_PATH)
reranked_rag_results_df.to_csv(RERANK_RESULTS_CSV_PATH, index=False, encoding="utf-8-sig")

print("[INFO] 저장 완료")
print(" -", RERANK_RESULTS_JSONL_PATH)
print(" -", RERANK_RESULTS_CSV_PATH)

[INFO] 저장 완료
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\outputs\reranked_rag_results_v1.jsonl
 - D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\outputs\reranked_rag_results_v1.csv


In [91]:
# ============================================================
# [STEP 6-3/6-4 - CELL 14] STEP 6 요약
# ============================================================

step6_rerank_summary = {
    "reranker_model_name": RERANKER_MODEL_NAME,
    "initial_top_n": RERANK_INITIAL_TOP_N,
    "final_top_k": RERANK_FINAL_TOP_K,
    "max_passages": RERANK_MAX_PASSAGES,
    "num_results": int(len(reranked_rag_results_df)),
    "reranked_hit_at_k": float(reranked_hit_rate),
    "results_jsonl_path": str(RERANK_RESULTS_JSONL_PATH),
}

print(json.dumps(step6_rerank_summary, ensure_ascii=False, indent=2))

{
  "reranker_model_name": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
  "initial_top_n": 8,
  "final_top_k": 3,
  "max_passages": 5,
  "num_results": 30,
  "reranked_hit_at_k": 0.9,
  "results_jsonl_path": "D:\\PyProject\\AIFFEL_AI\\LLM\\NLP\\RAG_proj\\outputs\\reranked_rag_results_v1.jsonl"
}


### STEP 7. Prompt 최적화
- 같은 retrieval이어도 prompt가 다르면 G-EVAL과 RAGAS 점수가 달라짐
#### Prompt A. 단순 grounded prompt: 
- 검색문서 기준으로만 답하도록 제한
#### Prompt B. 의료 상담형 prompt
- 보호자에게 쉽게 설명
- 위험 증상 시 병원 진료 권고
- 근거 밖 추정 금지

In [94]:
# ============================================================
# [STEP 7 - CELL 1] Prompt A / B 정의
# ============================================================

PROMPT_GROUNDED_STRICT = """
당신은 의료 QA 시스템입니다.

반드시 아래 규칙을 지켜 답변하세요:

[규칙]
1. 제공된 검색 문서(context)에 있는 내용만 사용하세요.
2. 문서에 없는 내용은 절대 추측하지 마세요.
3. 모르면 "제공된 정보로는 답변할 수 없습니다."라고 답하세요.
4. 불필요한 설명 없이 정확하게 답하세요.

[질문]
{question}

[검색 문서]
{context}

[답변]
"""

PROMPT_MEDICAL_CONSULT = """
당신은 소아과 상담을 돕는 AI입니다.

반드시 아래 규칙을 지켜 답변하세요:

[역할]
- 보호자가 이해하기 쉽게 설명하세요.
- 너무 전문적인 용어는 쉽게 풀어 설명하세요.

[안전 규칙]
1. 제공된 검색 문서(context)에 기반해서만 답하세요.
2. 근거 없는 추측은 하지 마세요.
3. 위험할 수 있는 증상은 병원 방문을 권유하세요.
4. 불확실하면 명확히 "정확한 판단은 의료진 상담이 필요합니다"라고 말하세요.

[스타일]
- 친절하고 차분하게 설명
- 3~6문장 정도
- 핵심 위주

[질문]
{question}

[검색 문서]
{context}

[답변]
"""

print("[INFO] Prompt A/B 준비 완료")

[INFO] Prompt A/B 준비 완료


In [96]:
# ============================================================
# [STEP 7 - CELL 2] prompt 선택형 생성 함수
# ============================================================

def generate_answer_with_prompt(query, retrieval_results, prompt_template, temperature=0.0):
    context = build_context_from_retrieval_results(retrieval_results)

    prompt = prompt_template.format(
        question=query,
        context=context
    )

    response = client.chat.completions.create(
        model=GEN_MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )

    return response.choices[0].message.content.strip()

In [102]:
# ============================================================
# [STEP 7 - CELL 3 - FIX] reranked + prompt 적용 inference
# ------------------------------------------------------------
# 현재 노트북에서 이미 정의된 rerank_hybrid_candidates()를 사용하도록 수정
# ============================================================

def run_rag_with_prompt(eval_df, prompt_template, limit=30, initial_top_n=8, final_top_k=3):
    rows = eval_df.head(limit).copy()

    outputs = []

    for i, (_, row) in enumerate(rows.iterrows(), start=1):

        # hybrid retrieval + reranker
        retrieval_results = rerank_hybrid_candidates(
            query=row["question"],
            initial_top_n=initial_top_n,
            final_top_k=final_top_k
        )

        answer = generate_answer_with_prompt(
            query=row["question"],
            retrieval_results=retrieval_results,
            prompt_template=prompt_template
        )

        outputs.append({
            "eval_id": row["eval_id"],
            "question": row["question"],
            "ground_truth_doc_id": row["ground_truth_doc_id"],
            "pred_answer": answer,
            "retrieved_doc_ids": [x["doc_id"] for x in retrieval_results],
            "rerank_scores": [x["rerank_score"] for x in retrieval_results],
        })

        if i % 10 == 0:
            print(f"[INFO] {i}/{len(rows)} 완료")

    return pd.DataFrame(outputs)

print("[INFO] run_rag_with_prompt 수정 완료")

[INFO] run_rag_with_prompt 수정 완료


In [104]:
# ============================================================
# [STEP 7 - CELL 4] Prompt A vs B 실행
# ============================================================

results_prompt_A = run_rag_with_prompt(
    eval_dataset_df,
    PROMPT_GROUNDED_STRICT,
    limit=30
)

results_prompt_B = run_rag_with_prompt(
    eval_dataset_df,
    PROMPT_MEDICAL_CONSULT,
    limit=30
)

print("[INFO] Prompt A/B 실행 완료")

[INFO] 10/30 완료
[INFO] 20/30 완료
[INFO] 30/30 완료
[INFO] 10/30 완료
[INFO] 20/30 완료
[INFO] 30/30 완료
[INFO] Prompt A/B 실행 완료


In [106]:
# ============================================================
# [STEP 7 - CELL 5] 결과 비교
# ============================================================

compare_df = pd.DataFrame({
    "question": results_prompt_A["question"],
    "A_answer": results_prompt_A["pred_answer"],
    "B_answer": results_prompt_B["pred_answer"]
})

display(compare_df.head(10))

,question,A_answer,B_answer
0,가와사키병의 약물치료가 효과적인가요?,가와사키병의 약물치료는 효과적입니다. 치료에는 주로 면역글로불린과 아스피린이 사용되...,가와사키병의 약물치료는 효과적일 수 있습니다. 주로 사용되는 약물은 면역글로불린과 ...
1,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,제공된 정보로는 답변할 수 없습니다.,"가와사키병의 약물 치료는 주로 소아에게 이루어지며, 임신 중이나 수유 중인 여성에게..."
2,가와사키병에 대한 예방법을 알고 싶어요.,가와사키병을 예방하기 위한 방법은 다음과 같습니다:\n\n1. 개인 위생 관리에 신...,가와사키병은 현재로서는 예방할 수 있는 방법이 개발되지 않았습니다. 하지만 개인 위...
3,가와사키병의 예방에 있어서 유전적 요인이 어떤 역할을 하는지 알려주세요. 예방에 효...,제공된 정보로는 답변할 수 없습니다.,가와사키병은 유전적 요인과 환경적 요인이 복합적으로 작용하여 발생할 수 있는 질환입...
4,가와사키병의 원인은 무엇인가요?,"가와사키병의 원인은 아직까지 밝혀지지 않았지만, 유전적 요인과 환경적 요인이 관여할...","가와사키병의 원인은 아직 명확하게 밝혀지지 않았습니다. 여러 연구에 따르면, 유전적..."
5,가와사키병이 유전적 요인과 관련이 있다는 말을 듣고 있습니다. 저희 부부도 가와사키...,제공된 정보로는 답변할 수 없습니다.,"가와사키병의 원인은 아직 완전히 밝혀지지 않았지만, 유전적 요인과 환경적 요인이 함..."
6,가와사키병이란 무엇인가요?,가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 주...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...
7,가와사키병이 어떤 의미를 가지는 질병인지 알고 싶어요. 가와사키병에 대한 정확한 정...,가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 주...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...
8,가와사키병이란 어떤 질병인가요?,가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 주...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...
9,가와사키병에 대해 자세히 알고 싶어요. 어린 아이의 열과 피부와 얼굴이 붉어지는 것...,가와사키병의 주요 증상에는 다음과 같은 것들이 있습니다:\n\n1. 5일 이상 지속...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 병의...


### B를 기반으로 A의 “모르면 멈추기” 규칙을 추가한 Prompt C가 최선

In [109]:
# ============================================================
# [STEP 7 - CELL 6] Prompt C 정의
# ------------------------------------------------------------
# Prompt B(의료 상담형)를 기반으로 하되,
# Prompt A의 grounded strict 규칙을 일부 결합한 안전형 프롬프트
# ============================================================

PROMPT_MEDICAL_SAFE = """
당신은 소아과 상담을 돕는 AI입니다.

반드시 아래 규칙을 지켜 답변하세요.

[역할]
- 보호자가 이해하기 쉽게 설명하세요.
- 너무 어려운 의학 용어는 쉽게 풀어 설명하세요.

[핵심 규칙]
1. 반드시 제공된 검색 문서(context)에 근거해서만 답하세요.
2. 검색 문서에 없는 내용을 추측해서 단정하지 마세요.
3. 질문에 일부만 답할 수 있는 경우, 답할 수 있는 범위만 설명하세요.
4. 문서 근거가 부족하면 "제공된 정보만으로는 정확히 답변하기 어렵습니다"라고 분명히 말하세요.
5. 위험하거나 증상이 심할 수 있는 경우에는 소아청소년과 진료가 필요할 수 있다고 안내하세요.
6. 진단을 확정하는 표현은 피하세요.

[스타일]
- 한국어 존댓말
- 친절하고 차분한 설명
- 핵심 위주로 4~6문장
- 불필요하게 장황하지 않게 작성

[질문]
{question}

[검색 문서]
{context}

[답변]
"""
print("[INFO] Prompt C 준비 완료")

[INFO] Prompt C 준비 완료


In [111]:
results_prompt_C = run_rag_with_prompt(
    eval_dataset_df,
    PROMPT_MEDICAL_SAFE,
    limit=30
)

compare_df_abc = pd.DataFrame({
    "question": results_prompt_A["question"],
    "A_answer": results_prompt_A["pred_answer"],
    "B_answer": results_prompt_B["pred_answer"],
    "C_answer": results_prompt_C["pred_answer"]
})

display(compare_df_abc.head(10))

[INFO] 10/30 완료
[INFO] 20/30 완료
[INFO] 30/30 완료


,question,A_answer,B_answer,C_answer
0,가와사키병의 약물치료가 효과적인가요?,가와사키병의 약물치료는 효과적입니다. 치료에는 주로 면역글로불린과 아스피린이 사용되...,가와사키병의 약물치료는 효과적일 수 있습니다. 주로 사용되는 약물은 면역글로불린과 ...,가와사키병의 약물치료는 효과적일 수 있습니다. 주로 사용되는 약물은 면역글로불린과 ...
1,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,제공된 정보로는 답변할 수 없습니다.,"가와사키병의 약물 치료는 주로 소아에게 이루어지며, 임신 중이나 수유 중인 여성에게...",제공된 정보만으로는 가와사키병에 대한 약물 치료가 임신 또는 수유 중인 여성에게 미...
2,가와사키병에 대한 예방법을 알고 싶어요.,가와사키병을 예방하기 위한 방법은 다음과 같습니다:\n\n1. 개인 위생 관리에 신...,가와사키병은 현재로서는 예방할 수 있는 방법이 개발되지 않았습니다. 하지만 개인 위...,가와사키병은 현재로서는 예방할 수 있는 방법이 개발되지 않았습니다. 하지만 개인 위...
3,가와사키병의 예방에 있어서 유전적 요인이 어떤 역할을 하는지 알려주세요. 예방에 효...,제공된 정보로는 답변할 수 없습니다.,가와사키병은 유전적 요인과 환경적 요인이 복합적으로 작용하여 발생할 수 있는 질환입...,가와사키병의 예방에 있어 유전적 요인은 중요한 역할을 할 수 있습니다. 연구에 따르...
4,가와사키병의 원인은 무엇인가요?,"가와사키병의 원인은 아직까지 밝혀지지 않았지만, 유전적 요인과 환경적 요인이 관여할...","가와사키병의 원인은 아직 명확하게 밝혀지지 않았습니다. 여러 연구에 따르면, 유전적...","가와사키병의 원인은 아직 명확하게 밝혀지지 않았습니다. 여러 연구에 따르면, 유전적..."
5,가와사키병이 유전적 요인과 관련이 있다는 말을 듣고 있습니다. 저희 부부도 가와사키...,제공된 정보로는 답변할 수 없습니다.,"가와사키병의 원인은 아직 완전히 밝혀지지 않았지만, 유전적 요인과 환경적 요인이 함...","가와사키병의 원인은 아직 완전히 밝혀지지 않았지만, 유전적 요인과 환경적 요인이 함..."
6,가와사키병이란 무엇인가요?,가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 주...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...
7,가와사키병이 어떤 의미를 가지는 질병인지 알고 싶어요. 가와사키병에 대한 정확한 정...,가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 주...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...
8,가와사키병이란 어떤 질병인가요?,가와사키병은 영아와 어린이에서 주로 발생하는 급성 열성 혈관염입니다. 이 질병은 주...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...
9,가와사키병에 대해 자세히 알고 싶어요. 어린 아이의 열과 피부와 얼굴이 붉어지는 것...,가와사키병의 주요 증상에는 다음과 같은 것들이 있습니다:\n\n1. 5일 이상 지속...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 병의...,가와사키병은 주로 5세 이하의 어린이에게 발생하는 급성 열성 혈관염입니다. 이 질환...


### STEP 8. RAGAS 평가

In [130]:
# ============================================================
# [STEP 8 - RESTART CELL 1] 경로 설정
# ============================================================

from pathlib import Path
import pandas as pd
import json
import os

PROJECT_ROOT = Path(r"D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj")
OUTPUT_DIR = PROJECT_ROOT / "outputs"

RAGAS_SUMMARY_CSV_PATH = OUTPUT_DIR / "ragas_summary_v1.csv"
G_EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / "g_eval_summary_v1.csv"

print("[INFO] PROJECT_ROOT:", PROJECT_ROOT)
print("[INFO] OUTPUT_DIR:", OUTPUT_DIR)

[INFO] PROJECT_ROOT: D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj
[INFO] OUTPUT_DIR: D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\outputs


## ragas_g_eval.py 실행
#### 터미널/명령 프롬프트/Anaconda Prompt에서 실행
conda activate aiffel
cd /d D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj

python ragas_g_eval.py

In [133]:
# ============================================================
# [STEP 8 - RESTART CELL 2] RAGAS / G-EVAL 결과 불러오기
# ============================================================

if RAGAS_SUMMARY_CSV_PATH.exists():
    ragas_summary_df = pd.read_csv(RAGAS_SUMMARY_CSV_PATH, encoding="utf-8-sig")
    print("[INFO] ragas_summary_df shape:", ragas_summary_df.shape)
    display(ragas_summary_df)
else:
    print("[WARN] RAGAS summary 파일이 아직 없습니다:", RAGAS_SUMMARY_CSV_PATH)

if G_EVAL_SUMMARY_CSV_PATH.exists():
    g_eval_summary_df = pd.read_csv(G_EVAL_SUMMARY_CSV_PATH, encoding="utf-8-sig")
    print("[INFO] g_eval_summary_df shape:", g_eval_summary_df.shape)
    display(g_eval_summary_df)
else:
    print("[WARN] G-EVAL summary 파일이 아직 없습니다:", G_EVAL_SUMMARY_CSV_PATH)

[INFO] ragas_summary_df shape: (3, 7)


,system,answer_relevancy,faithfulness,context_recall,context_precision,answer_correctness,n
0,naive_dense_only,0.127219,0.824299,0.877896,0.933333,0.611506,30
1,hybrid_dense_bm25_meta,0.146153,0.992963,0.924929,1.000000,0.751039,30
2,reranked_hybrid_compressed,0.158321,0.959127,0.901869,0.983333,0.748195,30


[INFO] g_eval_summary_df shape: (3, 8)


,system,groundedness_mean,relevance_mean,completeness_mean,clarity_mean,safety_mean,overall_mean,n
0,naive_dense_only,4.066667,4.300000,3.933333,4.233333,4.833333,4.273333,30
1,hybrid_dense_bm25_meta,4.700000,4.866667,4.566667,4.766667,4.933333,4.766667,30
2,reranked_hybrid_compressed,4.666667,4.866667,4.533333,4.700000,4.933333,4.740000,30


####  answer_relevancy (아직 낮음)
### STEP 7 업그레이드 필요
- ❌ "잘 설명하는 AI" → ✅ "질문에 정확히 답하는 AI" 로 변경 필요

In [113]:
# ============================================================
# [STEP 8 - CELL 1] RAGAS / G-EVAL 기본 import
# ------------------------------------------------------------
# 필요 시 1회 설치:
# !pip install ragas langchain-openai datasets -q
# ============================================================

import os
import json
import math
import pandas as pd
import numpy as np

from pathlib import Path
from datasets import Dataset

import ragas
print("[INFO] ragas version:", ragas.__version__)

from ragas import evaluate
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision,
    answer_correctness,
)

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

print("[INFO] STEP 8 import 완료")

D:\anaconda3\envs\aiffel\Lib\site-packages\instructor\providers\gemini\client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]


[INFO] ragas version: 0.4.3
[INFO] STEP 8 import 완료


C:\Users\hugctx\AppData\Local\Temp\ipykernel_41576\1992438667.py:21: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\hugctx\AppData\Local\Temp\ipykernel_41576\1992438667.py:21: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\hugctx\AppData\Local\Temp\ipykernel_41576\1992438667.py:21: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
C:\Users\hugctx\AppData\Local\Temp\ipykernel_4

In [115]:
# ============================================================
# [STEP 8 - CELL 2] RAGAS evaluator 준비
# ------------------------------------------------------------
# 평가용 LLM / embeddings를 별도로 준비한다.
# ============================================================

from dotenv import load_dotenv

ENV_PATH = r"D:\PyProject\env_keys\.env"
load_dotenv(ENV_PATH)

openai_api_key = os.getenv("OPENAI_API_KEY")
assert openai_api_key is not None and len(openai_api_key) > 0, "OPENAI_API_KEY를 확인하세요."

RAGAS_EVAL_MODEL = "gpt-4o-mini"
RAGAS_EMBED_MODEL = "text-embedding-3-small"

evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model=RAGAS_EVAL_MODEL,
        temperature=0.0,
        api_key=openai_api_key,
    )
)

evaluator_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(
        model=RAGAS_EMBED_MODEL,
        api_key=openai_api_key,
    )
)

print("[INFO] evaluator_llm 준비 완료:", RAGAS_EVAL_MODEL)
print("[INFO] evaluator_embeddings 준비 완료:", RAGAS_EMBED_MODEL)

[INFO] evaluator_llm 준비 완료: gpt-4o-mini
[INFO] evaluator_embeddings 준비 완료: text-embedding-3-small


C:\Users\hugctx\AppData\Local\Temp\ipykernel_41576\3609462239.py:18: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(
C:\Users\hugctx\AppData\Local\Temp\ipykernel_41576\3609462239.py:26: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(


In [117]:
# ============================================================
# [STEP 8 - CELL 3] doc_id -> full_text lookup 준비
# ------------------------------------------------------------
# retrieved_doc_ids를 retrieved_contexts로 바꾸는 데 사용한다.
# ============================================================

if "rag_docs_v4_df" not in globals():
    RAG_DOCS_V4_JSONL_PATH = Path(r"D:\PyProject\AIFFEL_AI\LLM\NLP\RAG_proj\data\processed\rag_docs_v4.jsonl")
    rag_docs_v4 = load_jsonl(RAG_DOCS_V4_JSONL_PATH)
    rag_docs_v4_df = pd.DataFrame(rag_docs_v4)

doc_text_lookup = {
    row["doc_id"]: row["full_text"]
    for _, row in rag_docs_v4_df.iterrows()
}

print("[INFO] doc_text_lookup size:", len(doc_text_lookup))

[INFO] doc_text_lookup size: 139


In [119]:
# ============================================================
# [STEP 8 - CELL 4] 결과 DataFrame -> RAGAS dataset 변환 함수
# ------------------------------------------------------------
# 각 시스템 결과를 RAGAS가 기대하는 컬럼으로 변환한다.
# required columns:
# - user_input
# - response
# - reference
# - retrieved_contexts
# ============================================================

def build_ragas_input_df(results_df, system_name="system"):
    rows = []

    for _, row in results_df.iterrows():
        retrieved_doc_ids = row["retrieved_doc_ids"] if isinstance(row["retrieved_doc_ids"], list) else []
        retrieved_contexts = [doc_text_lookup[d] for d in retrieved_doc_ids if d in doc_text_lookup]

        rows.append({
            "eval_id": row["eval_id"],
            "system": system_name,
            "user_input": row["question"],
            "response": row["pred_answer"],
            "reference": row["reference_answer"],
            "retrieved_contexts": retrieved_contexts,
        })

    return pd.DataFrame(rows)

print("[INFO] build_ragas_input_df 함수 준비 완료")

[INFO] build_ragas_input_df 함수 준비 완료


In [121]:
# ============================================================
# [STEP 8 - CELL 5] 비교 대상 시스템 준비
# ------------------------------------------------------------
# 현재 메모리에 있는 결과 DataFrame만 모아 비교한다.
# ============================================================

system_frames = {}

if "naive_rag_results_df" in globals():
    system_frames["naive_dense_only"] = naive_rag_results_df.copy()

if "hybrid_rag_results_df" in globals():
    system_frames["hybrid_dense_bm25_meta"] = hybrid_rag_results_df.copy()

if "reranked_rag_results_df" in globals():
    system_frames["reranked_hybrid_compressed"] = reranked_rag_results_df.copy()

print("[INFO] 비교 대상 시스템:", list(system_frames.keys()))
for k, v in system_frames.items():
    print(f" - {k}: {v.shape}")

[INFO] 비교 대상 시스템: ['naive_dense_only', 'hybrid_dense_bm25_meta', 'reranked_hybrid_compressed']
 - naive_dense_only: (30, 15)
 - hybrid_dense_bm25_meta: (30, 15)
 - reranked_hybrid_compressed: (30, 15)


In [123]:
# ============================================================
# [STEP 8 - CELL 6] 시스템별 RAGAS dataset 생성
# ============================================================

ragas_input_map = {}

for system_name, df in system_frames.items():
    ragas_input_df = build_ragas_input_df(df, system_name=system_name)
    ragas_input_map[system_name] = ragas_input_df

    print(f"[INFO] {system_name} ragas_input_df shape:", ragas_input_df.shape)
    display(ragas_input_df.head(2))

[INFO] naive_dense_only ragas_input_df shape: (30, 6)


,eval_id,system,user_input,response,reference,retrieved_contexts
0,EVAL_0001,naive_dense_only,가와사키병의 약물치료가 효과적인가요?,가와사키병의 약물 치료는 주로 증상 완화와 관상동맥 합병증 예방을 목표로 합니다. ...,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및...",[[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki D...
1,EVAL_0002,naive_dense_only,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,"죄송하지만, 제공된 검색 문서에는 가와사키병에 대한 약물 치료가 임신 또는 수유 중...","가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및...",[[질환분류] 소아청소년질환\n[질환명] 과숙아\n[영문명] Post-term in...


[INFO] hybrid_dense_bm25_meta ragas_input_df shape: (30, 6)


,eval_id,system,user_input,response,reference,retrieved_contexts
0,EVAL_0001,hybrid_dense_bm25_meta,가와사키병의 약물치료가 효과적인가요?,가와사키병의 약물치료는 효과적일 수 있습니다. 주로 사용되는 약물로는 면역글로불린과...,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및...",[[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki D...
1,EVAL_0002,hybrid_dense_bm25_meta,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,가와사키병의 약물 치료가 임신 또는 수유 중인 여성에게 미치는 영향에 대한 구체적인...,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및...",[[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki D...


[INFO] reranked_hybrid_compressed ragas_input_df shape: (30, 6)


,eval_id,system,user_input,response,reference,retrieved_contexts
0,EVAL_0001,reranked_hybrid_compressed,가와사키병의 약물치료가 효과적인가요?,가와사키병의 약물 치료는 효과적일 수 있습니다. 주로 사용되는 약물로는 면역글로불린...,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및...",[[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki D...
1,EVAL_0002,reranked_hybrid_compressed,가와사키병에 대한 약물 치료로 인해 임신 또는 수유 중인 여성의 임신 중 또는 수유...,가와사키병의 약물 치료가 임신 또는 수유 중인 여성에게 미치는 영향에 대한 구체적인...,"가와사키병은 혈액 검사를 통해 진단할 수 있으며, 치료는 증상에 따라 약물 치료 및...",[[질환분류] 소아청소년질환\n[질환명] 가와사키병\n[영문명] Kawasaki D...


In [125]:
# ============================================================
# [STEP 8 - CELL 7] 단일 시스템 RAGAS 평가 함수
# ============================================================

RAGAS_METRICS = [
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision,
    answer_correctness,
]

def run_ragas_for_system(ragas_input_df, system_name="system"):
    hf_dataset = Dataset.from_pandas(ragas_input_df)

    result = evaluate(
        dataset=hf_dataset,
        metrics=RAGAS_METRICS,
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
    )

    result_df = result.to_pandas()
    result_df["system"] = system_name

    summary = {
        "system": system_name,
        "answer_relevancy": float(result_df["answer_relevancy"].mean()),
        "faithfulness": float(result_df["faithfulness"].mean()),
        "context_recall": float(result_df["context_recall"].mean()),
        "context_precision": float(result_df["context_precision"].mean()),
        "answer_correctness": float(result_df["answer_correctness"].mean()),
        "n": int(len(result_df)),
    }

    return result_df, summary

print("[INFO] run_ragas_for_system 함수 준비 완료")

[INFO] run_ragas_for_system 함수 준비 완료


In [127]:
# ============================================================
# [STEP 8 - CELL 8] 시스템별 RAGAS 실행
# ------------------------------------------------------------
# 시간이 걸릴 수 있다.
# ============================================================

ragas_result_dfs = {}
ragas_summaries = []

for system_name, ragas_input_df in ragas_input_map.items():
    print(f"\n[INFO] RAGAS 실행 시작: {system_name}")
    result_df, summary = run_ragas_for_system(ragas_input_df, system_name=system_name)
    ragas_result_dfs[system_name] = result_df
    ragas_summaries.append(summary)

ragas_summary_df = pd.DataFrame(ragas_summaries)
print("[INFO] ragas_summary_df shape:", ragas_summary_df.shape)
display(ragas_summary_df.sort_values("answer_correctness", ascending=False))


[INFO] RAGAS 실행 시작: naive_dense_only


Evaluating:   0%|          | 0/150 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "D:\anaconda3\envs\aiffel\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x00000233FC5FEC80> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "D:\anaconda3\envs\aiffel\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x00000233FC5FEC80> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "D:\anaconda3\envs\aiffel\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x00000233FC5FEC80> is already entere


[INFO] RAGAS 실행 시작: hybrid_dense_bm25_meta


Evaluating:   0%|          | 0/150 [00:00<?, ?it/s]

...


[INFO] RAGAS 실행 시작: reranked_hybrid_compressed


Evaluating:   0%|          | 0/150 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "D:\anaconda3\envs\aiffel\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x00000233FC5FEC80> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "D:\anaconda3\envs\aiffel\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x00000233FC5FEC80> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "D:\anaconda3\envs\aiffel\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x00000233FC5FEC80> is already entere

KeyboardInterrupt: 

Exception raised in Job[74]: AttributeError('NoneType' object has no attribute 'generate')
Exception raised in Job[86]: AssertionError(LLM is not set)
Exception raised in Job[83]: AttributeError('NoneType' object has no attribute 'generate')
Exception raised in Job[87]: AssertionError(set LLM before use)
Exception raised in Job[88]: AssertionError(LLM is not set)
Exception raised in Job[89]: AssertionError(LLM must be set)
Exception raised in Job[90]: AssertionError(LLM is not set)
Exception raised in Job[91]: AssertionError(LLM is not set)
Exception raised in Job[92]: AssertionError(set LLM before use)
Exception raised in Job[93]: AssertionError(LLM is not set)
Exception raised in Job[94]: AssertionError(LLM must be set)
Exception raised in Job[95]: AssertionError(LLM is not set)
Exception raised in Job[96]: AssertionError(LLM is not set)
Exception raised in Job[97]: AssertionError(set LLM before use)
Exception raised in Job[98]: AssertionError(LLM is not set)
Exception raised in Job[

In [ ]:
# ============================================================
# [STEP 8 - CELL 9] RAGAS 결과 저장
# ============================================================

RAGAS_SUMMARY_CSV_PATH = OUTPUT_DIR / "ragas_summary_v1.csv"
RAGAS_SUMMARY_JSON_PATH = OUTPUT_DIR / "ragas_summary_v1.json"

ragas_summary_df.to_csv(RAGAS_SUMMARY_CSV_PATH, index=False, encoding="utf-8-sig")

with open(RAGAS_SUMMARY_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(ragas_summaries, f, ensure_ascii=False, indent=2)

print("[INFO] 저장 완료")
print(" -", RAGAS_SUMMARY_CSV_PATH)
print(" -", RAGAS_SUMMARY_JSON_PATH)

### STEP 9. G-EVAL 평가

In [ ]:
# ============================================================
# [STEP 9 - CELL 1] G-EVAL judge prompt 정의
# ------------------------------------------------------------
# 항목:
# - groundedness
# - relevance
# - completeness
# - clarity
# - safety
# 각 항목 1~5점
# ============================================================

G_EVAL_JUDGE_PROMPT = """
당신은 소아과 상담형 RAG 시스템의 답변 품질을 평가하는 엄격한 심사자입니다.

아래 항목을 각각 1점~5점으로 평가하세요.
반드시 JSON으로만 답하세요.

평가 항목:
1. groundedness: 답변이 제공된 검색 문서(context)에 잘 근거하고 있는가
2. relevance: 질문에 맞게 답했는가
3. completeness: 핵심 내용을 충분히 담았는가
4. clarity: 보호자가 이해하기 쉽게 설명했는가
5. safety: 근거 밖 추정이나 위험한 단정을 피했는가

[질문]
{question}

[검색 문서]
{context}

[답변]
{answer}

출력 형식(JSON only):
{{
  "groundedness": 1-5,
  "relevance": 1-5,
  "completeness": 1-5,
  "clarity": 1-5,
  "safety": 1-5
}}
""".strip()

print("[INFO] G-EVAL judge prompt 준비 완료")

In [ ]:
# ============================================================
# [STEP 9 - CELL 2] G-EVAL judge 함수
# ============================================================

def safe_json_parse(text):
    text = text.strip()

    # 코드블록 제거
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    return json.loads(text)

def judge_answer_with_g_eval(question, context, answer, judge_model="gpt-4o-mini"):
    prompt = G_EVAL_JUDGE_PROMPT.format(
        question=question,
        context=context,
        answer=answer
    )

    response = client.chat.completions.create(
        model=judge_model,
        temperature=0.0,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    raw_text = response.choices[0].message.content.strip()
    parsed = safe_json_parse(raw_text)
    return parsed

print("[INFO] judge_answer_with_g_eval 함수 준비 완료")

In [ ]:
# ============================================================
# [STEP 9 - CELL 3] G-EVAL용 context 복원 함수
# ------------------------------------------------------------
# retrieved_doc_ids를 다시 문서 텍스트로 복원하여 judge에 넣는다.
# ============================================================

def build_context_from_doc_ids(doc_ids):
    contexts = []
    for d in doc_ids:
        if d in doc_text_lookup:
            contexts.append(f"[doc_id: {d}]\n{doc_text_lookup[d]}")
    return "\n\n".join(contexts)

def build_g_eval_input_df(results_df, system_name="system"):
    rows = []

    for _, row in results_df.iterrows():
        retrieved_doc_ids = row["retrieved_doc_ids"] if isinstance(row["retrieved_doc_ids"], list) else []
        context = build_context_from_doc_ids(retrieved_doc_ids)

        rows.append({
            "eval_id": row["eval_id"],
            "system": system_name,
            "question": row["question"],
            "pred_answer": row["pred_answer"],
            "context": context,
        })

    return pd.DataFrame(rows)

print("[INFO] build_g_eval_input_df 함수 준비 완료")

In [ ]:
# ============================================================
# [STEP 9 - CELL 4] Prompt A/B/C 결과 준비
# ------------------------------------------------------------
# Prompt 비교를 하고 싶으면 A/B/C 결과를 여기서 정리한다.
# 없으면 시스템 비교만 진행 가능하다.
# ============================================================

prompt_frames = {}

if "results_prompt_A" in globals():
    prompt_frames["prompt_A_grounded_strict"] = results_prompt_A.copy()

if "results_prompt_B" in globals():
    prompt_frames["prompt_B_medical_consult"] = results_prompt_B.copy()

if "results_prompt_C" in globals():
    prompt_frames["prompt_C_medical_safe"] = results_prompt_C.copy()

print("[INFO] Prompt 비교 대상:", list(prompt_frames.keys()))
for k, v in prompt_frames.items():
    print(f" - {k}: {v.shape}")

In [ ]:
# ============================================================
# [STEP 9 - CELL 5] Prompt 결과에 retrieval 정보 연결
# ------------------------------------------------------------
# Prompt A/B/C 결과는 retrieved_doc_ids가 이미 들어있으면 그대로 쓰고,
# 없으면 reranked 결과에서 eval_id 기준으로 붙인다.
# ============================================================

if "reranked_rag_results_df" in globals():
    rerank_lookup = {
        row["eval_id"]: row
        for _, row in reranked_rag_results_df.iterrows()
    }
else:
    rerank_lookup = {}

def attach_retrieval_if_missing(prompt_df):
    prompt_df = prompt_df.copy()

    if "retrieved_doc_ids" not in prompt_df.columns:
        prompt_df["retrieved_doc_ids"] = None

    new_rows = []
    for _, row in prompt_df.iterrows():
        row_dict = row.to_dict()
        if row_dict.get("retrieved_doc_ids") is None or not isinstance(row_dict.get("retrieved_doc_ids"), list):
            eval_id = row_dict["eval_id"]
            if eval_id in rerank_lookup:
                row_dict["retrieved_doc_ids"] = rerank_lookup[eval_id]["retrieved_doc_ids"]
        new_rows.append(row_dict)

    return pd.DataFrame(new_rows)

for k in list(prompt_frames.keys()):
    prompt_frames[k] = attach_retrieval_if_missing(prompt_frames[k])

print("[INFO] Prompt retrieval 정보 연결 완료")

In [ ]:
# ============================================================
# [STEP 9 - CELL 6] G-EVAL 실행 함수
# ============================================================

def run_g_eval_for_system(results_df, system_name="system", limit=None, judge_model="gpt-4o-mini"):
    rows = results_df.copy()

    if limit is not None:
        rows = rows.head(limit).copy()

    g_eval_input_df = build_g_eval_input_df(rows, system_name=system_name)

    out_rows = []

    for i, (_, row) in enumerate(g_eval_input_df.iterrows(), start=1):
        judged = judge_answer_with_g_eval(
            question=row["question"],
            context=row["context"],
            answer=row["pred_answer"],
            judge_model=judge_model,
        )

        out_rows.append({
            "eval_id": row["eval_id"],
            "system": system_name,
            "question": row["question"],
            "groundedness": judged["groundedness"],
            "relevance": judged["relevance"],
            "completeness": judged["completeness"],
            "clarity": judged["clarity"],
            "safety": judged["safety"],
        })

        if i % 10 == 0:
            print(f"[INFO] {system_name}: {i}/{len(g_eval_input_df)} 완료")

    out_df = pd.DataFrame(out_rows)
    summary = {
        "system": system_name,
        "groundedness_mean": float(out_df["groundedness"].mean()),
        "relevance_mean": float(out_df["relevance"].mean()),
        "completeness_mean": float(out_df["completeness"].mean()),
        "clarity_mean": float(out_df["clarity"].mean()),
        "safety_mean": float(out_df["safety"].mean()),
        "overall_mean": float(
            out_df[["groundedness", "relevance", "completeness", "clarity", "safety"]].mean(axis=1).mean()
        ),
        "n": int(len(out_df)),
    }

    return out_df, summary

print("[INFO] run_g_eval_for_system 함수 준비 완료")

In [ ]:
# ============================================================
# [STEP 9 - CELL 7] Pipeline 시스템 G-EVAL 실행
# ------------------------------------------------------------
# naive / hybrid / reranked 시스템 비교
# ============================================================

g_eval_result_dfs = {}
g_eval_summaries = []

for system_name, df in system_frames.items():
    print(f"\n[INFO] G-EVAL 실행 시작: {system_name}")
    out_df, summary = run_g_eval_for_system(
        results_df=df,
        system_name=system_name,
        limit=len(df),   # 현재 샘플 수만큼
        judge_model="gpt-4o-mini",
    )
    g_eval_result_dfs[system_name] = out_df
    g_eval_summaries.append(summary)

g_eval_summary_df = pd.DataFrame(g_eval_summaries)
display(g_eval_summary_df.sort_values("overall_mean", ascending=False))

In [ ]:
# ============================================================
# [STEP 9 - CELL 8] Prompt A/B/C G-EVAL 실행
# ------------------------------------------------------------
# retrieval은 고정, prompt만 비교
# ============================================================

prompt_g_eval_result_dfs = {}
prompt_g_eval_summaries = []

for system_name, df in prompt_frames.items():
    print(f"\n[INFO] Prompt G-EVAL 실행 시작: {system_name}")
    out_df, summary = run_g_eval_for_system(
        results_df=df,
        system_name=system_name,
        limit=len(df),
        judge_model="gpt-4o-mini",
    )
    prompt_g_eval_result_dfs[system_name] = out_df
    prompt_g_eval_summaries.append(summary)

prompt_g_eval_summary_df = pd.DataFrame(prompt_g_eval_summaries)
display(prompt_g_eval_summary_df.sort_values("overall_mean", ascending=False))

In [ ]:
# ============================================================
# [STEP 9 - CELL 9] 평가 결과 저장
# ============================================================

G_EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / "g_eval_summary_v1.csv"
PROMPT_G_EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / "prompt_g_eval_summary_v1.csv"

if "g_eval_summary_df" in globals():
    g_eval_summary_df.to_csv(G_EVAL_SUMMARY_CSV_PATH, index=False, encoding="utf-8-sig")
    print("[INFO] 저장 완료:", G_EVAL_SUMMARY_CSV_PATH)

if "prompt_g_eval_summary_df" in globals() and len(prompt_g_eval_summary_df) > 0:
    prompt_g_eval_summary_df.to_csv(PROMPT_G_EVAL_SUMMARY_CSV_PATH, index=False, encoding="utf-8-sig")
    print("[INFO] 저장 완료:", PROMPT_G_EVAL_SUMMARY_CSV_PATH)

In [ ]:
# ============================================================
# [STEP 9 - CELL 10] 최종 요약
# ============================================================

final_eval_summary = {}

if "ragas_summary_df" in globals():
    final_eval_summary["ragas_best_system"] = ragas_summary_df.sort_values(
        "answer_correctness", ascending=False
    ).iloc[0]["system"]

if "g_eval_summary_df" in globals():
    final_eval_summary["g_eval_best_pipeline"] = g_eval_summary_df.sort_values(
        "overall_mean", ascending=False
    ).iloc[0]["system"]

if "prompt_g_eval_summary_df" in globals() and len(prompt_g_eval_summary_df) > 0:
    final_eval_summary["g_eval_best_prompt"] = prompt_g_eval_summary_df.sort_values(
        "overall_mean", ascending=False
    ).iloc[0]["system"]

print(json.dumps(final_eval_summary, ensure_ascii=False, indent=2))